# Deep Learning course Project 2026

## Dynamic and Hybrid Conditioning for Compositional Image Retrieval
### From Similarity to Probability: Amortized Bayesian Attribute-Posterior Fusion

*Authors: Alfonso Maria Antognozzi Caraffa [267504], Davide Cabitza [268264]*

---

## Abstract

Compositional image retrieval requires retrieving targets that preserve the identity of a reference
image $v_{\text{ref}}$ while satisfying positive ($T^+$) and negative ($T^-$) textual constraints.
This report documents our full experimental journey on the CelebA benchmark with frozen CLIP
ViT-B/32. We proceeded in three acts. **(1)** A systematic training-free investigation — latent
arithmetic and its geometric corrections, CLAY and GDE reproductions, visual-prototype geodesic
composition with rejection-based negation, and a reference-conditioned gated projection that
directly replaces CLAY's naïve pre-SVD stack — which establishes that single-vector cosine scoring
plateaus at mean R@5 ≈ 0.115 and fails absolutely on pure-negation queries. **(2)** A family of
*learned* fusion modules $\Phi$ over the same geometry, culminating in a cross-attention gate
trained with a listwise ranking loss, a learned negation head, online hard-negative mining and an
attribute-disentanglement regularizer, then a dual-channel variant adding calibrated probe presence/absence channels (mean R@5 0.175) — better, but still walled at 0.000 on
`-Male, -Mustache`. **(3)** The pivot: the benchmark's ground-truth rule is a *known generative
process over attribute space*, so we score it directly as **Bayesian inference over calibrated
attribute posteriors** read off the frozen features by linear probes (closed-form: R@5 0.260,
negation floor broken with zero training), refine it with temperature calibration and a Chow–Liu
attribute-dependency tree, and finally *amortize* it into the fusion module the task demands — a
single composite query embedding in a probe-augmented space, provably initialized at the
closed-form score. The trained module reaches **R@1 0.110 / R@5 0.291 / R@10 0.413** vs.
0.060 / 0.175 / 0.256 for our strongest geometric model (Dual Score). Two closing
cross-dataset studies put the result in perspective. On **AwA2** — where class-level
attributes force an adaptive-radius generalization of the edit rule — the rule-scoring bias
that wins CelebA is no longer dominant: Dual Score's learned fusion of a cosine backbone
with probe channels leads decisively. On **OVAD** — 117 tri-state attributes and roughly
26× less training data than CelebA — the same bias collapses outright: A-BAP finishes
**last of all seven methods on every metric**, the weakest score recorded anywhere in this
project. Together the two excursions turn the roster into a map of each method's inductive
bias: the more a scorer keeps a CLIP-cosine backbone, the better it transfers; the more it
depends on the learned probe/tree apparatus tuned for CelebA, the more exposed it is to an
attribute-count / data-volume regime it was never built for.

> *One sentence: CLAY turns VLM embedding space into a text-conditional similarity space; we turn
> it into a text-conditional probability space.*


## How to run this notebook

Everything needed to reproduce every reported result is **in this notebook**, including all
training code. Trained weights are **not** part of the delivery: the two `RETRAIN_*` flags decide
whether to load them or retrain, and **a plain Run-All never launches a training run by
surprise** — with a flag off, the model loads its cached weights if present and is otherwise
*skipped* with an explanatory message (the report's tables then rely on the committed result
CSVs). Cheap derived artifacts (features, prompt bank, directions, probes, calibration) follow
the load-or-build pattern, since they are prerequisites of every method. One flag sits above all
of these: `FETCH_ARTIFACT_BUNDLE` downloads a single precomputed bundle (every cached artifact,
result dump and trained weight) at the top of the notebook so a plain Run-All finishes in minutes
instead of rebuilding everything from raw CLIP features (~2h GPU total) — it is a no-op whenever
`ARTIFACTS_DIR` already has content, and every flag below still governs what happens underneath
it exactly as described.

| Flag | `False` (default) | `True` |
|---|---|---|
| `FETCH_ARTIFACT_BUNDLE` | fetch the precomputed bundle if `ARTIFACTS_DIR` is empty, else no-op | (set `ARTIFACT_BUNDLE_FILE_ID`; `False` always skips the fetch) |
| `RETRAIN_COMBINED` | load Φ-Combined weights if present, else **skip** | train from scratch (~45 min GPU) |
| `RETRAIN_ABAP` | load A-BAP Φ weights if present, else **skip** | train from scratch (~45 min GPU) |
| `REBUILD_FEATURES` | load cached CLIP feature tables, else build | force re-encode of CelebA train+test (~1 h GPU) |
| `REBUILD_PROMPT_BANK` | load cached prompt bank, else build | force re-encode (~1 min GPU) |
| `REMINE_DIRECTIONS` | load cached visual directions, else mine | force re-mine (~min, CPU ok) |
| `RETRAIN_PROBES` | load cached probes, else train (~10 s CPU) | force retrain |
| `REFIT_CALIBRATION` | load cached temperatures + tree, else fit | force refit (~min) |

**Prerequisites** (paths configurable in the next cell): the official `celeba_evaluation.json`
under `Evaluation/`. Raw CelebA images (aligned images + `list_attr_celeba.txt` +
`list_eval_partition.txt` at `CELEBA_ROOT`) are **only** needed if you go beyond the fetched
bundle — `REBUILD_FEATURES=True`, retraining from raw pixels, or live-rendering the section 9.4
qualitative grids; on the default bundle-fetched path no cell touches a CelebA image. All derived
artifacts (feature tables, prompt bank, directions, probes, weights) are built **once** and
cached under `ARTIFACTS_DIR`/`OUTPUT_DIR` — the frozen-database discipline of spec §3.2 is
enforced throughout by an explicit tripwire. The AwA2 section (§11) evaluates the adaptive-radius
benchmark `Evaluation/awa2_evaluation.json` (see §11.1–§11.2) and reads cached per-source dumps
from `output/awa2/per_source/`; any missing model is announced and skipped, never rebuilt by
surprise. The custom-query study (§10.11) likewise reads
`output/customQueries_celebA/per_source/`, regenerable inference-only via
`python src/generate_custom_gt.py` then `python src/run_custom_eval.py`.


In [ ]:
# == Configuration ============================================================
# Single source of truth for paths, flags and reproducibility. Edit here only.
from pathlib import Path
import os, sys, json, time, random

# Load-or-retrain flags (teacher contract: weights optional; True forces a rebuild)
REBUILD_FEATURES      = False
REBUILD_PROMPT_BANK   = False
REMINE_DIRECTIONS     = False
RETRAIN_PROBES        = False
REFIT_CALIBRATION     = False
RETRAIN_COMBINED      = False
RETRAIN_ABAP          = False

# Fetch the precomputed artifact/output bundle instead of rebuilding everything
# from scratch (~2h GPU total). A no-op whenever ARTIFACTS_DIR already has content
# (e.g. this dev checkout) or the file id below is unset -- it only ever saves
# time, and every REBUILD_*/RETRAIN_* flag above still governs what happens
# underneath it once the bundle (or its absence) is resolved.
FETCH_ARTIFACT_BUNDLE   = True
ARTIFACT_BUNDLE_FILE_ID = "REPLACE_WITH_GOOGLE_DRIVE_FILE_ID"   # public "anyone with the link" zip

SEED = 0

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q transformers gdown
    PROJECT_ROOT = Path("/content")
else:
    PROJECT_ROOT = Path.cwd()

# CelebA raw images are deliberately NOT part of the artifact bundle and are only
# needed if you go beyond the cached fast path: REBUILD_FEATURES=True, retraining
# from raw pixels, or live-rendering the section 9.4 qualitative grids. Every other
# cell reads pre-built tensors/dumps from ARTIFACTS_DIR/OUTPUT_DIR and never touches
# a CelebA image. To use those paths, get CelebA the normal way (official source /
# course-provided copy) and point CELEBA_ROOT below at it, e.g. on Colab:
#   from google.colab import drive; drive.mount("/content/drive")
#   CELEBA_ROOT = Path("/content/drive/MyDrive/<your CelebA folder>")
CELEBA_ROOT   = PROJECT_ROOT / "celeba"                # contains celeba/img_align_celeba
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"; ARTIFACTS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR    = PROJECT_ROOT / "output";    OUTPUT_DIR.mkdir(exist_ok=True)
EVAL_JSON     = PROJECT_ROOT / "Evaluation" / "celeba_evaluation.json"

# Bundle fetch -- ARTIFACTS_DIR/OUTPUT_DIR are the only things a fresh Colab
# runtime is missing; CelebA images are excluded on purpose (see note above).
if FETCH_ARTIFACT_BUNDLE and not any(ARTIFACTS_DIR.iterdir()):
    if ARTIFACT_BUNDLE_FILE_ID.startswith("REPLACE_"):
        print("[skipped] set ARTIFACT_BUNDLE_FILE_ID to fetch the precomputed bundle; "
              "falling through to the from-scratch load-or-build cells below")
    else:
        import gdown, zipfile
        bundle_zip = PROJECT_ROOT / "artifact_bundle.zip"
        gdown.download(id=ARTIFACT_BUNDLE_FILE_ID, output=str(bundle_zip), quiet=False)
        with zipfile.ZipFile(bundle_zip) as zf:
            zf.extractall(PROJECT_ROOT)
        bundle_zip.unlink()
        print(f"[fetched] artifact bundle extracted to {PROJECT_ROOT}")

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={DEVICE}  colab={IN_COLAB}  root={PROJECT_ROOT}")


---
# 1. Introduction

Given a reference image $v_{\text{ref}}$, positive textual constraints $T^+$ and negative
constraints $T^-$, the task (spec §1) is a fusion module $\Phi(v_{\text{ref}}, T^+, T^-)$
producing a composite query that retrieves targets preserving the reference identity while
satisfying every constraint, scored against a visual database encoded **once** with frozen CLIP
ViT-B/32 and never modified (spec §3.2–3.3). The assignment's named enemy is CLAY's *naïve
pre-SVD stacking* of condition embeddings — rigid, polarity-blind, with no dynamic control over
how conflicting conditions combine.

This notebook is deliberately written as a **chronicle of the investigation**, because the final
method only makes sense as the conclusion of everything that failed before it. Each section states
what we built, *why* we built it given what the previous experiment showed, and what its numbers
taught us.

**Contributions.**
1. A systematic **training-free study** (vanilla arithmetic → geometric corrections → CLAY/GDE
   reproductions → visual-prototype geodesic composition → dynamic gated projection) establishing
   that the single-vector-cosine paradigm plateaus structurally, not for lack of tuning (§4).
2. A **learned geometric fusion family** — cross-attention gate, listwise ranking loss, learned
   negation head, online mining, attribute disentanglement, and a dual-channel probe fusion (Dual Score) — that beats every training-free method
   yet inherits the same negation wall (§5).
3. **BAP**: a closed-form Bayesian scorer of the benchmark's own ground-truth generating process
   over calibrated probe posteriors, featuring an exact Poisson-binomial identity term (§6).
4. **A-BAP**: the amortization of BAP (+ its correlation-aware refinements) into the required
   fusion module $\Phi$ — a single composite query embedding with a **Bayes initialization**
   guaranteeing the untrained module reproduces the closed-form score exactly (§7).

# 2. Related Work

- **Assigned lineage.** GDE (Berasi et al., 2025) decomposes entangled VLM embeddings into
  separable latent structures via tangent-space geometry on the unit sphere; CLAY (Lim et al.,
  2026) reframes VLM space as a text-conditional similarity space (per-condition SVD subspaces +
  a modality-gap rotation, against a frozen visual DB). Its equal-weight pre-SVD prompt stack is
  the bottleneck this assignment targets.
- **The modality gap.** Liang et al. (2022) show image and text embeddings occupy two separate
  cones; the constant offset between them dominates naïve cross-modal arithmetic — the single
  most consequential fact of our training-free study.
- **Negation in VLMs.** NegBench (Alhamoud et al., 2025) documents CLIP's affirmation bias;
  PoS-Grounded Subspaces (Oldfield et al., 2023) formalizes negation as subspace rejection
  ("no preference on the X axis") rather than anti-X subtraction.
- **Composed image retrieval.** Combiner, Pic2Word, SEARLE, CIReVL optimize fused text–image
  embeddings for open-domain modification text; none faces our defining feature — a *known,
  closed-form ground-truth generating process*.
- **Probabilistic ingredients.** Temperature scaling (Guo et al., 2017); Chow–Liu dependency
  trees (Chow & Liu, 1968; TAN, Friedman et al., 1997); ListNet (Cao et al., 2007); LambdaLoss
  (Wang et al., 2018); querybank normalization (Bogolin et al., 2022).

**Positioning.** To our knowledge no prior work scores compositional retrieval with a
Poisson-binomial Hamming-tail identity term over probe posteriors, nor amortizes a Bayes-optimal
attribute scorer into a single augmented-space query embedding.


# 3. Problem Formulation: the Benchmark as a Generative Process

## 3.1 Ground-truth rule

Let $A(x) \in \{0,1\}^{40}$ be the attribute vector of image $x$ and $\mathcal{F}$ the *free*
attributes (not in $T^+ \cup T^-$). The benchmark (spec §3.1.1) declares $x$ a valid target for
$(v_{\text{ref}}, T^+, T^-)$ iff

$$
\underbrace{\bigwedge_{a \in T^+} A_a(x) = 1 \;\wedge\; \bigwedge_{b \in T^-} A_b(x) = 0}_{\text{strict constraint satisfaction}}
\quad\wedge\quad
\underbrace{\sum_{c \in \mathcal{F}} \mathbb{1}\!\left[A_c(x) \neq A_c(\text{ref})\right] \le 2}_{\text{identity: Hamming ball}}
$$

## 3.2 Metrics

For each query, averaged over its valid source images (spec §3.1.3), at $K \in \{1, 5, 10\}$:
$\text{Recall@}K = \mathbb{1}[\,|R_K \cap G| > 0\,]$ (a hit rate) and
$\text{Precision@}K = |R_K \cap G| / K$.

> **Note on the mandatory query set.** The official JSON contains 14 entries; `- Young` appears
> twice (entries 5 and 8, identical ground truth), and the JSON — the authoritative source — also
> differs slightly from the spec's 12-row table. We evaluate all entries as given; results
> dictionaries key by query string, so every aggregate in this report is a mean over the
> **13 unique queries**.

## 3.3 Why this rule will matter

Everything in §4–5 scores candidates by cosine against a single 512-d query vector. Cosine against
one point is a **linear functional** on the sphere: its super-level sets are spherical caps. The
valid-target set above — an intersection of attribute half-spaces with a Hamming ball — is not a
cap around any point, and on negation queries the natural identity anchor
$\cos(x, v_{\text{ref}})$ is *adversarial*: for `-Male, -Mustache` every source is a mustached
male, so valid targets are their *female attribute-twins* — images cosine ranks low precisely
because they changed gender. We did not begin the project with this analysis; we earned it, one
failed method at a time. Sections 4–5 are the evidence; §6 is the consequence.


## 3.4 Data pipeline — the frozen database

Built **once**: attribute tensors and L2-normalized CLIP feature tables for the *test* split (the
retrieval corpus, 19,962 images) and the *train* split (162,770 images — used only to mine
directions, train probes and train $\Phi$; never scored against). All image identifiers are
PyTorch **dataset indices** (spec §3.1.2: `celeba[13]` ≠ `000013.jpg`).


In [ ]:
# == Attribute vocabulary (fixed order = columns of list_attr_celeba.txt) ====
ATTRIBUTE_NAMES = [
    "5_o_Clock_Shadow", "Arched_Eyebrows", "Attractive", "Bags_Under_Eyes", "Bald",
    "Bangs", "Big_Lips", "Big_Nose", "Black_Hair", "Blond_Hair",
    "Blurry", "Brown_Hair", "Bushy_Eyebrows", "Chubby", "Double_Chin",
    "Eyeglasses", "Goatee", "Gray_Hair", "Heavy_Makeup", "High_Cheekbones",
    "Male", "Mouth_Slightly_Open", "Mustache", "Narrow_Eyes", "No_Beard",
    "Oval_Face", "Pale_Skin", "Pointy_Nose", "Receding_Hairline", "Rosy_Cheeks",
    "Sideburns", "Smiling", "Straight_Hair", "Wavy_Hair", "Wearing_Earrings",
    "Wearing_Hat", "Wearing_Lipstick", "Wearing_Necklace", "Wearing_Necktie", "Young",
]
ATTR_TO_IDX = {name: i for i, name in enumerate(ATTRIBUTE_NAMES)}
N_ATTR = 40


In [ ]:
# == CelebA loading + frozen attribute tensors ===============================
from torchvision import datasets, transforms

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]


def load_celeba_dataset(split: str = "test"):
    # CelebA split with CLIP preprocessing (resize 224 + CLIP mean/std normalize).
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ])
    return datasets.CelebA(root=CELEBA_ROOT, split=split, download=False, transform=transform)


def setup_frozen_attributes(split: str = "test", force: bool = False) -> torch.Tensor:
    # Frozen [N, 40] attribute tensor for a split: filter list_attr_celeba.txt to the
    # split's filenames, map -1/+1 -> 0/1. Built once, never modified.
    cache = ARTIFACTS_DIR / f"celeba_attributes_{split}.pt"
    if cache.exists() and not force:
        return torch.load(cache)
    ds = load_celeba_dataset(split)
    keep = set(ds.filename)
    rows = []
    with open(CELEBA_ROOT / "celeba" / "list_attr_celeba.txt") as f:
        f.readline(); f.readline()                     # count line + header line
        for line in f:
            parts = line.strip().split()
            if parts[0] in keep:
                rows.append([int(v) for v in parts[1:]])
    attrs = (torch.tensor(rows, dtype=torch.float32) + 1) / 2
    torch.save(attrs, cache)
    print(f"[built] {cache}  shape={tuple(attrs.shape)}")
    return attrs


In [ ]:
# == Frozen CLIP ViT-B/32 feature tables ======================================
# Row i == celeba[i] (shuffle=False is load-bearing: eval JSON keys are dataset
# indices). Rows L2-normalized so cosine == dot product downstream.
from torch.utils.data import DataLoader

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
FEATURE_DIM = 512


@torch.no_grad()
def extract_clip_features(split: str = "test", batch_size: int = 256,
                          force: bool = False) -> torch.Tensor:
    cache = ARTIFACTS_DIR / f"clip_image_features_{split}.pt"
    if cache.exists() and not force:
        return torch.load(cache)
    from transformers import CLIPModel
    model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()   # frozen
    ds = load_celeba_dataset(split)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0,
                        collate_fn=lambda b: torch.stack([img for img, _ in b]))
    feats, done = [], 0
    for pixels in loader:
        # Explicit vision-tower + projection: robust across transformers versions.
        out = model.vision_model(pixel_values=pixels.to(DEVICE))
        f = model.visual_projection(out.pooler_output)
        feats.append(F.normalize(f, p=2, dim=1).cpu())
        done += pixels.shape[0]
        print(f"  encoded {done}/{len(ds)}", end="\r")
    table = torch.cat(feats).to(torch.float32)
    torch.save(table, cache)
    print(f"\n[built] {cache}  shape={tuple(table.shape)}")
    return table


@torch.no_grad()
def extract_attribute_text_features(force: bool = False) -> torch.Tensor:
    # One prompt per attribute -> frozen [40, 512] text table (the +/- direction
    # vectors of the vanilla arithmetic family).
    cache = ARTIFACTS_DIR / "clip_attr_text_features.pt"
    if cache.exists() and not force:
        return torch.load(cache)
    from transformers import CLIPModel, AutoTokenizer
    model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(CLIP_MODEL_NAME)
    prompts = [f"a photo of a person with {n.replace('_', ' ').lower()}"
               for n in ATTRIBUTE_NAMES]
    tokens = tok(prompts, padding=True, return_tensors="pt").to(DEVICE)
    out = model.text_model(**tokens)
    t = F.normalize(model.text_projection(out.pooler_output), p=2, dim=1)
    t = t.cpu().to(torch.float32)
    torch.save(t, cache)
    return t


attrs_test  = setup_frozen_attributes("test")
attrs_train = setup_frozen_attributes("train")
feats_test  = extract_clip_features("test",  force=REBUILD_FEATURES).to(DEVICE)
feats_train = extract_clip_features("train", force=REBUILD_FEATURES).to(DEVICE)
text_feats  = extract_attribute_text_features().to(DEVICE)

# Tripwires: alignment + normalization — illegal states fail loudly, here and now.
assert feats_test.shape == (attrs_test.shape[0], FEATURE_DIM)
assert torch.allclose(feats_test.norm(dim=1),
                      torch.ones(feats_test.shape[0], device=DEVICE), atol=1e-3)
print(f"test corpus: {feats_test.shape[0]} | train: {feats_train.shape[0]} images")


## 3.5 Evaluation engine

The single ruler every method is measured through — built and unit-tested *before* any retrieval
method existed, so all methods are compared apples-to-apples. A method plugs in as a
`score_chunk(sources, pos_idx, neg_idx, free_idx) -> [S, N]` callable; the driver owns query
parsing, self-exclusion (a source never ranks itself), ranking and metric aggregation.


In [ ]:
# == Metrics (verbatim from spec section 3.1.3) ===============================
def recall_at_k(ranking: list[int], ground_truth: set[int], k: int) -> float:
    # Hit rate: 1.0 if top-K intersects G (NOT textbook recall |hits|/|relevant|).
    return 1.0 if set(ranking[:k]) & ground_truth else 0.0


def precision_at_k(ranking: list[int], ground_truth: set[int], k: int) -> float:
    return len(set(ranking[:k]) & ground_truth) / k


def parse_query(query_str: str) -> tuple[list[str], list[str]]:
    # '+Eyeglasses, -Smiling' or '+ Eyeglasses & - Smiling' -> (pos, neg) name lists.
    pos, neg = [], []
    for piece in query_str.replace("&", ",").split(","):
        piece = piece.strip()
        if not piece:
            continue
        sign, name = piece[0], piece[1:].strip().replace(" ", "_")
        if sign == "+":
            pos.append(name)
        elif sign == "-":
            neg.append(name)
        else:
            raise ValueError(f"query piece must start with + or -: {piece!r}")
    return pos, neg


def query_indices(query_str: str) -> tuple[list[int], list[int], list[int]]:
    # Attribute indices of (T+, T-, free) for a query string.
    pos, neg = parse_query(query_str)
    pi = [ATTR_TO_IDX[a] for a in pos]
    ni = [ATTR_TO_IDX[b] for b in neg]
    free = [c for c in range(N_ATTR) if c not in set(pi) | set(ni)]
    return pi, ni, free


def load_eval_json(path=EVAL_JSON) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


In [ ]:
# == Chunked evaluation driver ================================================
TOP_K_KEEP = 100        # metrics need K <= 10; keeping top-100 bounds memory


@torch.no_grad()
def evaluate_scorer(score_chunk, gt_list: list[dict], ks=(1, 5, 10),
                    chunk: int = 256, name: str = "") -> dict:
    # Score every JSON query through one seam. Duplicate query strings (the doubled
    # '- Young') overwrite each other -> aggregates are over unique queries.
    results = {}
    for entry in gt_list:
        q = entry["query"]
        pi, ni, free = query_indices(q)
        sources = [int(s) for s in entry["ground_truth"].keys()]
        sums = {f"{m}@{k}": 0.0 for k in ks for m in ("recall", "precision")}
        for lo in range(0, len(sources), chunk):
            batch = sources[lo:lo + chunk]
            scores = score_chunk(batch, pi, ni, free)              # [S, N]
            rows = torch.arange(len(batch), device=scores.device)
            cols = torch.as_tensor(batch, device=scores.device)
            scores[rows, cols] = float("-inf")                     # self-exclusion
            top = torch.topk(scores, TOP_K_KEEP, dim=1).indices.cpu()
            for j, src in enumerate(batch):
                ranking = top[j].tolist()
                G = {int(t) for t in entry["ground_truth"][str(src)]}
                for k in ks:
                    sums[f"recall@{k}"] += recall_at_k(ranking, G, k)
                    sums[f"precision@{k}"] += precision_at_k(ranking, G, k)
        n = len(sources)
        results[q] = {m: v / n for m, v in sums.items()} | {"num_sources": n}
        if name:
            print(f"  [{name}] {q:<46s} R@5={results[q]['recall@5']:.4f}")
    return results


def assert_db_frozen(image_features: torch.Tensor, score_chunk, gt_list) -> None:
    # Tripwire (spec section 3.2): the frozen CLIP DB must be byte-identical before and
    # after a query — scoring must never mutate or re-encode the database.
    before = image_features.clone()
    pi, ni, free = query_indices(gt_list[0]["query"])
    src = int(next(iter(gt_list[0]["ground_truth"])))
    score_chunk([src], pi, ni, free)
    assert torch.equal(image_features, before), "FROZEN-DB VIOLATION"


def results_table(results: dict, ks=(1, 5, 10)):
    # DataFrame view with a MEAN row (over the unique queries).
    df = pd.DataFrame(results).T
    df.loc["MEAN"] = df.mean()
    df.loc["MEAN", "num_sources"] = df["num_sources"][:-1].sum()
    cols = [f"recall@{k}" for k in ks] + [f"precision@{k}" for k in ks] + ["num_sources"]
    return df[cols].round(4)


def sweep_summary(configs: dict[str, Path], ks=(1, 5, 10)) -> pd.DataFrame:
    # Cached-sweep table (reporting utility) -- one row per already-computed config's
    # own MEAN. Pure file I/O against committed CSVs, no live compute: lets every
    # hyperparameter claim in the training-free sections be checked against the
    # underlying grid without retraining or rerunning anything.
    rows = {}
    for label, path in configs.items():
        if not path.exists():
            print(f"[missing] {path.name} -- '{label}' skipped")
            continue
        mean = pd.read_csv(path).set_index("query").loc["MEAN"]
        rows[label] = {f"R@{k}": mean[f"R@{k}"] for k in ks}
    return pd.DataFrame(rows).T.astype(float).round(4)

gt_list = load_eval_json()
print(f"{len(gt_list)} JSON entries, {len({e['query'] for e in gt_list})} unique queries")


---
# 4. The Training-Free Investigation

This section follows our actual experimental order. Each subsection opens with the reasoning that
led to it.

## 4.1 CLIP — the vanilla arithmetic baseline, and what its failure taught us

The spec's mandated lower bound (§3.2): $q = \mathrm{normalize}(v_{\text{ref}} + \alpha(\sum t^+ -
\sum t^-))$, cosine-scored. Result at the literal vanilla $\alpha{=}1$: **R@1 0.0224 / R@5 0.0699 /
R@10 0.1048**. Negations collapse entirely (`-Male, -Mustache` = 0.000).

The low scores were *investigated, not assumed*. The decisive diagnostic was an $\alpha{=}0$
sanity run (pure image→image retrieval, text ignored): **R@5 0.0496** — barely below $\alpha{=}1$.
The interpretation defined the whole project:

- $\alpha{=}0$ ignores the text → retrieves visual look-alikes that keep the source's *original*
  attribute values → they violate the query constraints.
- $\alpha{=}1$ adds a unit text vector which — because of CLIP's **modality gap** — yanks the query
  into the text cone and washes out reference identity.

**Neither endpoint wins.** "Ignore the text" and "drown the identity" both fail, for opposite
reasons: that tension *is* the fusion problem the assignment poses, and it makes the vanilla
baseline an honest lower bound we deliberately did not tune upward. (An $\alpha$-sweep is an
ablation of the vanilla method, not a new baseline.) We also verified the harness itself: the
feature table is unit-normed, self-retrieval returns the source at cosine 1.0, and an oracle/
adversary pair brackets the metrics — the numbers are a property of the method, not a bug.

*The baseline's code is a strict special case of the enhanced scorer below
(`center=False, norm_delta=False, use_prompt_bank=False`), so it is not duplicated here.*

## 4.2 CLIP Enhanced — three geometric fixes

The $\alpha{=}0/\alpha{=}1$ diagnosis pointed at *geometry*, not semantics. Before reaching for
subspaces or learning, we asked: how far does vanilla arithmetic go if we only fix its geometric
mistakes? Three independent, well-cited, training-free corrections:

**FIX 1 — modality-gap centering** (Liang et al. 2022). Every attribute text vector decomposes as
$t(a) = \mu_{\text{txt}} + (t(a) - \mu_{\text{txt}})$, where $\mu_{\text{txt}}$ — the centre of
the text cone — is *identical for all 40 attributes*: it encodes "this is a text embedding," not
"this is *Smiling*". The naïve formula spends most of $\alpha$'s budget adding this useless common
component. Fix: subtract $\mu_{\text{txt}}$ from every text vector and do the arithmetic in the
image-centered frame ($v_{\text{ref}} - \mu_{\text{img}}$, re-adding $\mu_{\text{img}}$ at the end
so the query lands back in the image cone).

**FIX 2 — delta normalization.** $\|\sum t^+ - \sum t^-\|$ grows with the constraint count, so one
fixed $\alpha$ secretly means a different push strength per query. Normalizing the delta makes
$\alpha$ one honest angle on the sphere for every query.

**FIX 3 — prompt ensembling** (the classic CLIP 80-template trick). One sentence mixes the concept
with phrasing noise; the mean of ~60 differently-worded paraphrases (the *prompt bank*, built
below — it will also be CLAY's raw material) is a lower-variance attribute direction. Order is
load-bearing: ensemble **first**, then subtract $\mu_{\text{txt}}$ — averaging paraphrases does not
remove the common-mode offset.

| Config | R@1 | R@5 | R@10 |
|---|---|---|---|
| naïve (vanilla) | 0.0224 | 0.0699 | 0.1048 |
| FIX 3 prompt-bank | 0.0243 | 0.0715 | 0.1050 |
| FIX 2 delta-normalize | 0.0242 | 0.0713 | 0.1083 |
| **FIX 1 centering** | 0.0377 | **0.1142** | **0.1755** |
| **all three (= CLIP Enhanced)** | **0.0397** | 0.1140 | 0.1651 |

**Centering alone is the entire win: +63% relative R@5, the single most valuable geometric trick of
the whole project.** FIX 2/3 are noise-level alone (the common-mode bias still dominates) and the
fixes are not additive (all-three trades R@5/R@10 for the best R@1). And the residual dead queries
— `-Male, -Mustache` = 0.000 under *every* config — expose the **negation problem**, which
centering provably cannot touch. That is what §4.4–4.6 attack.

We adopt the all-three-fixes configuration as the report's standard **CLIP Enhanced** — it takes the best R@1, and it is the configuration used in every cross-method comparison and in the statistical suite (§10). *(The all-fixes row above is the official-CSV value over the 13 unique queries.)*


In [ ]:
# == Prompt bank — ~60 paraphrases per attribute (FIX 3 + CLAY's raw material) ==
# Frames x synonyms grid, deterministic (from-scratch substitute for CLAY's LLM
# prompts, spec section 6). The spread across paraphrases IS what SVD-style methods
# later operate on; here we only use the centroid.
FRAMES = [
    "a photo of a person with {phrase}", "a photo of a person who has {phrase}",
    "a portrait of someone with {phrase}", "a close-up of a face with {phrase}",
    "an image of a person showing {phrase}", "a headshot of an individual with {phrase}",
    "a picture of a face with {phrase}", "a cropped photo of a person with {phrase}",
    "a selfie of someone with {phrase}", "a candid shot of a person having {phrase}",
    "a frontal portrait of a face with {phrase}", "a clear photograph of a person with {phrase}",
]
PREDICATIVE_FRAMES = [
    "a photo of a {phrase} person", "a photo of someone who is {phrase}",
    "a portrait of a {phrase} person", "a close-up of a {phrase} face",
    "an image of a {phrase} individual", "a headshot of a {phrase} person",
    "a picture of a {phrase} face", "a cropped photo of a {phrase} person",
    "a selfie of a {phrase} person", "a candid shot of a {phrase} individual",
    "a frontal portrait of a {phrase} person", "a clear photograph of a {phrase} person",
]
ATTR_PHRASES = {
    "5_o_Clock_Shadow": ("noun", ["a five o'clock shadow", "light stubble", "a faint beard shadow", "short stubble on the face", "a day's worth of stubble"]),
    "Arched_Eyebrows": ("noun", ["arched eyebrows", "curved eyebrows", "highly arched brows", "sharply arched eyebrows", "elegantly curved brows"]),
    "Attractive": ("adj", ["attractive", "good looking", "beautiful", "handsome", "striking-looking"]),
    "Bags_Under_Eyes": ("noun", ["bags under the eyes", "puffy under-eyes", "eye bags", "swollen under-eye skin", "dark circles under the eyes"]),
    "Bald": ("adj", ["bald", "hairless on the head", "with no hair", "completely bald", "with a shaved head"]),
    "Bangs": ("noun", ["bangs", "a fringe of hair over the forehead", "hair covering the forehead", "a straight fringe", "front bangs across the brow"]),
    "Big_Lips": ("noun", ["big lips", "full lips", "large lips", "plump lips", "thick full lips"]),
    "Big_Nose": ("noun", ["a big nose", "a large nose", "a prominent nose", "a wide nose", "a broad nose"]),
    "Black_Hair": ("noun", ["black hair", "dark black hair", "jet-black hair", "deep black hair", "raven-black hair"]),
    "Blond_Hair": ("noun", ["blond hair", "blonde hair", "light golden hair", "pale yellow hair", "bright blonde hair"]),
    "Blurry": ("adj", ["blurry", "out of focus", "unsharp", "blurred and hazy", "low in sharpness"]),
    "Brown_Hair": ("noun", ["brown hair", "brunette hair", "dark brown hair", "chestnut-brown hair", "medium brown hair"]),
    "Bushy_Eyebrows": ("noun", ["bushy eyebrows", "thick eyebrows", "dense brows", "heavy bushy eyebrows", "full thick brows"]),
    "Chubby": ("adj", ["chubby", "plump", "round-faced", "full-cheeked", "heavyset in the face"]),
    "Double_Chin": ("noun", ["a double chin", "a second chin", "extra fold under the chin", "a fleshy under-chin", "a sagging under-chin"]),
    "Eyeglasses": ("noun", ["eyeglasses", "glasses", "spectacles", "a pair of glasses", "reading glasses"]),
    "Goatee": ("noun", ["a goatee", "a goatee beard", "a small pointed beard", "a chin goatee", "a trimmed goatee"]),
    "Gray_Hair": ("noun", ["gray hair", "grey hair", "silver hair", "white-gray hair", "salt-and-pepper hair"]),
    "Heavy_Makeup": ("noun", ["heavy makeup", "a lot of makeup", "thick makeup", "bold heavy makeup", "heavily applied makeup"]),
    "High_Cheekbones": ("noun", ["high cheekbones", "prominent cheekbones", "sharp high cheekbones", "well-defined cheekbones", "raised cheekbones"]),
    "Male": ("noun", ["a male face", "the face of a man", "masculine features", "a man's face", "a masculine appearance"]),
    "Mouth_Slightly_Open": ("noun", ["a slightly open mouth", "lips parted slightly", "a mouth slightly open", "barely parted lips", "a softly open mouth"]),
    "Mustache": ("noun", ["a mustache", "a moustache", "hair on the upper lip", "a thick mustache", "a trimmed mustache"]),
    "Narrow_Eyes": ("noun", ["narrow eyes", "squinted eyes", "thin eyes", "slightly closed eyes", "narrowed eyes"]),
    "No_Beard": ("adj", ["clean-shaven", "without a beard", "with a beardless face", "smooth-shaven", "free of facial hair"]),
    "Oval_Face": ("noun", ["an oval face", "an oval-shaped face", "a long oval face", "a softly oval face", "an egg-shaped face"]),
    "Pale_Skin": ("noun", ["pale skin", "fair skin", "light-toned skin", "very pale skin", "porcelain-pale skin"]),
    "Pointy_Nose": ("noun", ["a pointy nose", "a sharp nose", "a pointed nose", "a narrow pointed nose", "a thin pointy nose"]),
    "Receding_Hairline": ("noun", ["a receding hairline", "a hairline pulled back", "thinning hair at the temples", "a balding hairline", "a hairline receding at the front"]),
    "Rosy_Cheeks": ("noun", ["rosy cheeks", "pink cheeks", "flushed cheeks", "reddish rosy cheeks", "blushing cheeks"]),
    "Sideburns": ("noun", ["sideburns", "long sideburns", "hair down the sides of the face", "thick sideburns", "sideburns along the cheeks"]),
    "Smiling": ("adj", ["smiling", "with a smile", "grinning", "smiling broadly", "happily smiling"]),
    "Straight_Hair": ("noun", ["straight hair", "sleek straight hair", "perfectly straight hair", "smooth straight hair", "flat straight hair"]),
    "Wavy_Hair": ("noun", ["wavy hair", "curly wavy hair", "loosely wavy hair", "soft wavy hair", "gently waving hair"]),
    "Wearing_Earrings": ("noun", ["earrings", "ear jewelry", "studs in the ears", "dangling earrings", "a pair of earrings"]),
    "Wearing_Hat": ("noun", ["a hat", "a cap on the head", "headwear", "a brimmed hat", "a hat on the head"]),
    "Wearing_Lipstick": ("noun", ["lipstick", "colored lipstick", "lip color", "bright lipstick", "bold lipstick"]),
    "Wearing_Necklace": ("noun", ["a necklace", "a chain around the neck", "neck jewelry", "a pendant necklace", "a beaded necklace"]),
    "Wearing_Necktie": ("noun", ["a necktie", "a tie", "a knotted tie", "a formal necktie", "a tie around the collar"]),
    "Young": ("adj", ["young", "youthful", "young-looking", "youthful-faced", "in their youth"]),
}
assert set(ATTR_PHRASES) == set(ATTRIBUTE_NAMES)      # no drift vs the master list


def build_prompts_for_attribute(name: str) -> list[str]:
    # Frames x synonyms, de-duplicated; "adj" attributes use the predicative frames.
    kind, phrases = ATTR_PHRASES[name]
    frames = PREDICATIVE_FRAMES if kind == "adj" else FRAMES
    seen, prompts = set(), []
    for phrase in phrases:
        for frame in frames:
            text = frame.format(phrase=phrase)
            if text not in seen:
                seen.add(text); prompts.append(text)
    return prompts


@torch.no_grad()
def extract_prompt_bank(force: bool = False) -> torch.Tensor:
    # [40, n, 512] unit rows; stacks padded to a common n with repeated first rows
    # (SVD-safe: a duplicated row adds no new span direction).
    cache = ARTIFACTS_DIR / "clip_attr_prompt_bank.pt"
    if cache.exists() and not force:
        return torch.load(cache)
    from transformers import CLIPModel, AutoTokenizer
    model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(CLIP_MODEL_NAME)
    per_attr = [build_prompts_for_attribute(n) for n in ATTRIBUTE_NAMES]
    n = max(len(p) for p in per_attr)
    bank = torch.empty(N_ATTR, n, FEATURE_DIM, dtype=torch.float32)
    for j, prompts in enumerate(per_attr):
        padded = prompts + [prompts[0]] * (n - len(prompts))
        tokens = tok(padded, padding=True, return_tensors="pt").to(DEVICE)
        out = model.text_model(**tokens)
        f = F.normalize(model.text_projection(out.pooler_output), p=2, dim=1)
        bank[j] = f.cpu().to(torch.float32)
        print(f"  encoded {j + 1}/{N_ATTR} ({ATTRIBUTE_NAMES[j]})", end="\r")
    torch.save(bank, cache)
    print(f"\n[built] {cache}  shape={tuple(bank.shape)}")
    return bank


prompt_bank = extract_prompt_bank(force=REBUILD_PROMPT_BANK).to(DEVICE)


In [ ]:
# == Enhanced arithmetic — vanilla latent arithmetic in CORRECT CLIP geometry ==
# q = normalize((v_ref − μ_img) + α·d + μ_img),  d = normalize(Σ t̂⁺ − Σ t̂⁻),
# t̂(a) = normalize(normalize(meanₖ bank[a,k]) − μ_txt)      (FIX 3 then FIX 1).
# Three independent toggles; all off ≈ the vanilla baseline. No SVD, no learning.
def build_attr_directions(use_prompt_bank: bool, center: bool):
    # Per-attribute unit directions + the image-cone mean (for the image-side centering).
    mu_img = feats_test.mean(dim=0)
    if use_prompt_bank:
        attr = F.normalize(prompt_bank.mean(dim=1), p=2, dim=1)      # FIX 3 centroid
    else:
        attr = text_feats.clone()
    if center:
        attr = F.normalize(attr - attr.mean(dim=0), p=2, dim=1)     # FIX 1 (text side)
    return attr, mu_img


def make_enhanced_chunk(alpha: float = 1.0, use_prompt_bank: bool = False,
                        center: bool = True, norm_delta: bool = False):
    attr, mu_img = build_attr_directions(use_prompt_bank, center)

    def score_chunk(srcs, pi, ni, free):
        delta = attr[pi].sum(0) - attr[ni].sum(0)                    # [512]
        if norm_delta and delta.norm() > 1e-8:
            delta = F.normalize(delta, dim=0)                        # FIX 2
        v_ref = feats_test[srcs]                                     # [S, 512]
        if center:
            q = (v_ref - mu_img) + alpha * delta + mu_img            # FIX 1 (image side)
        else:
            q = v_ref + alpha * delta
        return F.normalize(q, dim=1) @ feats_test.T
    return score_chunk


# The report-standard config — CLIP Enhanced = all three fixes (see table above).
results_enhanced = evaluate_scorer(
    make_enhanced_chunk(use_prompt_bank=True, center=True, norm_delta=True),
    gt_list, name="CLIP Enhanced")
results_table(results_enhanced).tail(3)


## 4.3 Reproducing the assigned literature: CLAY and GDE

With arithmetic exhausted, we reproduced the two assigned methods faithfully, as the spec's
roadmap implies.

**CLAY** (tangent-space SVD subspaces per condition + the modality-gap rotation $H$, scored as
cosine inside the projected subspace) needs a *stack* of prompts per condition — that is exactly
what the prompt bank provides. Faithful CLAY is polarity-blind: `+` and `−` prompts are stacked
into one subspace. Concretely, each condition's subspace is $P_c = V_k V_k^\top$ from a rank-$k$
truncated SVD of the tangent-mapped prompt stack at the condition's own mean, and $H$ is a
rotation aligning the image-cone mean to the text-cone mean before scoring — the same
modality-gap correction as FIX 1 (§4.2), but applied as a basis rotation rather than a
translation. Two knobs are therefore swept below: the truncation rank $k$ (how much of each
condition's tangent-space variance survives projection) and whether $H$ is applied. Best
configuration (k=50, rotation on): **R@5 0.0368 — below the vanilla baseline at every
configuration we tried.** Reading the grid directly, not just its endpoints: R@5 rises strictly
monotonically with $k$ (0.0075 → 0.0134 → 0.0213 → 0.0368 for $k=5,10,20,50$) with **no sign of
saturation** in this range — more of the condition's tangent-space variance keeps helping all the
way to $k=50$ — and at fixed $k=50$ the rotation $H$ is responsible for most of the method's
usable signal (R@10 0.0313 → 0.0541 without/with, **+73%**).

**GDE** (visual-prototype directions: Karcher mean $\mu$ of the train corpus, per-attribute
tangent means, geodesic composition) reached R@5 0.0607 — 65% above CLAY on every query, but still
below vanilla arithmetic.

This is a first-class *finding*, not an implementation failure: both methods optimize semantic
direction quality, which is not this benchmark's objective (identity-constrained retrieval under a
Hamming rule). It empirically validates the assignment's own premise that the naïve pre-SVD stack
is a bottleneck (§1 of the spec) — and it told us where to dig next: GDE's clean *image-space*
directions (no modality gap, no phrasing noise) were worth extending; CLAY's fixed equal-weight
subspace was worth *replacing*.

*Reproduction code lives in the project repository (`src/Baselines/clay.py`, `src/Baselines/gde.py`);
per the scope decision for this report, only the two strongest training-free methods are inlined
below. The cell immediately after this one reads the full $k$/rotation grid from the cached
per-configuration CSVs, so every number above is directly checkable against the underlying run.*


In [ ]:
# == CLAY k / rotation grid (cached CSVs, no live compute) ====================
# Each file's own MEAN row is the exact statistic evaluate_scorer would produce;
# substantiates the k-monotonicity / rotation-H claims in the prose above.
CLAY_SWEEP = {
    "k=5,  rotation H":        OUTPUT_DIR / "Baselines" / "CLAY" / "tier1_CLAY_k5_rotH.csv",
    "k=10, rotation H":        OUTPUT_DIR / "Baselines" / "CLAY" / "tier1_CLAY_k10_rotH.csv",
    "k=20, rotation H":        OUTPUT_DIR / "Baselines" / "CLAY" / "tier1_CLAY_k20_rotH.csv",
    "k=50, rotation H (best)": OUTPUT_DIR / "Baselines" / "CLAY" / "tier1_CLAY_k50_rotH.csv",
    "k=50, no rotation":       OUTPUT_DIR / "Baselines" / "CLAY" / "tier1_CLAY_k50_norot.csv",
}
sweep_summary(CLAY_SWEEP)


## 4.4 Geodesic Visual-Prototype composition (GDE-VP) — negation as rejection

The first of our two best training-free methods extends GDE with the three levers its theory names
but its base implementation leaves unused. The core design decision: **directions are mined on the
train split, ranking happens on the test split** — computing "what Smiling looks like" from the
very images we rank would be evaluation-set leakage.

**Mining.** Karcher (intrinsic) mean $\mu$ of all train images; per attribute $a$, the primitive
direction is the tangent mean $v_a = \frac{1}{|Z_a|}\sum_{x \in Z_a} \mathrm{Log}_\mu(x)$
(GDE Prop. 1) — deliberately *not* normalized, its magnitude encodes average angular displacement.

**Composition** (per query, on the sphere):

$$
q_{\text{tan}} = \mathrm{Log}_\mu(v_{\text{ref}}) + \sum_{a \in T^+} \alpha\, v_a, \qquad
q_{\text{tan}} \leftarrow q_{\text{tan}} - Q\,(Q^\top q_{\text{tan}}), \qquad
q = \mathrm{Exp}_\mu(q_{\text{tan}})
$$

where $Q$ is a thin-QR orthonormal basis of the stacked negated directions. The negation operator
is the conceptual step: **"not X" is not anti-X**. Subtracting a text vector overshoots toward an
anti-concept; *orthogonal rejection* removes the attribute axis entirely — "not red hair" makes
blonde, brown and black all equally acceptable (Oldfield et al. 2023; the affirmation-bias account
of Alhamoud et al. 2025). Joint QR projection (rather than sequential per-attribute rejection) is
order-independent and correct for correlated axes like Male↔Mustache.

**Findings.** $\alpha{=}1.5$ is the clear winner (+19% R@5 over base GDE; R@5 **0.0724**, beating
vanilla arithmetic) — the gains concentrate where the direction is discriminative (`+Male` R@10
0.070→0.159). CLIP-softmax-weighted direction mining (GDE Prop. 2) brought *nothing* (+0.0001):
CelebA face crops are clean, so the denoising weights end up near-uniform. And the hard negation
queries stayed at 0.000 — the joint-QR scores were *identical* to sequential rejection, proving
the wall is a **direction-quality problem, not an operator problem**. The push-strength sweep
($\alpha \in \{0.5, 1.0, 1.5\}$, the same $\alpha$ as the composition formula above) and the
weighted-vs-unweighted mining ablation (GDE Prop. 2) are tabulated from their cached CSVs
immediately after the live run below.


In [ ]:
# == Riemannian primitives on the unit hypersphere S^{d-1} (GDE App. A) =======
def log_map(mu: torch.Tensor, X: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    # log_mu(x) = θ·(x − cosθ·μ)/sinθ, θ = arccos(xᵀμ). Points at μ map to 0.
    dots = (X @ mu).clamp(-1.0, 1.0)
    theta = torch.arccos(dots)
    tangent = X - dots.unsqueeze(1) * mu
    norms = tangent.norm(dim=1, keepdim=True)
    scale = torch.where(norms > eps, theta.unsqueeze(1) / norms, torch.zeros_like(norms))
    return tangent * scale


def exp_map(mu: torch.Tensor, V: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    # exp_mu(v) = cos(‖v‖)·μ + sin(‖v‖)·v̂ ; V is re-projected onto T_mu first.
    V = V - (V @ mu).unsqueeze(1) * mu
    norms = V.norm(dim=1, keepdim=True)
    v_hat = torch.where(norms > eps, V / norms, torch.zeros_like(V))
    return torch.cos(norms) * mu + torch.sin(norms) * v_hat


def intrinsic_mean(X: torch.Tensor, n_iter: int = 50, eps: float = 1e-7) -> torch.Tensor:
    # Karcher mean — μ ← Exp_μ(mean of Log_μ(x_i)) until the tangent gradient vanishes.
    # Warm-started from the normalized Euclidean mean (<20 iters on CelebA-scale data).
    mu = F.normalize(X.mean(dim=0), dim=0)
    for _ in range(n_iter):
        grad = log_map(mu, X).mean(dim=0)
        if grad.norm() < eps:
            break
        mu = F.normalize(exp_map(mu, grad.unsqueeze(0)).squeeze(0), dim=0)
    return mu


def tangent_mean(mu: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    # Primitive direction v_a = mean of Log_mu(has-a images) — NOT normalized, the
    # magnitude carries the average angular displacement (GDE Prop. 1).
    return log_map(mu, X).mean(dim=0)


In [ ]:
# == GDE-VP — mine directions once (train split), cache, compose, evaluate =====
def load_or_mine_directions(force: bool = False):
    # Once-built offline artifact: {mu, [40,512] directions}, mined from TRAIN only.
    cache = ARTIFACTS_DIR / "visual_directions.pt"
    if cache.exists() and not force:
        ckpt = torch.load(cache, weights_only=True)
        return ckpt["mu"].to(DEVICE), ckpt["directions"].to(DEVICE)
    print("Mining visual directions (Karcher mean + tangent means, TRAIN split)...")
    mu = intrinsic_mean(feats_train)
    directions = torch.zeros(N_ATTR, FEATURE_DIM, device=DEVICE)
    for j, name in enumerate(ATTRIBUTE_NAMES):
        has_a = feats_train[attrs_train[:, j].to(DEVICE) > 0.5]
        if has_a.shape[0]:
            directions[j] = tangent_mean(mu, has_a)
        print(f"  [{j + 1:02d}/40] {name}", end="\r")
    torch.save({"mu": mu.cpu(), "directions": directions.cpu()}, cache)
    print(f"\n[built] {cache}")
    return mu, directions


def make_gde_vp_chunk(mu, directions, alpha: float = 1.5, eps: float = 1e-8):
    # q_tan = Log_mu(v_ref) + Σ α·v_a ; joint-QR rejection of span{v̂_b : b∈T⁻}; Exp_mu.
    def score_chunk(srcs, pi, ni, free):
        q_tan = log_map(mu, feats_test[srcs])                       # [S, 512]
        for a in pi:
            q_tan = q_tan + alpha * directions[a]
        neg = [directions[b] / directions[b].norm() for b in ni
               if directions[b].norm() > eps]
        if neg:
            Q, _ = torch.linalg.qr(torch.stack(neg, dim=0).T)       # [512, k]
            q_tan = q_tan - (q_tan @ Q) @ Q.T                       # order-independent
        q = F.normalize(exp_map(mu, q_tan), dim=1)
        return q @ feats_test.T
    return score_chunk


mu_vis, directions_vis = load_or_mine_directions(force=REMINE_DIRECTIONS)
results_gde_vp = evaluate_scorer(make_gde_vp_chunk(mu_vis, directions_vis, alpha=1.5),
                                 gt_list, name="GDE-VP a1.5")
results_table(results_gde_vp).tail(3)


In [ ]:
# == GDE-VP alpha / weighted-mining grid (cached CSVs, no live compute) =======
# alpha=1.0 here IS base GDE (Sec 4.3, same mined directions, single-direction
# rejection); the sweep shows why alpha=1.5 -- not more, not less -- was selected.
GDE_VP_SWEEP = {
    "alpha=0.5":                                     OUTPUT_DIR / "Training-Free" / "GDE_VP" / "tier2b_gde_alpha0.5.csv",
    "alpha=1.0 (= base GDE, Sec 4.3)":                OUTPUT_DIR / "Training-Free" / "GDE_VP" / "tier2b_gde_alpha1.0.csv",
    "alpha=1.5 (winner)":                             OUTPUT_DIR / "Training-Free" / "GDE_VP" / "tier2b_gde_alpha1.5.csv",
    "alpha=1.0, CLIP-softmax-weighted mining (Prop.2)": OUTPUT_DIR / "Training-Free" / "GDE_VP" / "tier2b_gde_weighted.csv",
}
sweep_summary(GDE_VP_SWEEP)


In [ ]:
# == GDE-VP alpha sweep -- R@K across push-strength and weighted-mining (cached) ================
_gvp = sweep_summary(GDE_VP_SWEEP)
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(_gvp))
for j, k in enumerate((1, 5, 10)):
    ax.plot(x, _gvp[f"R@{k}"], marker="o", label=f"R@{k}")
ax.set_xticks(x), ax.set_xticklabels(_gvp.index, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("macro mean"), ax.set_title("GDE-VP -- R@K across the alpha / mining grid")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "visual" / "report_4_4_gdevp_alpha_sweep.png", dpi=120)
plt.show()


## 4.5 Asymmetric subspaces and the subspace-union synthesis

In parallel we pursued the *text-side* subspace branch. **Asymmetric Conditional Subspaces**
split CLAY's polarity-blind stack into separate per-polarity subspaces and scored
$\cos(v_d, v_{\text{src}}) + \text{mean}_a \cos_{S^+_a}(v_d, v_{\text{src}}) - \lambda \sum_b
\|\text{proj}_{S^-_b}(v_d)\|$: an identity anchor, a manifold-aware positive focus, and a
subspace-energy negation penalty. Each **positive** condition $a$ gets its own rank-$k$ subspace
$S^+_a$ from a per-attribute tangent-space SVD — **per-condition**, as opposed to a **stacked**
ablation that pools all positive conditions' prompts into one joint subspace, closer to CLAY's
own naïve merge but still split from the negative side — and $\lambda$ weighs how hard the
negative-subspace energy is penalized. Two lessons: the **identity anchor is decisive** (removing
it, at $k{=}20,\lambda{=}0.1$, collapses R@5 0.039→0.017 ≈ CLAY), and the **negation penalty has
almost no leverage** — its magnitude (~0.02) is an order below the anchor's range, so when the
reference itself carries the negated attribute, the anchor's pull cannot be overcome by an
additive penalty; the table below shows this directly, with R@5 barely moving across a 10×
range of $\lambda$ (0.05 to 0.5). The best configuration found (per-condition, $k{=}50,
\lambda{=}0.1$) is the "Asymmetric subspaces" row cited in the §4.7 training-free ledger.


In [ ]:
# == Track-S (asymmetric subspaces) k / lambda / anchor / stacking grid (cached) =
# One row per already-computed config; substantiates the identity-anchor and
# lambda-penalty claims above, and shows which config is the ledger's headline row.
TRACK_S_SWEEP = {
    "per-condition, anchor, k=5,  lambda=0.1":                     OUTPUT_DIR / "Training-Free" / "Asymmetric_Subspaces" / "tier2a_percond_anchor_k5_5_lam0.1_rotH.csv",
    "per-condition, anchor, k=10, lambda=0.1":                     OUTPUT_DIR / "Training-Free" / "Asymmetric_Subspaces" / "tier2a_percond_anchor_k10_10_lam0.1_rotH.csv",
    "per-condition, anchor, k=20, lambda=0.05":                    OUTPUT_DIR / "Training-Free" / "Asymmetric_Subspaces" / "tier2a_percond_anchor_k20_20_lam0.05_rotH.csv",
    "per-condition, anchor, k=20, lambda=0.1":                     OUTPUT_DIR / "Training-Free" / "Asymmetric_Subspaces" / "tier2a_percond_anchor_k20_20_lam0.1_rotH.csv",
    "per-condition, anchor, k=20, lambda=0.2":                     OUTPUT_DIR / "Training-Free" / "Asymmetric_Subspaces" / "tier2a_percond_anchor_k20_20_lam0.2_rotH.csv",
    "per-condition, anchor, k=20, lambda=0.5":                     OUTPUT_DIR / "Training-Free" / "Asymmetric_Subspaces" / "tier2a_percond_anchor_k20_20_lam0.5_rotH.csv",
    "per-condition, anchor, k=20, lambda=0.1, no rotation H":      OUTPUT_DIR / "Training-Free" / "Asymmetric_Subspaces" / "tier2a_percond_anchor_k20_20_lam0.1_norot.csv",
    "per-condition, anchor, k=50, lambda=0.1 (best, ledger row)":  OUTPUT_DIR / "Training-Free" / "Asymmetric_Subspaces" / "tier2a_percond_anchor_k50_50_lam0.1_rotH.csv",
    "per-condition, NO anchor, k=20, lambda=0.1":                  OUTPUT_DIR / "Training-Free" / "Asymmetric_Subspaces" / "tier2a_percond_noanchor_k20_20_lam0.1_rotH.csv",
    "stacked subspace, anchor, k=20, lambda=0.1":                  OUTPUT_DIR / "Training-Free" / "Asymmetric_Subspaces" / "tier2a_stacked_anchor_k20_20_lam0.1_rotH.csv",
}
sweep_summary(TRACK_S_SWEEP)


The natural synthesis — replace the single mined direction of §4.4 with a full $k$-dimensional
*visual* negative subspace, rejected from the query tangent — was built and swept
($k_{\text{neg}} \in \{1,5,10,20\}$). Two symmetric variants were tried, rejecting either the
*query's* own tangent vector (as in GDE-VP) or, mirroring CLAY's own symmetric formulation, the
*database*'s image features into the same orthogonal complement. Performance is **monotonically
worse with $k_{\text{neg}}$** in both variants (query-side R@5 0.0603→0.0600→0.0577→0.0550 for
$k_{\text{neg}}=1,5,10,20$ — larger subspaces over-reject and wash out identity), and
$k_{\text{neg}}{=}1$ at $\alpha{=}1.0$ nearly reproduces base GDE's single-direction rejection
(§4.3, R@5 0.0607) almost exactly — the $\alpha$ sweep at $k_{\text{neg}}{=}10$ shows the same
push-strength dependence as §4.4, peaking below GDE-VP's own R@5 0.0724 because joint rejection
across a $k$-dim subspace per attribute is not the same operator as GDE-VP's joint-QR rejection of
single directions. Subspace rejection, single-axis rejection, and text-energy penalties all
converge to the same ceiling — all below enhanced arithmetic. This is the result that first told
us the training-free negation axis was *exhausted*, and that something would have to be learned.


In [ ]:
# == Subspace-union k_neg grid -- query-side vs DB-side rejection, + alpha (cached) =
# Confirms "monotonically worse with k" for both rejection sides, and shows the
# k_neg=1 <-> base-GDE correspondence referenced in the prose above.
SUBSPACE_UNION_SWEEP = {
    "query-side reject, k_neg=1,  alpha=1.0": OUTPUT_DIR / "Training-Free" / "Subspace_Union" / "tier2c_query_kneg1_a1.0.csv",
    "query-side reject, k_neg=5,  alpha=1.0": OUTPUT_DIR / "Training-Free" / "Subspace_Union" / "tier2c_query_kneg5_a1.0.csv",
    "query-side reject, k_neg=10, alpha=1.0": OUTPUT_DIR / "Training-Free" / "Subspace_Union" / "tier2c_query_kneg10_a1.0.csv",
    "query-side reject, k_neg=20, alpha=1.0": OUTPUT_DIR / "Training-Free" / "Subspace_Union" / "tier2c_query_kneg20_a1.0.csv",
    "query-side reject, k_neg=10, alpha=0.5": OUTPUT_DIR / "Training-Free" / "Subspace_Union" / "tier2c_query_kneg10_a0.5.csv",
    "query-side reject, k_neg=10, alpha=1.5": OUTPUT_DIR / "Training-Free" / "Subspace_Union" / "tier2c_query_kneg10_a1.5.csv",
    "DB-side reject,    k_neg=1,  alpha=1.0": OUTPUT_DIR / "Training-Free" / "Subspace_Union" / "tier2c_db_kneg1_a1.0.csv",
    "DB-side reject,    k_neg=5,  alpha=1.0": OUTPUT_DIR / "Training-Free" / "Subspace_Union" / "tier2c_db_kneg5_a1.0.csv",
    "DB-side reject,    k_neg=10, alpha=1.0": OUTPUT_DIR / "Training-Free" / "Subspace_Union" / "tier2c_db_kneg10_a1.0.csv",
    "DB-side reject,    k_neg=20, alpha=1.0": OUTPUT_DIR / "Training-Free" / "Subspace_Union" / "tier2c_db_kneg20_a1.0.csv",
}
sweep_summary(SUBSPACE_UNION_SWEEP)


In [ ]:
# == Tier-2c summary -- mean R@1 across the subspace-union (k_neg, rejection side, alpha) grid ===
_t2c = sweep_summary(SUBSPACE_UNION_SWEEP)
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#4C72B0" if "query-side" in n else "#DD8452" for n in _t2c.index]
ax.barh(_t2c.index, _t2c["R@1"], color=colors)
ax.set_xlabel("R@1"), ax.set_title("Tier-2c (subspace-union negation) -- R@1 across the grid")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "visual" / "report_4_5_tier2c_summary.png", dpi=120)
plt.show()


## 4.6 Dynamic Gated Projection (DGP) — replacing the pre-SVD stack head-on

One training-free idea remained, and it was the one aimed at the spec's exact words: CLAY builds
each condition's subspace by an SVD over the prompt stack with **every paraphrase weighted
equally** — query-agnostic and reference-blind. DGP replaces that fixed step with a closed-form,
**reference-conditioned** gate over the same rows — the hand-coded shadow of cross-attention (the
mechanism the spec names as a suggested avenue):

$$
\hat t_i = \mathrm{normalize}(t_i - \mu_{\text{txt}}) \;\;\to\;\;
a_i = \mathrm{softmax}_i\!\left(\tfrac{\langle v_{\text{ref}}, \hat t_i\rangle}{\tau}\right)
\;\;\to\;\;
d_c = \mathrm{normalize}\!\left(\textstyle\sum_i a_i \hat t_i\right)
$$

then geodesic addition of $d_c$ for $T^+$ and row-wise orthogonal rejection for $T^-$ (each
reference gets its *own* gated direction, so the rejection subspace is reference-dependent — the
very dynamism the spec asks for). $\tau$ is the softmax temperature of the gate — small $\tau$
sharpens it toward whichever paraphrase best aligns with the reference, $\tau \to \infty$ makes
it uniform over the prompt stack — and $\alpha$ is the same positive push-strength scalar as
§4.4's GDE-VP, applied here to the gated direction $d_c$ instead of a mined prototype. A rejected
design detour is recorded for honesty: a "modality-reliability fusion" that blended text and
visual rank-1 directions was discarded *before implementation* because it sidesteps the SVD
bottleneck instead of replacing it.

| Config | R@1 | R@5 | R@10 |
|---|---|---|---|
| sharp gate τ=0.07, α=0.5, centered | 0.0252 | 0.0757 | 0.1137 |
| sharp gate τ=0.07, α=1.0, centered | 0.0359 | 0.1141 | 0.1719 |
| sharp gate τ=0.07, α=1.5, centered | 0.0276 | 0.1067 | 0.1649 |
| **uniform gate τ→∞, α=1.0, centered** | **0.0363** | **0.1152** | **0.1777** |
| sharp gate τ=0.07, α=1.0, **no centering** | 0.0246 | 0.0659 | 0.1042 |

The honest reading, which became the hinge of the project: **(1)** centering again does the heavy
lifting — the no-centering row is far worse than every centered configuration in this grid;
**(2)** push-strength is *non-monotone* at fixed $\tau=0.07$ ($\alpha{=}1.0$ beats both $0.5$
and $1.5$, unlike GDE-VP's monotone-to-1.5 profile in §4.4 — the closed-form gate's own
paraphrase-weighting already carries some of the "how far to push" burden that $\alpha$ shoulders
alone in GDE-VP); **(3)** the hand-coded sharp gate *loses* to its own uniform-weight ablation — a
closed-form softmax over-commits to whichever paraphrase happens to align with the reference;
**(4)** yet the re-weighting signal is *real*: per-query, the sharp gate helps some composed
queries (`-Smiling, +Eyeglasses, +Wearing_Hat` R@10 0.380→0.405) and hurts others. The axis
exists; a *fixed* rule cannot exploit it. **That is precisely the evidence that justifies a
learned gate — the subject of the next section.**


In [ ]:
# == DGP — closed-form reference-conditioned gate over the prompt stacks ======
# Step 0: center rows (FIX 1). Step 1-2: a = softmax(v_ref·T̂ᵀ/τ), d = norm(a@T̂).
# Step 3: geodesic add (T⁺) + row-wise rejection (T⁻) at the shared train-mined μ.
# CLIP is never invoked: gate = dot products + weighted mean vs read-only tensors.
_UNPADDED_N = {name: len(build_prompts_for_attribute(name)) for name in ATTRIBUTE_NAMES}


def _attr_stack(name: str) -> torch.Tensor:
    # Strip the bank's padding rows — duplicates would dominate the softmax gate.
    return prompt_bank[ATTR_TO_IDX[name], :_UNPADDED_N[name]]


def compute_mu_txt() -> torch.Tensor:
    # μ_txt = mean of the 40 per-attribute stack means (self-consistent centering frame).
    return torch.stack([_attr_stack(n).mean(dim=0) for n in ATTRIBUTE_NAMES]).mean(dim=0)


MU_TXT = compute_mu_txt()


def _centered_stack(name: str, center: bool = True) -> torch.Tensor:
    T = _attr_stack(name)
    return F.normalize(T - MU_TXT if center else T, dim=1)


def _gated_directions(V_ref: torch.Tensor, T_hat: torch.Tensor, tau: float) -> torch.Tensor:
    # Each reference re-weights the SAME stack differently — the dynamic per-condition
    # re-weighting that replaces SVD's fixed equal-weight truncation. [S, 512].
    a = F.softmax((V_ref @ T_hat.T) / tau, dim=1)
    return F.normalize(a @ T_hat, dim=1)


def make_dgp_chunk(mu, tau: float = 100.0, alpha: float = 1.0, center: bool = True):
    # tau=0.07 -> sharp gate; tau=100 ≈ uniform gate (the winning config — see table).
    def score_chunk(srcs, pi, ni, free):
        V_ref = feats_test[srcs]
        stacks = {j: _centered_stack(ATTRIBUTE_NAMES[j], center) for j in (*pi, *ni)}
        q_tan = log_map(mu, V_ref)
        for j in pi:
            q_tan = q_tan + alpha * _gated_directions(V_ref, stacks[j], tau)
        for j in ni:
            d_c = _gated_directions(V_ref, stacks[j], tau)
            q_tan = q_tan - (q_tan * d_c).sum(dim=1, keepdim=True) * d_c   # row-wise rejection
        q = F.normalize(exp_map(mu, q_tan), dim=1)
        return q @ feats_test.T
    return score_chunk


results_dgp = evaluate_scorer(make_dgp_chunk(mu_vis, tau=100.0), gt_list, name="DGP")
results_table(results_dgp).tail(3)


In [ ]:
# == DGP -- mean R@1 across the tau (gate sharpness) / alpha (push-strength) grid ===============
DGP_SWEEP = {
    "sharp gate tau=0.07, alpha=0.5, centered":    OUTPUT_DIR / "tier2d" / "tier2d_tau0.07_a0.5_center.csv",
    "sharp gate tau=0.07, alpha=1.0, centered":    OUTPUT_DIR / "tier2d" / "tier2d_tau0.07_a1.0_center.csv",
    "sharp gate tau=0.07, alpha=1.5, centered":    OUTPUT_DIR / "tier2d" / "tier2d_tau0.07_a1.5_center.csv",
    "uniform gate tau->inf, alpha=1.0, centered":  OUTPUT_DIR / "tier2d" / "tier2d_tau100.0_a1.0_center.csv",
    "sharp gate tau=0.07, alpha=1.0, no centering": OUTPUT_DIR / "tier2d" / "tier2d_tau0.07_a1.0_nocenter.csv",
}
_dgp_sweep = sweep_summary(DGP_SWEEP)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(len(_dgp_sweep)), _dgp_sweep["R@1"], marker="o", color="#55A868")
ax.set_xticks(range(len(_dgp_sweep)))
ax.set_xticklabels(_dgp_sweep.index, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("R@1"), ax.set_title("DGP -- mean R@1 across the tau / alpha grid")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "visual" / "report_4_6_dgp_tau_alpha_sweep.png", dpi=120)
plt.show()


## 4.7 Training-Free Results

| Method (best config) | R@1 | R@5 | R@10 |
|---|---|---|---|
| CLIP (vanilla arithmetic, α=1) | 0.0224 | 0.0699 | 0.1048 |
| **CLIP Enhanced (all fixes)** | 0.0397 | 0.1140 | 0.1651 |
| CLAY reproduction (k=50, rot H) | 0.0098 | 0.0368 | 0.0541 |
| GDE reproduction | 0.0221 | 0.0607 | 0.0910 |
| Asymmetric subspaces | 0.0137 | 0.0454 | 0.0659 |
| Subspace-union negation | 0.0200 | 0.0699 | 0.1038 |
| **GDE-VP (α=1.5)** | 0.0248 | 0.0724 | 0.1085 |
| **DGP (uniform gate)** | 0.0363 | 0.1152 | 0.1777 |

Four families of subspace surgery and not one meaningfully beat CLIP Enhanced
(best: DGP at R@5 0.1152 vs 0.1140). Within single-vector cosine scoring, **the training-free
design space was exhausted** — the ladder was worth building because it proves the plateau is
structural, not a tuning failure. And `-Male, -Mustache` = 0.000, everywhere.


In [ ]:
# == 4.2 CLIP vs CLIP Enhanced -- mean performance comparison ====================================
_pair = {"CLIP (vanilla)": LEDGER["CLIP (vanilla arithmetic)"],
         "CLIP Enhanced (all fixes)": LEDGER["CLIP Enhanced (all fixes)"]}
fig, ax = plt.subplots(figsize=(6, 4.5))
x = np.arange(3)
width = 0.35
for i, (name, vals) in enumerate(_pair.items()):
    ax.bar(x + i * width, vals, width, label=name)
ax.set_xticks(x + width / 2), ax.set_xticklabels(["R@1", "R@5", "R@10"])
ax.set_ylabel("macro mean"), ax.set_title("CLIP vs CLIP Enhanced")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "visual" / "report_4_2_clip_vs_enhanced.png", dpi=120)
plt.show()


In [ ]:
# == 4.7 Training-free methods -- R@K comparison across every tier ===============================
_TF_ORDER = ["CLIP (vanilla arithmetic)", "CLIP Enhanced (all fixes)", "CLAY reproduction (best)",
             "GDE reproduction", "GDE-VP (alpha=1.5)", "DGP sharp gate (tau=0.07)"]
_tf_ledger = {k: LEDGER[k] for k in _TF_ORDER if k in LEDGER}
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(3)
width = 0.8 / len(_tf_ledger)
for i, (name, vals) in enumerate(_tf_ledger.items()):
    ax.bar(x + i * width, vals, width, label=name)
ax.set_xticks(x + width * (len(_tf_ledger) - 1) / 2), ax.set_xticklabels(["R@1", "R@5", "R@10"])
ax.set_ylabel("macro mean"), ax.set_title("Training-free results -- all tiers")
ax.legend(fontsize=7, loc="upper left")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "visual" / "report_4_7_training_free_comparison.png", dpi=120)
plt.show()


---
# 5. Learned Geometric Fusion

## 5.1 Φ-Gate: cross-attention replaces the closed-form gate

DGP proved the re-weighting axis exists but a fixed rule cannot exploit it. The natural learned
successor keeps DGP's skeleton **fixed where DGP is fixed and learns exactly where DGP hand-codes**:

$$
\begin{aligned}
\text{Step 0 (fixed):}\quad & \hat t_i = \mathrm{normalize}(t_i - \mu_{\text{txt}}) \\
\text{Step 1 (learned):}\quad & h_{\text{ref}} = v_{\text{ref}} + \mathrm{MLP}(v_{\text{ref}}), \qquad
d_a = \mathrm{normalize}\big(\mathrm{CrossAttn}(q{=}h_{\text{ref}},\, kv{=}\hat T_a)\big) \\
\text{Step 3 (fixed):}\quad & q_{\text{tan}} = \mathrm{Log}_\mu(v_{\text{ref}}) + \textstyle\sum_{a \in T^+} \alpha\, d_a;
\quad q_{\text{tan}} \mathrel{-}= (q_{\text{tan}} \cdot d_b)\, d_b \;\; \forall b \in T^-;
\quad q = \mathrm{Exp}_\mu(q_{\text{tan}})
\end{aligned}
$$

Near-identity initialization (zero-init MLP tail) makes untrained Φ ≈ DGP — a clean ablation
anchor. Training data comes from a **synthetic query generator that mirrors the ground-truth rule
verbatim on the train split** (positives = strict ±-satisfaction ∧ Hamming ≤ 2 from the sampled
reference; hard negatives = images satisfying $T^+$ but violating a $T^-$ — exactly the images
naïve arithmetic wrongly retrieves). First trained result (InfoNCE, ~1.3M params, CLIP frozen):
**R@1 0.0383 / R@5 0.1110 / R@10 0.1607** — the learned gate beats the closed-form gate, as
predicted. But the per-query rows exposed two structural residues: negation was still a *fixed*
post-gate operator (`-Male, -Mustache` = 0.000), and InfoNCE's pairwise objective is not
rank-consistent with R@K.

## 5.2 The variation family — isolating each suspect

We built one variation per suspected cause, each changing exactly one lever on the shared
skeleton:

- **ListNet loss** (Cao et al. 2007) instead of InfoNCE — gradients flow through the *full ranked
  list*, making the objective metric-consistent with R@K/P@K.
- **Learned NegationHead** — replaces fixed rejection with a learned suppression coefficient
  $\lambda(q_{\text{tan}}, d_b) \in (0, 2)$; zero-init gives $\lambda{=}1$ = exact rejection, so
  the model can only *improve* on the fixed operator.
- **Composer** — the architectural leap: drop the arithmetic skeleton entirely; a 2-layer
  transformer over the token sequence $[\mathrm{CLS}, v_{\text{ref}}, t^+_1 \dots, t^-_1 \dots]$
  (with polarity embeddings) emits a residual on $v_{\text{ref}}$. Its mean collapsed
  (R@5 0.0233 — a from-scratch transformer under-trains in 30 epochs) **yet it was the only
  method in the entire project to break `-Male, -Mustache`** (R@10 0.111): the first hard evidence
  that a *learned* operator can express what fixed rejection cannot.
- **Symmetric CLAY negation / negation steering / gated negation** — three further training-free
  negation attempts, evaluated and then pruned once they confirmed the closed-form negation axis
  was exhausted (kept in git history as the ablation record).

## 5.3 Φ-Combined — the best geometric model

The winning levers folded into one model: **ListNet + NegationHead + distance-based hard-negative
mining** (among attribute-logic hard negatives, keep the K closest to the *current* query — online
FaceNet-style curriculum) **+ an attribute-disentanglement auxiliary loss**
$L_{\text{dis}} = \frac{1}{|\text{pairs}|}\sum_{(a,b)} |\cos(d_a, d_b)|$ that pushes attribute
directions apart, attacking CLIP's Male↔Mustache entanglement so rejection is geometrically
cleaner. Total loss $L = L_{\text{ListNet}} + \lambda_{\text{dis}} L_{\text{dis}}$.


In [ ]:
# == Synthetic query generator — mirrors the GT rule on the TRAIN split ========
# Positives: strict T⁺/T⁻ satisfaction AND Hamming ≤ 2 on free attrs vs the sampled
# reference (identical to spec §3.1.1 — no test data anywhere). Hard negatives:
# satisfy T⁺ but violate ≥1 T⁻ — the only negatives with a meaningful gradient.
# Reused verbatim by every trained model in this report (Φ-Combined, A-BAP).
from dataclasses import dataclass, field, replace


@dataclass(frozen=True)
class GenConfig:
    k_pos_max: int = 3
    k_neg_max: int = 2
    n_pos_targets: int = 8
    n_hard_neg: int = 32
    hamming_max: int = 2
    seed: int = SEED


class QueryGenerator:
    def __init__(self, train_attrs: torch.Tensor, cfg: GenConfig):
        self.A = train_attrs                                 # [N, 40] in {0,1}, CPU
        self.cfg = cfg
        self.N, self.n_attr = train_attrs.shape
        self.g = torch.Generator().manual_seed(cfg.seed)

    def _rand(self, hi: int) -> int:
        return int(torch.randint(hi, (1,), generator=self.g).item())

    def _choice(self, pool: list[int], k: int) -> list[int]:
        perm = torch.randperm(len(pool), generator=self.g)[:k]
        return [pool[i] for i in perm.tolist()]

    def _take(self, ids: torch.Tensor, k: int) -> torch.Tensor:
        if ids.numel() <= k:
            return ids
        return ids[torch.randperm(ids.numel(), generator=self.g)[:k]]

    def sample_query(self):
        # One synthetic query, or None when it has too few positives to teach anything.
        r = self._rand(self.N)
        has = self.A[r] > 0.5
        has_idx = has.nonzero(as_tuple=True)[0].tolist()
        lacks_idx = (~has).nonzero(as_tuple=True)[0].tolist()
        if not has_idx or not lacks_idx:
            return None
        pos_attrs = self._choice(has_idx, min(1 + self._rand(self.cfg.k_pos_max), len(has_idx)))
        neg_attrs = self._choice(lacks_idx, min(self._rand(self.cfg.k_neg_max + 1), len(lacks_idx)))

        sat = torch.ones(self.N, dtype=torch.bool)
        for a in pos_attrs:
            sat &= self.A[:, a] > 0.5
        for b in neg_attrs:
            sat &= self.A[:, b] < 0.5
        free = torch.tensor([c for c in range(self.n_attr)
                             if c not in set(pos_attrs) | set(neg_attrs)])
        ham = (self.A[:, free] != self.A[r, free]).sum(dim=1)
        pos_mask = sat & (ham <= self.cfg.hamming_max)
        pos_mask[r] = False                                  # never its own target
        pos_ids = pos_mask.nonzero(as_tuple=True)[0]
        if pos_ids.numel() < 1:
            return None

        sat_pos = torch.ones(self.N, dtype=torch.bool)
        for a in pos_attrs:
            sat_pos &= self.A[:, a] > 0.5
        viol = torch.zeros(self.N, dtype=torch.bool)
        for b in neg_attrs:
            viol |= self.A[:, b] > 0.5
        hard_ids = (sat_pos & viol).nonzero(as_tuple=True)[0]
        if neg_attrs and hard_ids.numel() < 1:
            return None
        return (r, [ATTRIBUTE_NAMES[a] for a in pos_attrs],
                [ATTRIBUTE_NAMES[b] for b in neg_attrs],
                self._take(pos_ids, self.cfg.n_pos_targets),
                self._take(hard_ids, self.cfg.n_hard_neg) if hard_ids.numel() else hard_ids)


In [ ]:
# == Φ-Gate + NegationHead — the Φ-Combined architecture =======================
@dataclass(frozen=True)
class PhiGeoConfig:
    d_model: int = 512
    n_heads: int = 4
    alpha: float = 1.0
    lr: float = 1e-4
    weight_decay: float = 1e-2
    tau: float = 0.07              # ListNet temperature on cosine-scale scores
    epochs: int = 30
    batch_queries: int = 64
    n_hard_neg_final: int = 16     # kept after distance-based reranking
    lambda_dis: float = 0.1
    n_dis_pairs: int = 8
    seed: int = SEED


RAW_STACKS = {name: _attr_stack(name) for name in ATTRIBUTE_NAMES}   # uncentered


class FusionPhi(nn.Module):
    # Learned gated projection — cross-attention replaces DGP's closed-form gate.
    # Geometry (Step-0 centering, Step-3 geodesic add + rejection + Exp_mu) is fixed;
    # only the per-attribute direction is learned. Zero-init MLP tail => untrained ≈ DGP.
    def __init__(self, cfg: PhiGeoConfig, mu: torch.Tensor, mu_txt: torch.Tensor):
        super().__init__()
        self.cfg = cfg
        self.mlp_ref = nn.Sequential(
            nn.Linear(cfg.d_model, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, cfg.d_model),
        )
        nn.init.zeros_(self.mlp_ref[-1].weight)
        nn.init.zeros_(self.mlp_ref[-1].bias)
        self.attn = nn.MultiheadAttention(cfg.d_model, cfg.n_heads, batch_first=True)
        self.register_buffer("mu", mu)
        self.register_buffer("mu_txt", mu_txt)

    def _center(self, T: torch.Tensor) -> torch.Tensor:
        return F.normalize(T - self.mu_txt, dim=-1)

    def _direction(self, h_ref: torch.Tensor, T_hat: torch.Tensor) -> torch.Tensor:
        # Single-query attention over one attribute's paraphrase stack — the learned
        # analogue of DGP's softmax gate. [B, d].
        q = h_ref.unsqueeze(1)
        kv = T_hat.unsqueeze(0).expand(h_ref.shape[0], -1, -1)
        out, _ = self.attn(q, kv, kv)
        return F.normalize(out.squeeze(1), dim=-1)


class NegationHead(nn.Module):
    # λ(q_tan, d_b) ∈ (0, 2); zero-init tail => λ=1 = exact orthogonal rejection, so
    # the learned operator starts AT the fixed one and can only move if data demands.
    def __init__(self, d_model: int = 512):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model * 3 + 1, 256), nn.GELU(),
            nn.Linear(256, 64), nn.GELU(), nn.Linear(64, 1),
        )
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.constant_(self.mlp[-1].bias, 0.0)

    def forward(self, q_tan: torch.Tensor, d_b: torch.Tensor) -> torch.Tensor:
        dot = (q_tan * d_b).sum(dim=1, keepdim=True)
        return torch.sigmoid(self.mlp(torch.cat([q_tan, d_b, q_tan * d_b, dot], dim=1))) * 2.0


class FusionPhiNeg(nn.Module):
    # Φ-Combined forward pass: attention gate for T⁺, learned-λ rejection for T⁻.
    def __init__(self, cfg: PhiGeoConfig, mu: torch.Tensor, mu_txt: torch.Tensor):
        super().__init__()
        self.phi = FusionPhi(cfg, mu, mu_txt)
        self.neg_head = NegationHead(cfg.d_model)

    def forward(self, v_ref, T_pos_stacks, T_neg_stacks) -> torch.Tensor:
        cfg, mu = self.phi.cfg, self.phi.mu
        h_ref = v_ref + self.phi.mlp_ref(v_ref)
        q_tan = log_map(mu, v_ref)
        for T_a in T_pos_stacks:
            q_tan = q_tan + cfg.alpha * self.phi._direction(h_ref, self.phi._center(T_a))
        for T_b in T_neg_stacks:
            d_b = self.phi._direction(h_ref, self.phi._center(T_b))
            q_tan = q_tan - self.neg_head(q_tan, d_b) * d_b
        return F.normalize(exp_map(mu, q_tan), dim=1)


In [ ]:
# == Φ-Combined losses + mining utilities ======================================
def listnet_loss(q: torch.Tensor, pos: torch.Tensor, neg: torch.Tensor, tau: float):
    # ListNet — cross-entropy to the ideal permutation distribution, y = 1/|pos| on
    # positives. Differentiates through the FULL ranked list (unlike InfoNCE's pairs).
    all_vecs = torch.cat([pos, neg], dim=0)
    scores = (all_vecs @ q) / tau
    labels = torch.zeros(scores.shape[0], device=q.device)
    labels[:pos.shape[0]] = 1.0 / pos.shape[0]
    return -(labels * F.log_softmax(scores, dim=0)).sum()


def distance_mine(q: torch.Tensor, hard_ids: torch.Tensor, train_feats, k: int):
    # Among attribute-logic hard negatives keep the k highest-cosine to the CURRENT
    # query — the images the model confuses most right now (FaceNet-style curriculum).
    if hard_ids.numel() <= k:
        return hard_ids
    sims = train_feats[hard_ids] @ q
    return hard_ids[torch.topk(sims, k).indices]


def disentanglement_loss(model: FusionPhiNeg, train_feats, n_pairs: int, rng):
    # L_dis = mean |cos(d_a, d_b)| over sampled attribute pairs — pushes attribute
    # directions apart so negation is geometrically cleaner (Male↔Mustache).
    v_ref = train_feats[rng.randint(0, train_feats.shape[0] - 1)].unsqueeze(0)
    h_ref = v_ref + model.phi.mlp_ref(v_ref)
    names = rng.sample(ATTRIBUTE_NAMES, min(n_pairs * 2, N_ATTR))
    dirs = [F.normalize(model.phi._direction(h_ref, model.phi._center(RAW_STACKS[n]))
                        .squeeze(0), dim=0) for n in names]
    loss, tried = torch.zeros((), device=v_ref.device), 0
    for i in range(len(dirs)):
        for j in range(i + 1, len(dirs)):
            loss = loss + (dirs[i] * dirs[j]).sum().abs()
            tried += 1
            if tried >= n_pairs:
                return loss / tried
    return loss / max(tried, 1)


In [ ]:
# == Φ-Combined training (flag-guarded: load weights or retrain) ===============
COMBINED_WEIGHTS = OUTPUT_DIR / "Trained" / "Phi_Combined" / "tier3_combined_phi.pt"


def train_combined(cfg: PhiGeoConfig = PhiGeoConfig()) -> FusionPhiNeg:
    # ListNet + NegationHead + online mining + disentanglement, on synthetic queries.
    torch.manual_seed(cfg.seed)
    rng = random.Random(cfg.seed)
    model = FusionPhiNeg(cfg, mu_vis, MU_TXT).to(DEVICE)
    gen = QueryGenerator(attrs_train, GenConfig(seed=cfg.seed))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    steps_per_epoch = max(1, gen.N // (cfg.batch_queries * 50))
    for epoch in range(cfg.epochs):
        model.train()
        running, seen = 0.0, 0
        for _ in range(steps_per_epoch):
            opt.zero_grad()
            batch_rank, n = torch.zeros((), device=DEVICE), 0
            for _ in range(cfg.batch_queries):
                sample = gen.sample_query()
                if sample is None:
                    continue
                r, pos_names, neg_names, pos_ids, hard_ids = sample
                q = model(feats_train[r].unsqueeze(0),
                          [RAW_STACKS[nm] for nm in pos_names],
                          [RAW_STACKS[nm] for nm in neg_names]).squeeze(0)
                if hard_ids.numel():
                    with torch.no_grad():
                        hard_ids = distance_mine(q.detach(), hard_ids.to(DEVICE),
                                                 feats_train, cfg.n_hard_neg_final)
                pos = feats_train[pos_ids.to(DEVICE)]
                neg = feats_train[hard_ids.to(DEVICE)] if hard_ids.numel() else feats_train[:0]
                batch_rank = batch_rank + listnet_loss(q, pos, neg, cfg.tau)
                n += 1
            if n == 0:
                continue
            loss = batch_rank / n + cfg.lambda_dis * disentanglement_loss(
                model, feats_train, cfg.n_dis_pairs, rng)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            running += (batch_rank / n).item() * n
            seen += n
        sched.step()
        print(f"  epoch {epoch + 1:02d}/{cfg.epochs}  ListNet={running / max(seen, 1):.4f}")
    return model


# Training is strictly opt-in: a plain Run-All never launches a GPU run by surprise.
# No weights + flag off => the model is skipped and the report relies on the cached
# result tables (weights are optional per the delivery contract).
cfg_combined = PhiGeoConfig()
phi_combined = None
if RETRAIN_COMBINED:
    phi_combined = train_combined(cfg_combined)
    torch.save(phi_combined.state_dict(), COMBINED_WEIGHTS)
    print(f"[trained+saved] {COMBINED_WEIGHTS}")
elif COMBINED_WEIGHTS.exists():
    phi_combined = FusionPhiNeg(cfg_combined, mu_vis, MU_TXT).to(DEVICE)
    phi_combined.load_state_dict(torch.load(COMBINED_WEIGHTS, map_location=DEVICE))
    print(f"[loaded] {COMBINED_WEIGHTS}")
else:
    print(f"[skipped] no weights at {COMBINED_WEIGHTS} — set RETRAIN_COMBINED=True "
          "(~45 min GPU) to train; reported numbers come from the cached CSVs (section 9)")


def make_combined_chunk(model: FusionPhiNeg):
    model.eval()

    @torch.no_grad()
    def score_chunk(srcs, pi, ni, free):
        q = model(feats_test[srcs],
                  [RAW_STACKS[ATTRIBUTE_NAMES[j]] for j in pi],
                  [RAW_STACKS[ATTRIBUTE_NAMES[j]] for j in ni])
        return q @ feats_test.T
    return score_chunk


if phi_combined is not None:
    results_combined = evaluate_scorer(make_combined_chunk(phi_combined), gt_list,
                                       name="Phi-Combined")
    display(results_table(results_combined).tail(3))


## 5.4 Learned Geometric Fusion Results: two failed detours that pointed the way

| Method | R@1 | R@5 | R@10 |
|---|---|---|---|
| Φ-Gate (InfoNCE) | 0.0383 | 0.1110 | 0.1607 |
| Cross-attention variant | 0.0311 | 0.1038 | 0.1607 |
| Composer (transformer) | 0.0052 | 0.0233 | 0.0375 |
| Φ-Combined | 0.0563 | 0.1576 | 0.2329 |
| **Dual Score** | **0.0597** | **0.1749** | **0.2557** |

Φ-Combined (+47% R@1 / +42% R@5 over the trained gate) was the best *pure-geometry* model; the
last variant of the family then surpassed it. **Dual Score** keeps the geodesic positive fusion
as a cosine channel and adds two *calibrated attribute-probe channels* — per-image
presence/absence probabilities rewarded for every constraint — with query-conditioned channel
weights trained jointly by ListNet. At R@1 0.0597 / R@5 0.1749 / R@10 0.2557 it is the
strongest single-vector-plus-channels model of the project, and the first clear evidence that
*probes carry the signal geometry lacks*. But the per-query rows of both models carry the same
message: `-Male, -Mustache` = **0.0000** for Φ-Combined and barely off the floor for Dual Score
(R@10 0.037), and every negation query sits near the bottom. The wall was now understood as two distinct problems: an *operator*
problem (the Composer proved a learned operator can break it, but at the cost of everything else)
and a *representation* problem (27 sources, cross-gender attribute-twins — the CLIP-cosine
identity anchor actively fights the valid targets).

Two further attempts closed this line of experimentation and are recorded because they *pointed at the answer*:

- **Hybrid probe-penalty negation** — Dual Score's single-global-β precursor. Bolting a linear-probe penalty for negated attributes onto
  cosine scoring weakly broke the floor (`-Male, -Mustache` R@10 0.037) but degraded the mean
  (R@5 0.1256 < Combined) and zeroed the lipstick composite. *Right ingredient — probes — wrong
  integration: an additive penalty on a cosine core, with no identity term.*
- **LingoCIR** (textual inversion: learn $v_{\text{ref}} \to$ pseudo-token, compose the query in
  language, let CLIP's own text encoder be Φ, antonym-rewrite the negations). Elegant on paper;
  results were poor — fine-grained identity does not survive one pseudo-token.

At this point we stopped iterating on scoring machinery and re-read the benchmark itself.


In [ ]:
# == 5.1 DGP (training-free) vs Phi-Gate (its learned successor) ================================
_pair = {"DGP (training-free, uniform gate)": (0.0363, 0.1152, 0.1777),
         "Phi-Gate (learned, InfoNCE)": (0.0383, 0.1110, 0.1607)}
fig, ax = plt.subplots(figsize=(6, 4.5))
x = np.arange(3)
width = 0.35
for i, (name, vals) in enumerate(_pair.items()):
    ax.bar(x + i * width, vals, width, label=name)
ax.set_xticks(x + width / 2), ax.set_xticklabels(["R@1", "R@5", "R@10"])
ax.set_ylabel("macro mean"), ax.set_title("DGP vs its learned successor Phi-Gate")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "visual" / "report_5_1_dgp_vs_phigate.png", dpi=120)
plt.show()


In [ ]:
# == 5.4 Learned geometric fusion -- R@K comparison (Composer excluded, see prose) ===============
# Composer omitted deliberately: at R@1 0.0052 it is two orders of magnitude below every other
# row and would compress the other bars to illegibility -- its failure is discussed in text instead.
_lf_ledger = {
    "Phi-Gate (InfoNCE)": (0.0383, 0.1110, 0.1607),
    "Cross-attention variant": (0.0311, 0.1038, 0.1607),
    "Phi-Combined": (0.0563, 0.1576, 0.2329),
    "Dual Score": (0.0597, 0.1749, 0.2557),
}
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(3)
width = 0.8 / len(_lf_ledger)
for i, (name, vals) in enumerate(_lf_ledger.items()):
    ax.bar(x + i * width, vals, width, label=name)
ax.set_xticks(x + width * (len(_lf_ledger) - 1) / 2), ax.set_xticklabels(["R@1", "R@5", "R@10"])
ax.set_ylabel("macro mean"), ax.set_title("Learned geometric fusion -- R@K (Composer excluded)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "visual" / "report_5_4_learned_fusion_comparison.png", dpi=120)
plt.show()


---
# 6. The Probabilistic Pivot: Bayesian Attribute-Posterior Scoring (BAP)

## 6.1 Diagnosis: three structural bottlenecks

Everything in the training-free and learned geometric fusion sections above shares one silent assumption: the query must be **a single 512-d vector
scored by cosine**, and identity must be expressed as **CLIP-similarity to $v_{\text{ref}}$**.
Both are wrong *for this benchmark*:

1. **The single-vector cosine bottleneck.** Cosine against one point is a rank-1 linear
   functional; the valid-target set (§3.1) is an intersection of ~40 attribute half-spaces with a
   Hamming ball. No Φ compressing everything into one CLIP vector can represent it — no matter
   how it is trained. Evidence: ListNet, learned λ, subspace unions, symmetric complements all
   landed within a few points of each other.
2. **Identity ≠ CLIP cosine.** The benchmark defines identity as *attribute agreement*
   (Hamming ≤ 2). On `-Male, -Mustache` the two diverge adversarially: valid targets are female
   attribute-twins that cosine ranks low *because* they changed gender. This — not weak rejection
   geometry — is the root cause of the 0.000 floor (it survived every rejection variant).
3. **Uncalibrated geometry.** $\cos(x, t_{\text{Male}})$ lives in a narrow, per-attribute-shifted
   band; rejection and subtraction operate on these raw quantities. Linear probes turn the *same
   frozen features* into calibrated posteriors $p_a(x) \in [0,1]$ — the correct currency for
   "has / does not have".

The GT rule is *fully known* (§3.1) — we had even reproduced it inside `QueryGenerator` to make
training data, then asked a cosine scorer to rediscover it implicitly. The optimal move is to
score it **explicitly**: the benchmark is, mathematically, an attribute-inference problem with
CLIP features as the observation. A side effect dissolves the modality gap we fought for weeks:
in probability space, text only picks *which probe channels to gate*; scoring happens in
$[0,1]^{40}$ where both modalities are on the same scale. No rotation, no centering, no gap.

## 6.2 Attribute probes

40 logistic probes (one `Linear(512→40)`, BCE) on the **train split's** frozen features:
$p_a(x) = \sigma(w_a^\top v_x + b_a)$, clamped to $[10^{-4}, 1-10^{-4}]$ so log-channels stay
finite. Mean test accuracy **88.5%** (min 69.6%, `Big_Lips`). Ten CPU-seconds to train; cached and
frozen like every other artifact.

**Integrity invariant (holds everywhere below):** test-split attribute labels are *never* used in
scoring. The reference's attributes are *estimated by the probes*; the eval JSON is read only to
compute metrics. (Test labels appear once, in a printed probe-accuracy diagnostic.)


In [ ]:
# == Stage 0 — attribute probes (frozen features → posteriors) =================
EPS = 1e-4          # posterior clamp keeps log p / log(1−p) finite (≈ ±9.2)
TAIL_FLOOR = 1e-30  # numerical guard on P(K≤2) before log — never rank-relevant

PROBE_EPOCHS, PROBE_LR, PROBE_WD, PROBE_BATCH = 10, 1e-2, 1e-5, 8192


def train_probes() -> nn.Linear:
    # 40 logistic probes as one Linear(512→40), BCE on frozen TRAIN features — turns
    # raw geometry into the calibrated currency the Bayes score needs.
    Y = attrs_train.float().to(DEVICE)
    probe = nn.Linear(FEATURE_DIM, N_ATTR).to(DEVICE)
    opt = torch.optim.Adam(probe.parameters(), lr=PROBE_LR, weight_decay=PROBE_WD)
    lossf = nn.BCEWithLogitsLoss()
    for epoch in range(PROBE_EPOCHS):
        perm = torch.randperm(feats_train.shape[0], device=DEVICE)
        tot = 0.0
        for lo in range(0, feats_train.shape[0], PROBE_BATCH):
            idx = perm[lo:lo + PROBE_BATCH]
            opt.zero_grad()
            loss = lossf(probe(feats_train[idx]), Y[idx])
            loss.backward(); opt.step()
            tot += loss.item() * idx.numel()
        print(f"  probe epoch {epoch + 1:02d}/{PROBE_EPOCHS}  BCE={tot / feats_train.shape[0]:.4f}")
    return probe


def load_or_train_probes(force: bool = False) -> nn.Linear:
    path = ARTIFACTS_DIR / "bap_probe.pt"          # name kept for artifact-cache compat
    probe = nn.Linear(FEATURE_DIM, N_ATTR)
    if path.exists() and not force:
        probe.load_state_dict(torch.load(path, map_location="cpu"))
        print(f"[loaded] {path}")
    else:
        probe = train_probes().cpu()
        torch.save(probe.state_dict(), path)
        print(f"[trained+saved] {path}")
    return probe.to(DEVICE)


@torch.no_grad()
def posteriors(probe: nn.Linear, feats: torch.Tensor) -> torch.Tensor:
    # p(x) = σ(probe(v_x)), clamped so the log-channels stay finite. [N, 40].
    return torch.sigmoid(probe(feats)).clamp(EPS, 1 - EPS)


probe = load_or_train_probes(force=RETRAIN_PROBES)
P_test_raw = posteriors(probe, feats_test)
P_train_raw = posteriors(probe, feats_train)

# DIAGNOSTIC ONLY — reads test labels to report probe quality; never used in scoring.
acc = ((P_test_raw.cpu() > 0.5).float() == attrs_test.float()).float().mean(0)
print(f"probe test acc: mean {acc.mean():.4f}  min {acc.min():.4f} "
      f"({ATTRIBUTE_NAMES[int(acc.argmin())]})")


## 6.3 The closed-form BAP score

Under an attribute-independence approximation, the Bayes-optimal ranking score for the generative
rule of §3.1, given probe posteriors, is (log domain):

$$
S(x \mid \text{ref}, T^+, T^-) =
\underbrace{\sum_{a \in T^+} \log p_a(x) + \sum_{b \in T^-} \log\big(1 - p_b(x)\big)}_{\text{constraint log-likelihood}}
+ \underbrace{\log P\!\Big(\textstyle\sum_{c \in \mathcal{F}} D_c \le 2\Big)}_{\text{identity term}}
$$

with per-attribute disagreement events
$D_c \sim \mathrm{Bern}(d_c)$, $d_c = p_c(x)(1 - p_c(\text{ref})) + (1 - p_c(x))\,p_c(\text{ref})$.
$\sum_c D_c$ is **Poisson-binomial**; its tail is computed *exactly* by a 3-state dynamic program
over the 38 free attributes, vectorized over the corpus:
$s_k \leftarrow s_k(1 - d_c) + s_{k-1} d_c$ (for $k = 2, 1$), $s_0 \leftarrow s_0(1 - d_c)$.

The identity term is the ingredient no geometric method had: it expresses "same person, up to two
free attributes" in the benchmark's own currency. On `-Male, -Mustache` it ranks the reference's
*female attribute-twins* first — which CLIP-cosine structurally cannot. The reference's posterior
$p(\text{ref})$ comes from the probes on $v_{\text{ref}}$, never from labels.


In [ ]:
# == BAP core — exact Poisson-binomial identity tail + closed-form scorer ======
def hamming_tail_log(p_ref: torch.Tensor, p_db: torch.Tensor,
                     hamming_max: int = 2) -> torch.Tensor:
    # log P(Σ_c D_c ≤ hamming_max), D_c ~ Bern(p_db·(1−p_ref) + (1−p_db)·p_ref).
    # Exact Poisson-binomial tail via a (hamming_max+1)-state DP over free attrs —
    # the identity term, and where most of the negation lift comes from.
    # p_ref [S, F], p_db [N, F] -> [S, N].
    S, F_ = p_ref.shape
    N = p_db.shape[0]
    states = [torch.ones(S, N, device=p_db.device)]
    states += [torch.zeros(S, N, device=p_db.device) for _ in range(hamming_max)]
    for c in range(F_):
        d = p_db[:, c].unsqueeze(0) * (1 - p_ref[:, c]).unsqueeze(1) \
            + (1 - p_db[:, c]).unsqueeze(0) * p_ref[:, c].unsqueeze(1)   # [S, N]
        omd = 1 - d
        for k in range(hamming_max, 0, -1):        # descend so states[k-1] is pre-update
            states[k] = states[k] * omd + states[k - 1] * d
        states[0] = states[0] * omd
    return sum(states).clamp_min(TAIL_FLOOR).log()


def make_bap_chunk(P: torch.Tensor, hamming_max: int = 2):
    # Closed-form BAP: constraint log-likelihood + identity tail. Training-free
    # given probes; no CLIP channel.
    logP, log1mP = P.log(), (1 - P).log()

    def score_chunk(srcs, pi, ni, free):
        constraint = logP[:, pi].sum(1) + log1mP[:, ni].sum(1)          # [N]
        tail = hamming_tail_log(P[srcs][:, free], P[:, free], hamming_max)
        return constraint.unsqueeze(0) + tail
    return score_chunk


assert_db_frozen(feats_test, make_bap_chunk(P_test_raw), gt_list)
results_bap = evaluate_scorer(make_bap_chunk(P_test_raw), gt_list, name="BAP")
results_table(results_bap)


## 6.4 What the closed form proves and where it fails

**R@1 0.0868 / R@5 0.2596 / R@10 0.3554 — +48% mean R@5 over Dual Score (the strongest single-vector model), with zero training beyond
ten CPU-seconds of probe fitting.** The negation floor is broken for the first time by a method
with an intact mean: `-Male, -Mustache` 0.000 → **0.222** R@5; `+Chubby, -Young` 0.009 → 0.146
(16×); `-Young` ×6.5; `-Heavy_Makeup` ×3.8.

Three queries lose, and each loss is *diagnostic*:

| Query | BAP R@5 | Φ-Combined R@5 | Why |
|---|---|---|---|
| `+Smiling` | 0.163 | 0.187 | broad attribute — CLIP cosine carries real tie-breaking signal BAP discards |
| `+Mustache` | 0.123 | 0.206 | rare attribute (~4% positive) — probe miscalibration amplified by log-scoring |
| `+Lipstick, -Makeup, +Smiling` | **0.000** | 0.088 | correlation-violating combination — under independence, $p(\text{Lip}){\cdot}(1{-}p(\text{Makeup}))$ is structurally wrong because the probes co-fire |

These three failure modes — a missing visual channel, miscalibration, attribute dependence —
define the refinement agenda of §7 and the learned stage of §8. The lipstick composite becomes the
report's central worked example.


---
# 7. BAP+ — Correlation-Aware Refinements

Each of BAP's diagnosed failure modes gets exactly one component; each defaults to a setting that
reproduces plain BAP **bit-for-bit** (a hard parity invariant asserted below), so every delta in
the ablation table is attributable to one change.

**C1 — Calibrated probes** (temperature scaling, Guo et al. 2017). BCE sigmoids are used *as if*
calibrated, but modern classifiers are systematically over-confident, and log-likelihood scoring
amplifies the error unboundedly. Fit one $\tau_a$ per attribute by NLL on a held-out train fold:
$p_a(x) = \sigma(z_a / \tau_a)$; report ECE before/after.

**C2 — Chow–Liu dependency tree** (Chow & Liu 1968; TAN lineage). CelebA attributes are strongly
dependent — Lipstick↔Makeup co-fire, which is exactly why BAP zeroes the lipstick composite. Learn
the maximum-spanning tree over pairwise mutual information (counted **once, offline, on train
labels**), then replace the independent constraint product with the tree-factorized joint under
probe soft evidence, computed by exact sum-product over the 40 binary nodes:

$$
\text{constraint}(x) = \log p_T\!\big(A_{T^+}{=}\mathbf 1, A_{T^-}{=}\mathbf 0 \mid
\text{evidence} = p(x)\big), \qquad p_T(A) = \prod_a p\big(A_a \mid A_{\pi(a)}\big)
$$

The soft evidence enters as scaled likelihoods $u_a(s) = \log q_a(s) - \log \pi_a(s)$ — dividing
by the train prior makes sum-product over an *edgeless* forest collapse to the independent
product, which is the algebra behind the parity invariant.

**C3 — Conditioned Hamming tail.** The Poisson-binomial DP stays (exact given marginals); what
changes is the marginals it consumes: one belief-propagation pass conditions the free attributes'
posteriors on the constrained assignment (conditioning on `Male=0` shifts the
Beard/Mustache/Sideburns priors), then the unchanged DP runs on the conditioned marginals.

**C4 — λ-cosine identity channel** and **C5 — QB-Norm** (Bogolin et al. 2022) are two further
training-free candidates — a single global CLIP-cosine tie-breaker (the non-learned counterpart of
a learned visual channel) and hubness normalization via a querybank of synthetic train-query score
vectors. Both scalars are grid-tuned **on synthetic train queries only**. Spoiler from the
measured ablation (§9): *both are rejected on evidence* — kept here because rejecting plausible
refinements is part of the validation.


In [ ]:
# == C1 — per-attribute temperature scaling + ECE diagnostic ===================
CALIB_HOLDOUT, CALIB_SEED, CALIB_STEPS, CALIB_LR, ECE_BINS = 0.2, 7, 300, 0.05, 15


def fit_temperature_scalars(logits, labels, steps: int, lr: float) -> torch.Tensor:
    # τ = argmin_τ NLL(σ(z/τ), y) on held-out logits; one scalar per attribute,
    # optimized jointly in log-space (independent terms of one mean-BCE objective).
    log_tau = torch.zeros(logits.shape[1], device=logits.device, requires_grad=True)
    opt = torch.optim.Adam([log_tau], lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(logits / log_tau.exp(), labels)
        loss.backward(); opt.step()
    return log_tau.detach().exp()


def expected_calibration_error(p, y, n_bins: int) -> torch.Tensor:
    # Per-attribute reliability-diagram gap: Σ_b (n_b/M)·|E[y|b] − E[p|b]|.
    ece = torch.zeros(p.shape[1], device=p.device)
    edges = torch.linspace(0.0, 1.0, n_bins + 1, device=p.device)
    for b in range(n_bins):
        in_bin = (p > edges[b]) & (p <= edges[b + 1]) if b else (p <= edges[1])
        n_b = in_bin.float().sum(0)
        gap = ((y * in_bin).sum(0) - (p * in_bin).sum(0)).abs()
        ece += torch.where(n_b > 0, gap / p.shape[0], torch.zeros_like(gap))
    return ece


@torch.no_grad()
def calibrated_posteriors(probe, feats, tau) -> torch.Tensor:
    # tau=None ⇒ bit-identical to posteriors() — the C1-off arm of the invariant.
    if tau is None:
        return posteriors(probe, feats)
    return torch.sigmoid(probe(feats) / tau).clamp(EPS, 1 - EPS)


def load_or_fit_temperatures(force: bool = False) -> torch.Tensor:
    path = ARTIFACTS_DIR / "bap_plus_temperatures.pt"    # cache-compat name
    if path.exists() and not force:
        ckpt = torch.load(path, map_location="cpu")
        print(f"[loaded] {path}  (mean ECE {ckpt['ece_before'].mean():.4f} "
              f"-> {ckpt['ece_after'].mean():.4f})")
        return ckpt["tau"].to(DEVICE)
    print("Fitting per-attribute temperatures (C1)...")
    labels = attrs_train.float().to(DEVICE)
    g = torch.Generator().manual_seed(CALIB_SEED)
    hold = torch.randperm(feats_train.shape[0], generator=g)[
        :int(CALIB_HOLDOUT * feats_train.shape[0])]
    with torch.no_grad():
        z = probe(feats_train[hold])
    y = labels[hold]
    tau = fit_temperature_scalars(z, y, CALIB_STEPS, CALIB_LR)
    ece_b = expected_calibration_error(torch.sigmoid(z), y, ECE_BINS)
    ece_a = expected_calibration_error(torch.sigmoid(z / tau), y, ECE_BINS)
    print(f"  mean ECE {ece_b.mean():.4f} -> {ece_a.mean():.4f}")
    torch.save({"tau": tau.cpu(), "ece_before": ece_b.cpu(), "ece_after": ece_a.cpu()}, path)
    return tau


tau_cal = load_or_fit_temperatures(force=REFIT_CALIBRATION)
P_test = calibrated_posteriors(probe, feats_test, tau_cal)
P_train = calibrated_posteriors(probe, feats_train, tau_cal)


In [ ]:
# == C2 — Chow-Liu dependency tree + exact sum-product (from scratch) ==========
NEG = -1e30         # log-space "impossible" — finite so logsumexp stays NaN-free
LAPLACE_ALPHA = 1.0


@dataclass(frozen=True)
class DependencyTree:
    # Rooted forest over the attributes with CPTs — the once-built offline C2 artifact.
    parent: tuple[int, ...]
    order: tuple[int, ...]        # topological, roots first
    log_cpt: torch.Tensor         # [n, 2, 2] log p(child | parent); root rows unused
    log_marginal: torch.Tensor    # [n, 2] log priors (roots + evidence-ratio denominators)

    def to(self, dev):
        return DependencyTree(self.parent, self.order,
                              self.log_cpt.to(dev), self.log_marginal.to(dev))


def _pair_counts(A: torch.Tensor) -> torch.Tensor:
    # counts[a, b, s_a, s_b] — the O(n²) sufficient statistics for MI, CPTs, marginals.
    n1 = A.sum(0)
    c11 = A.T @ A
    counts = torch.empty(A.shape[1], A.shape[1], 2, 2)
    counts[:, :, 1, 1] = c11
    counts[:, :, 1, 0] = n1.unsqueeze(1) - c11
    counts[:, :, 0, 1] = n1.unsqueeze(0) - c11
    counts[:, :, 0, 0] = A.shape[0] - counts[:, :, 1, 1] - counts[:, :, 1, 0] - counts[:, :, 0, 1]
    return counts


def mutual_information_matrix(A: torch.Tensor, alpha: float) -> torch.Tensor:
    # Chow-Liu edge weights on Laplace-smoothed pairwise joints; diagonal zeroed.
    joint = (_pair_counts(A) + alpha) / (A.shape[0] + 4 * alpha)
    ma, mb = joint.sum(dim=3), joint.sum(dim=2)
    mi = (joint * (joint.log() - ma.unsqueeze(3).log() - mb.unsqueeze(2).log())).sum(dim=(2, 3))
    mi.fill_diagonal_(0.0)
    return mi


def maximum_spanning_tree(weights: torch.Tensor) -> list[int]:
    # Prim on the dense 40x40 matrix; parent[] rooted at node 0.
    n = weights.shape[0]
    parent = [-1] * n
    in_tree = torch.zeros(n, dtype=torch.bool); in_tree[0] = True
    best_w = weights[0].clone(); best_w[0] = float("-inf")
    best_from = torch.zeros(n, dtype=torch.long)
    for _ in range(n - 1):
        v = int(best_w.masked_fill(in_tree, float("-inf")).argmax())
        parent[v] = int(best_from[v]); in_tree[v] = True
        better = (weights[v] > best_w) & ~in_tree
        best_from[better] = v
        best_w = torch.where(better, weights[v], best_w)
    return parent


def _topological_order(parent: list[int]) -> list[int]:
    children = [[] for _ in parent]
    order = []
    for v, p in enumerate(parent):
        (order if p < 0 else children[p]).append(v)
    i = 0
    while i < len(order):
        order.extend(children[order[i]]); i += 1
    return order


def load_or_build_tree(force: bool = False) -> DependencyTree:
    # Counted once from TRAIN labels, cached, frozen (spec §3.2 discipline).
    path = ARTIFACTS_DIR / "bap_plus_chowliu.pt"         # cache-compat name
    if path.exists() and not force:
        ckpt = torch.load(path, map_location="cpu")
        print(f"[loaded] {path}")
        return DependencyTree(tuple(ckpt["parent"]), tuple(ckpt["order"]),
                              ckpt["log_cpt"], ckpt["log_marginal"]).to(DEVICE)
    print("Building Chow-Liu dependency tree (C2)...")
    A = attrs_train.float().cpu()
    counts = _pair_counts(A)
    mi = mutual_information_matrix(A, LAPLACE_ALPHA)
    parent = maximum_spanning_tree(mi)
    order = _topological_order(parent)
    n1 = A.sum(0)
    log_marginal = torch.stack([(A.shape[0] - n1 + LAPLACE_ALPHA), (n1 + LAPLACE_ALPHA)],
                               dim=1).div(A.shape[0] + 2 * LAPLACE_ALPHA).log()
    log_cpt = torch.zeros(N_ATTR, 2, 2)
    for c, p in enumerate(parent):
        if p >= 0:
            pair = counts[p, c] + LAPLACE_ALPHA
            log_cpt[c] = (pair / pair.sum(dim=1, keepdim=True)).log()
    top = sorted(((mi[c, p].item(), c, p) for c, p in enumerate(parent) if p >= 0), reverse=True)
    for w, c, p in top[:5]:
        print(f"  top-MI edge: {ATTRIBUTE_NAMES[p]} -- {ATTRIBUTE_NAMES[c]}  (MI={w:.4f})")
    torch.save({"parent": parent, "order": order, "log_cpt": log_cpt,
                "log_marginal": log_marginal}, path)
    return DependencyTree(tuple(parent), tuple(order), log_cpt, log_marginal).to(DEVICE)


tree = load_or_build_tree(force=REFIT_CALIBRATION)


In [ ]:
# == C2/C3 — sum-product with soft evidence (batched over the whole DB) ========
def evidence_log_ratio(P: torch.Tensor, log_marginal: torch.Tensor) -> torch.Tensor:
    # u_a(s) = log q_a(s) − log π_a(s). Dividing the probe posterior by the TRAIN prior
    # makes sum-product over an EDGELESS forest reproduce the independent product exactly
    # — the algebra behind the BAP-parity invariant.
    u = torch.stack([(1 - P).log(), P.log()], dim=2)                    # [N, n, 2]
    return u - log_marginal.unsqueeze(0)


def clamp_evidence(log_u: torch.Tensor, pi: list[int], ni: list[int]) -> torch.Tensor:
    # Constrain T⁺=1 / T⁻=0 by making the forbidden state log-impossible.
    u = log_u.clone()
    u[:, pi, 0] = NEG
    u[:, ni, 1] = NEG
    return u


def _upward_pass(tree: DependencyTree, log_u: torch.Tensor):
    # Leaves→roots: m_{c→p}(s_p) = logsumexp_{s_c}[log p(s_c|s_p) + acc_c(s_c)].
    acc = log_u.clone()
    msgs = [None] * len(tree.parent)
    for c in reversed(tree.order):
        p = tree.parent[c]
        if p < 0:
            continue
        m = torch.logsumexp(tree.log_cpt[c].unsqueeze(0) + acc[:, c].unsqueeze(1), dim=2)
        msgs[c] = m
        acc[:, p] = acc[:, p] + m
    return acc, msgs


def tree_log_partition(tree: DependencyTree, log_u: torch.Tensor) -> torch.Tensor:
    # log Z = log Σ_A p_T(A)·Π_a exp(u_a(A_a)) — exact on trees (one upward pass).
    acc, _ = _upward_pass(tree, log_u)
    z = torch.zeros(log_u.shape[0], device=log_u.device)
    for r, p in enumerate(tree.parent):
        if p < 0:
            z = z + torch.logsumexp(tree.log_marginal[r].unsqueeze(0) + acc[:, r], dim=1)
    return z


def tree_marginals(tree: DependencyTree, log_u: torch.Tensor) -> torch.Tensor:
    # p(A_a = 1 | evidence) for every node — full two-pass belief propagation (C3).
    acc, msgs = _upward_pass(tree, log_u)
    belief = torch.empty_like(acc)
    for a in tree.order:
        p = tree.parent[a]
        if p < 0:
            belief[:, a] = tree.log_marginal[a].unsqueeze(0) + acc[:, a]
        else:
            down = torch.logsumexp(
                tree.log_cpt[a].unsqueeze(0) + (belief[:, p] - msgs[a]).unsqueeze(2), dim=1)
            belief[:, a] = down + acc[:, a]
    return (belief[:, :, 1] - torch.logsumexp(belief, dim=2)).exp()


def tree_constraint_log(tree, log_u, log_z_free, pi, ni) -> torch.Tensor:
    # C2 scoring term — log p_tree(T⁺=1, T⁻=0 | evidence) = logZ(clamped) − logZ(free).
    if not pi and not ni:
        return torch.zeros(log_u.shape[0], device=log_u.device)
    return tree_log_partition(tree, clamp_evidence(log_u, pi, ni)) - log_z_free


In [ ]:
# == BAP+ scorer (C1+C2+C3 composed; all-off ⇒ plain BAP, asserted) ============
class BAPPlusScorer:
    # S(x|ref) = tree-constraint + conditioned Poisson-binomial tail [+ λ·cos].
    # Per-query tree work (BP over the whole DB) is memoized so the chunked eval
    # driver pays it once per query, not once per source chunk.
    def __init__(self, feats, P, tree=None, conditioned_tail=False, lam=0.0,
                 hamming_max: int = 2):
        if conditioned_tail and tree is None:
            raise ValueError("C3 requires C2 (the dependency tree)")
        self.feats, self.P = feats, P
        self.logP, self.log1mP = P.log(), (1 - P).log()
        self.tree, self.conditioned_tail, self.lam = tree, conditioned_tail, lam
        self.hamming_max = hamming_max
        if tree is not None:
            self.log_u = evidence_log_ratio(P, tree.log_marginal)
            self.log_z_free = tree_log_partition(tree, self.log_u)
        self._key, self._terms = None, None

    def query_terms(self, pi, ni, free):
        # (constraint [N], tail marginals [N, F]) — query-dependent, source-independent.
        key = (tuple(pi), tuple(ni))
        if self._key != key:
            if self.tree is not None:
                constraint = tree_constraint_log(self.tree, self.log_u, self.log_z_free, pi, ni)
                if self.conditioned_tail:
                    marg = tree_marginals(self.tree, clamp_evidence(self.log_u, pi, ni))
                    marg = marg[:, free].clamp(EPS, 1 - EPS)
                else:
                    marg = self.P[:, free]
            else:
                constraint = self.logP[:, pi].sum(1) + self.log1mP[:, ni].sum(1)
                marg = self.P[:, free]
            self._key, self._terms = key, (constraint, marg)
        return self._terms

    @torch.no_grad()
    def chunk(self, srcs, pi, ni, free):
        constraint, marg = self.query_terms(pi, ni, free)
        tail = hamming_tail_log(self.P[srcs][:, free], marg, self.hamming_max)
        s = constraint.unsqueeze(0) + tail
        if self.lam:
            s = s + self.lam * (self.feats[srcs] @ self.feats.T)
        return s


# HARD INVARIANT — the all-off scorer must reproduce plain BAP before anything runs.
pi0, ni0, free0 = query_indices(gt_list[0]["query"])
srcs0 = [int(s) for s in gt_list[0]["ground_truth"]][:8]
want = make_bap_chunk(P_test_raw)(srcs0, pi0, ni0, free0)
got = BAPPlusScorer(feats_test, P_test_raw).chunk(srcs0, pi0, ni0, free0)
assert (want - got).abs().max().item() < 1e-4, "all-off BAP+ deviates from BAP"
print(f"parity invariant OK: max |all-off BAP+ − BAP| = {(want - got).abs().max():.2e}")

bap_plus = BAPPlusScorer(feats_test, P_test, tree, conditioned_tail=True)
results_bap_plus = evaluate_scorer(bap_plus.chunk, gt_list, name="BAP+")
results_table(results_bap_plus).tail(3)


---
# 8. A-BAP — Amortizing the Bayes Score into the Fusion Module Φ

Two legitimate objections to the closed form: it is not literally "a composite query embedding",
and its fixed weights lose on 3/13 queries. Both are solved by **amortization**.

## 8.1 The augmented database (offline, once, frozen)

$$
x_{\text{aug}} = \big[\, v_x \,;\; \log p(x) \,;\; \log(1 - p(x)) \,;\; p(x) \,\big] \in \mathbb{R}^{512 + 40 + 40 + 40 = 632}
$$

The probe channels have the same once-built-frozen status as CLAY's projection matrices; the CLIP
block *is* the frozen DB.

## 8.2 The forward pass (spec §4: architecture, shapes, and flow)

For a batch of $B$ references with constraint masks $\mathbf m^+, \mathbf m^- \in \{0,1\}^{40}$:

$$
\textbf{(1) Input assembly:}\quad
z = \big[\, v_{\text{ref}} \,\oplus\, p(\text{ref}) \,\oplus\, \mathbf m^+ \,\oplus\, \mathbf m^- \,\big]
\;\in\; \mathbb{R}^{B \times 632}
$$

$$
\textbf{(2) Trunk:}\quad
h = \mathrm{GELU}\big(\mathrm{LN}(W_2\, \mathrm{GELU}(\mathrm{LN}(W_1 z)))\big)
\;\in\; \mathbb{R}^{B \times 512}
\qquad (W_1 \in \mathbb{R}^{512 \times 632},\; W_2 \in \mathbb{R}^{512 \times 512})
$$

$$
\textbf{(3) Heads (all zero-initialized):}\quad
\Delta^+ = W_{p} h,\;\; \Delta^- = W_{n} h,\;\; w^p = W_{r} h \in \mathbb{R}^{B \times 40};
\quad q_v = W_{v} h \in \mathbb{R}^{B \times 512};
\quad \kappa \in \mathbb{R} \;(\text{init } 1)
$$

$$
\textbf{(4) Weight assembly:}\quad
w^+ = \mathbf m^+ + \Delta^+, \qquad w^- = \mathbf m^- + \Delta^-
$$

$$
\textbf{(5) Score — one inner product against the frozen augmented DB } (X \in \mathbb{R}^{N \times 632}):
$$

$$
S(x) \;=\; \underbrace{\langle q_{\text{aug}},\, x_{\text{aug}} \rangle}_{q_{\text{aug}} = [\,q_v ;\, w^+ ;\, w^- ;\, w^p\,]}
\;+\; \kappa \cdot \underbrace{\log P(K \le 2)}_{\text{identity tail (fixed channel)}}
\;=\; w^+ \!\cdot \log p(x) + w^- \!\cdot \log(1{-}p(x)) + w^p \!\cdot p(x) + q_v \!\cdot v_x + \kappa\, t(x)
$$

Shapes at score time: $[B, 40] \times [40, N] + [B, 512] \times [512, N] + \kappa \cdot [B, N]
\to [B, N]$. One $\Phi$ forward per source batch; the DB-side channels never change.

**Proposition (Bayes initialization).** With all heads zero and $\kappa = 1$: $w^+ = \mathbf m^+$,
$w^- = \mathbf m^-$, $w^p = 0$, $q_v = 0$, so $S(x)$ reduces term-by-term to the closed-form BAP
score of §6.3 — *exactly*, since zero-weight matmuls contribute exactly zero. The untrained module
therefore starts at the validated closed-form floor, and training can only improve on it (verified
numerically below: deviation 0.00e+00). In the refined variant used for the headline, the
independent constraint term is replaced by the tree constraint of §7 as the *base*, and Φ's
$\Delta$-heads learn residuals on top — the same invariant holds against BAP+.

Training learns exactly what the closed form lacks: per-query probe-reliability re-weighting
(fixes `+Mustache`), corrections on correlated unconstrained channels (attacks the lipstick
composite), and a learned CLIP-identity channel $q_v$ (fixes `+Smiling`).

## 8.3 Training protocol

Synthetic queries from the same `QueryGenerator` that already mirrors the GT rule (§5.3);
candidates per query = valid targets ⊕ constraint-violating hard negatives ⊕ random negatives.
Listwise **ListNet** loss on the composite scores with $\tau = 2$ (prior geometric models used
$\tau = 0.07$ on cosine in $[-1,1]$; these scores span $\approx[-30, 0]$, so $\tau \approx 2$
gives comparable softmax sharpness). AdamW ($\text{lr}=10^{-4}$, weight decay $10^{-2}$ — decay
toward zero heads encodes "stay near the Bayes init unless data demands otherwise"), cosine
schedule, 40 epochs, gradient clipping at 5. A **LambdaLoss** alternative (Wang et al. 2018) is
implemented for the loss ablation of §9.


In [ ]:
# == A-BAP — the fusion module Φ ===============================================
@dataclass(frozen=True)
class ABAPConfig:
    hidden: int = 512
    epochs: int = 40
    lr: float = 1e-4
    weight_decay: float = 1e-2
    tau_list: float = 2.0          # ListNet temperature on log-likelihood-scale scores
    sigma_lambda: float = 1.0      # LambdaLoss pairwise logistic scale
    batch_queries: int = 64
    steps_div: int = 25            # steps/epoch = N_train // (batch_queries·steps_div)
    n_rand_neg: int = 16
    hamming_max: int = 2
    seed: int = SEED


class BayesPhi(nn.Module):
    # Φ(v_ref, p_ref, T⁺, T⁻) → per-channel weights over the frozen augmented DB
    # plus a tail gain κ. Zero-init heads + κ=1 ⇒ init ≡ closed-form score exactly
    # (the provable floor); see the forward-pass derivation above.
    def __init__(self, cfg: ABAPConfig):
        super().__init__()
        d_in = FEATURE_DIM + 3 * N_ATTR              # v_ref ⊕ p_ref ⊕ m⁺ ⊕ m⁻
        self.trunk = nn.Sequential(
            nn.Linear(d_in, cfg.hidden), nn.LayerNorm(cfg.hidden), nn.GELU(),
            nn.Linear(cfg.hidden, cfg.hidden), nn.LayerNorm(cfg.hidden), nn.GELU(),
        )
        self.head_wpos = nn.Linear(cfg.hidden, N_ATTR)       # Δ⁺ on log p channels
        self.head_wneg = nn.Linear(cfg.hidden, N_ATTR)       # Δ⁻ on log(1−p) channels
        self.head_wp   = nn.Linear(cfg.hidden, N_ATTR)       # w_p on raw p channels
        self.head_qv   = nn.Linear(cfg.hidden, FEATURE_DIM)  # CLIP-channel query
        for head in (self.head_wpos, self.head_wneg, self.head_wp, self.head_qv):
            nn.init.zeros_(head.weight); nn.init.zeros_(head.bias)
        self.kappa = nn.Parameter(torch.tensor(1.0))         # identity-tail gain

    def forward(self, v_ref, p_ref, mh_pos, mh_neg) -> dict:
        h = self.trunk(torch.cat([v_ref, p_ref, mh_pos, mh_neg], dim=1))
        return {"w_pos": mh_pos + self.head_wpos(h),          # Bayes indicators + Δ
                "w_neg": mh_neg + self.head_wneg(h),
                "w_p": self.head_wp(h), "q_v": self.head_qv(h), "kappa": self.kappa}


def phi_composite(w, feats, P, logP, log1mP, constraint, tail, mh_pos, mh_neg,
                  use_qv: bool = True):
    # S = base constraint + Δw⁺·log p + Δw⁻·log(1−p) + w_p·p [+ q_v·v] + κ·tail.
    # Subtracting mh keeps only the learned Δ, so the tree constraint (not the
    # independent product) is the base — zero heads + κ=1 ⇒ S ≡ BAP+ exactly.
    s = (constraint.unsqueeze(0)
         + (w["w_pos"] - mh_pos) @ logP.T
         + (w["w_neg"] - mh_neg) @ log1mP.T
         + w["w_p"] @ P.T
         + w["kappa"] * tail)
    if use_qv:
        s = s + w["q_v"] @ feats.T
    return s


def listnet_scores(scores: torch.Tensor, n_pos: int, tau: float) -> torch.Tensor:
    # ListNet on raw score vectors: y = 1/|pos| on the positive prefix.
    labels = torch.zeros_like(scores)
    labels[:n_pos] = 1.0 / n_pos
    return -(labels * F.log_softmax(scores / tau, dim=0)).sum()


def lambda_loss(scores: torch.Tensor, n_pos: int, sigma: float) -> torch.Tensor:
    # LambdaLoss (NDCG-Loss form): pairwise logistic weighted by |ΔNDCG| at the
    # current (detached) ranks — top-of-list mistakes cost the most. The §9 ablation
    # shows it UNDERPERFORMS ListNet here; kept as the measured negative result.
    C = scores.numel()
    order = scores.detach().argsort(descending=True)
    ranks = torch.empty(C, device=scores.device)
    ranks[order] = torch.arange(1, C + 1, dtype=ranks.dtype, device=scores.device)
    D = 1.0 / torch.log2(1.0 + ranks)
    max_dcg = (1.0 / torch.log2(torch.arange(2, n_pos + 2, dtype=ranks.dtype,
                                             device=scores.device))).sum()
    diff = scores[:n_pos].unsqueeze(1) - scores[n_pos:].unsqueeze(0)
    w = (1.0 / max_dcg) * (D[:n_pos].unsqueeze(1) - D[n_pos:].unsqueeze(0)).abs()
    return (w * F.softplus(-sigma * diff)).sum()


In [ ]:
# == A-BAP training (flag-guarded) + init-sanity invariant =====================
ABAP_WEIGHTS = OUTPUT_DIR / "Trained" / "BAP_Plus_ABAP" / "tier5_Bprime_listnet_seed0.pt"
GEN_CFG_ABAP = GenConfig(seed=SEED)          # same sampling protocol as Φ-Combined


def train_abap(cfg: ABAPConfig, loss_name: str = "listnet",
               use_qv: bool = True, seed: int = SEED):
    # Trains Φ on the BAP+ base (tree constraint + conditioned tail as fixed
    # channels, exactly like the closed form's terms). Pluggable loss for the §9
    # ablation; returns (model, per-epoch loss curve).
    torch.manual_seed(seed)
    logP_tr, log1mP_tr = P_train.log(), (1 - P_train).log()
    log_u_tr = evidence_log_ratio(P_train, tree.log_marginal)
    log_z_free_tr = tree_log_partition(tree, log_u_tr)
    loss_fn = {"listnet": lambda s, n: listnet_scores(s, n, cfg.tau_list),
               "lambdaloss": lambda s, n: lambda_loss(s, n, cfg.sigma_lambda)}[loss_name]

    model = BayesPhi(cfg).to(DEVICE)
    gen = QueryGenerator(attrs_train, replace(GEN_CFG_ABAP, seed=seed))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    steps_per_epoch = max(1, gen.N // (cfg.batch_queries * cfg.steps_div))
    curve = []
    for epoch in range(cfg.epochs):
        model.train()
        running, seen = 0.0, 0
        for _ in range(steps_per_epoch):
            opt.zero_grad()
            batch_loss, n = torch.zeros((), device=DEVICE), 0
            for _ in range(cfg.batch_queries):
                sample = gen.sample_query()
                if sample is None:
                    continue
                r, pos_names, neg_names, pos_ids, hard_ids = sample
                pi = [ATTR_TO_IDX[a] for a in pos_names]
                ni = [ATTR_TO_IDX[b] for b in neg_names]
                free = [c for c in range(N_ATTR) if c not in set(pi) | set(ni)]
                rand_ids = torch.randint(gen.N, (cfg.n_rand_neg,), generator=gen.g)
                cand = torch.cat([pos_ids, hard_ids, rand_ids]).to(DEVICE)
                mh_pos = torch.zeros(1, N_ATTR, device=DEVICE)
                mh_neg = torch.zeros(1, N_ATTR, device=DEVICE)
                mh_pos[0, pi], mh_neg[0, ni] = 1.0, 1.0
                w = model(feats_train[r].unsqueeze(0), P_train[r].unsqueeze(0),
                          mh_pos, mh_neg)
                with torch.no_grad():              # base terms are fixed channel values
                    u_c = log_u_tr[cand]
                    constraint = tree_constraint_log(tree, u_c, log_z_free_tr[cand], pi, ni)
                    marg = tree_marginals(tree, clamp_evidence(u_c, pi, ni))
                    marg = marg[:, free].clamp(EPS, 1 - EPS)
                    tail = hamming_tail_log(P_train[r].unsqueeze(0)[:, free], marg,
                                            cfg.hamming_max)
                scores = phi_composite(w, feats_train[cand], P_train[cand], logP_tr[cand],
                                       log1mP_tr[cand], constraint, tail,
                                       mh_pos, mh_neg, use_qv).squeeze(0)
                batch_loss = batch_loss + loss_fn(scores, pos_ids.numel())
                n += 1
            if n == 0:
                continue
            loss = batch_loss / n
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            running += loss.item() * n; seen += n
        sched.step()
        curve.append(running / max(seen, 1))
        print(f"  epoch {epoch + 1:02d}/{cfg.epochs}  {loss_name}={curve[-1]:.4f}  "
              f"kappa={model.kappa.item():.3f}")
    return model, curve


def make_abap_chunk(model: BayesPhi, scorer: BAPPlusScorer, use_qv: bool = True):
    # One Φ forward per source chunk; composite score against the frozen channels.
    model.eval()

    @torch.no_grad()
    def score_chunk(srcs, pi, ni, free):
        constraint, marg = scorer.query_terms(pi, ni, free)
        S = len(srcs)
        mh_pos = torch.zeros(S, N_ATTR, device=DEVICE)
        mh_neg = torch.zeros(S, N_ATTR, device=DEVICE)
        mh_pos[:, pi], mh_neg[:, ni] = 1.0, 1.0
        w = model(scorer.feats[srcs], scorer.P[srcs], mh_pos, mh_neg)
        tail = hamming_tail_log(scorer.P[srcs][:, free], marg, scorer.hamming_max)
        return phi_composite(w, scorer.feats, scorer.P, scorer.logP, scorer.log1mP,
                             constraint, tail, mh_pos, mh_neg, use_qv)
    return score_chunk


# Init-sanity: an UNTRAINED Φ must reproduce BAP+ (the Bayes-initialization proposition).
cfg_abap = ABAPConfig()
fresh = BayesPhi(cfg_abap).to(DEVICE)
a = bap_plus.chunk(srcs0[:1], pi0, ni0, free0)
b = make_abap_chunk(fresh, bap_plus)(srcs0[:1], pi0, ni0, free0)
print(f"init sanity: max |Phi_init − BAP+| = {(a - b).abs().max().item():.2e}")

# Training is strictly opt-in (same policy as Φ-Combined): no weights + flag off
# => skip and rely on the cached result tables; RETRAIN_ABAP=True trains for real.
phi_abap, abap_curve = None, None
if RETRAIN_ABAP:
    phi_abap, abap_curve = train_abap(cfg_abap, loss_name="listnet", use_qv=True)
    torch.save(phi_abap.state_dict(), ABAP_WEIGHTS)
    print(f"[trained+saved] {ABAP_WEIGHTS}")
elif ABAP_WEIGHTS.exists():
    phi_abap = BayesPhi(cfg_abap).to(DEVICE)
    phi_abap.load_state_dict(torch.load(ABAP_WEIGHTS, map_location=DEVICE))
    print(f"[loaded] {ABAP_WEIGHTS}")
else:
    print(f"[skipped] no weights at {ABAP_WEIGHTS} — set RETRAIN_ABAP=True "
          "(~45 min GPU) to train; reported numbers come from the cached CSVs (section 9)")

if phi_abap is not None:
    results_abap = evaluate_scorer(make_abap_chunk(phi_abap, bap_plus), gt_list, name="A-BAP")
    display(results_table(results_abap))


---
# 9. Results

## 9.1 The complete ledger

All numbers are means over the 13 unique mandatory queries (§3.2 note). Rows above the line are
the training-free and learned geometric fusion sections above (single-vector cosine); rows below score in probability space.


In [ ]:
# == Headline ledger (cached numbers; live rows recomputed above) ==============
# Every tuple below was checked against its committed MEAN row under output/ (all 13
# match to 4 decimals); transcribed here rather than recomputed so this table renders
# even on a bundle-only Run-All where not every tier is retrained.
LEDGER = {
    "CLIP (vanilla arithmetic)"           : (0.0224, 0.0699, 0.1048),
    "CLIP Enhanced (all fixes)"            : (0.0397, 0.1140, 0.1651),
    "CLAY reproduction (best)"             : (0.0098, 0.0368, 0.0541),
    "GDE reproduction"                     : (0.0221, 0.0607, 0.0910),
    "GDE-VP (alpha=1.5)"                   : (0.0248, 0.0724, 0.1085),
    "DGP sharp gate (tau=0.07)"            : (0.0359, 0.1141, 0.1719),
    "DGP (uniform gate)"                   : (0.0363, 0.1152, 0.1777),
    "Phi-Gate (InfoNCE)"                   : (0.0383, 0.1110, 0.1607),
    "Phi-Combined"                         : (0.0563, 0.1576, 0.2329),
    "Dual Score (best single-vector)"      : (0.0597, 0.1749, 0.2557),
    "BAP (closed-form, training-free)"     : (0.0868, 0.2596, 0.3554),
    "BAP+ (C1+C2+C3, training-free)"       : (0.0742, 0.2747, 0.3656),
    "A-BAP (trained Phi) — ours"           : (0.1104, 0.2913, 0.4128),
}
ledger_df = pd.DataFrame(LEDGER, index=["R@1", "R@5", "R@10"]).T
ledger_df.style.highlight_max(axis=0, props="font-weight: bold;")


## 9.2 Ablations: every component earns its place (or is rejected)

Every row defaults to the previous stage's behaviour (parity anchors asserted in-code), so each
delta is attributable to exactly one change:

| Row | Components | R@1 | R@5 | R@10 | Verdict |
|---|---|---|---|---|---|
| BAP (anchor) | — | 0.0868 | 0.2596 | 0.3554 | floor |
| + C1 calibration | temp. scaling | 0.0726 | 0.2598 | **0.3787** | keep (R@10) |
| + C2 dependency tree | Chow–Liu | 0.0764 | 0.2500 | 0.3533 | keep only with C3 |
| + C3 conditioned tail | = BAP+ | 0.0742 | **0.2747** | 0.3656 | best training-free R@5 |
| + C4 λ·cos channel | tuned λ | 0.0759 | 0.2626 | 0.3720 | marginal — **rejected** |
| + C5 QB-Norm | tuned β | 0.0686 | 0.2449 | 0.3335 | hurts — **rejected** |
| Φ / ListNet (headline) | on BAP+ | **0.1104** | **0.2913** | **0.4128** | **selected** |
| Φ / LambdaLoss (3 seeds) | loss swap | 0.065–0.081 | 0.205–0.241 | 0.314–0.347 | hypothesis **refuted** |
| Φ / no $q_v$ (3 seeds) | CLIP channel off | 0.058–0.074 | 0.219–0.234 | 0.324–0.352 | $q_v$ helps — kept |

**Negative results, reported as findings.** (i) LambdaLoss was motivated by an R@1/R@5 trade-off
observed in an earlier Φ run; on the improved BAP+ base it *underperformed* ListNet across all
three seeds — the hypothesis is refuted, and the refined base itself recovered R@1.
(ii) QB-Norm, a reliable free win in CLIP-cosine retrieval, *hurts* in probability space —
log-likelihood scores are already per-query calibrated, so bank normalization only adds noise.


In [ ]:
# == Ablation table from the cached VM run =====================================
ABLATION_CSV = OUTPUT_DIR / "Trained" / "BAP_Plus_ABAP" / "tier5_ablation_summary_full.csv"
if ABLATION_CSV.exists():
    display(pd.read_csv(ABLATION_CSV))
else:
    print(f"[cached CSV not found: {ABLATION_CSV}]")
    # If needed, pull output/Trained/BAP_Plus_ABAP/*.csv next to the notebook, or
    # re-run the C4/C5 arms here — code for both components:
    # scorer_c4 = BAPPlusScorer(feats_test, P_test, tree, conditioned_tail=True, lam=<tuned>)
    # (λ/β grid-tuning on synthetic TRAIN queries mirrors src/Trained/cdbayes.py::TrainTuner;
    #  omitted from the live path because both components were REJECTED on evidence.)


In [ ]:
# == 7. BAP+ ablation -- every C-component, fusion model Phi excluded ============================
# Bprime_* rows are the amortized fusion module (A-BAP / Sec 8), out of scope for this chart,
# which isolates what each closed-form component (C1-C5) contributes on top of plain BAP.
if ABLATION_CSV.exists():
    _abl = pd.read_csv(ABLATION_CSV).set_index("row")
    _bap_rows = _abl[~_abl.index.str.startswith("Bprime")]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(_bap_rows))
    for k in ("R@1", "R@5", "R@10"):
        ax.plot(x, _bap_rows[k], marker="o", label=k)
    ax.set_xticks(x), ax.set_xticklabels(_bap_rows.index, rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("macro mean"), ax.set_title("BAP+ -- component ablation (C1-C5, no fusion Phi)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "visual" / "report_7_bapplus_ablation.png", dpi=120)
    plt.show()
else:
    print(f"[cached CSV not found: {ABLATION_CSV}]")


## 9.3 Learning curves


In [ ]:
# == Training curve (live if retrained this session, else from cached logs) ===
if abap_curve is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(range(1, len(abap_curve) + 1), abap_curve)
    ax.set_xlabel("epoch"); ax.set_ylabel("ListNet loss")
    ax.set_title("A-BAP $\\Phi$ training")
    plt.show()
else:
    print("[Phi loaded from cache — flip RETRAIN_ABAP=True to regenerate the curve]")
    # Cached per-epoch loss curves from the VM runs (champion + LambdaLoss seeds).
    from IPython.display import Image as _Img, display as _dsp
    _curves = [("A-BAP / ListNet seed0 (champion)", "tier5_Bprime_listnet_seed0_curve.png")] + \
              [(f"A-BAP / LambdaLoss seed{s}", f"tier5_Bprime_lambdaloss_seed{s}_curve.png")
               for s in (0, 1, 2)]
    for _label, _fname in _curves:
        _p = OUTPUT_DIR / "Trained" / "BAP_Plus_ABAP" / _fname
        if _p.exists():
            print(_label)
            _dsp(_Img(str(_p), width=420))
    # (Per-epoch validation-R@5 was not logged on the VM; loss-only curves shown.)


## 9.4 Qualitative analysis — hard / negated queries

Aggregate metrics hide *how* a model fails. Below we render retrieval grids for the hardest
compositional queries — one row per model, reference first, then the top-5 retrievals, green frame
for a ground-truth hit. Showcase sources are selected **systematically, not cherry-picked**:
"success" sources are those where A-BAP hits within top-5 while vanilla CLIP misses its entire
top-10; "failure" sources are those where **all** models miss — reported with equal prominence.
The grids read the models' stored top-10 prefixes from the §10 per-source dumps, so they require
no model weights — only the CelebA images (indexed as `celeba[idx]`, never by filename, per spec
§3.1.2).

| Query | Source | CLIP / CLIP Enhanced top-5 verdict | A-BAP top-5 verdict | Diagnosis |
|---|---|---|---|---|
| `-Male, -Mustache` | 2098 (success) | miss — retrieves male faces with mustaches: cosine arithmetic cannot encode absence | HIT — calibrated posteriors + dependency tree score absence explicitly | geometry vs probability: negation needs a discriminative channel |
| `-Smiling, +Eyeglasses, +Wearing_Hat` | 131 (success) | miss — positive terms dominate, `-Smiling` ignored | HIT — the conditioned tail enforces all three constraints jointly | independent-term arithmetic vs joint constraint satisfaction |

*(Source indices are reproduced deterministically by `select_showcases` below.)*


In [ ]:
# == Systematic showcase selection + retrieval grids (dump-driven) =============
def select_showcases(query: str, champ_name: str = "A-BAP", base_name: str = "CLIP",
                     n: int = 3):
    # success = champion hit@5 & baseline miss@10; failure = ALL models miss@10.
    dc, db = stat_dumps[champ_name][query], stat_dumps[base_name][query]
    assert (dc["sources"] == db["sources"]).all()
    succ = np.flatnonzero(dc["hit5"].astype(bool) & ~db["hit10"].astype(bool))
    fail = np.flatnonzero(~np.any([stat_dumps[m][query]["hit10"]
                                   for m in stat_dumps], axis=0))
    return dc["sources"][succ[:n]].tolist(), dc["sources"][fail[:n]].tolist()


try:
    celeba_raw = datasets.CelebA(root=CELEBA_ROOT, split="test", download=False)
except Exception as e:                          # images optional: grids skip cleanly
    celeba_raw = None
    print(f"[missing] CelebA images unavailable ({e}) — grids are skipped; "
          "showcase selection still runs on indices.")


def show_retrieval_grid(query: str, source_idx: int, model_names=None, top_k: int = 5):
    # Rows = models; cols = reference + top-k retrievals; green/red frame = GT hit/miss.
    if celeba_raw is None:
        return print("CelebA images unavailable — skipping grid.")
    model_names = model_names or list(stat_dumps)
    gt = {int(s): set(map(int, t)) for s, t in
          next(e for e in gt_list if e["query"] == query)["ground_truth"].items()}
    targets = gt[int(source_idx)]
    fig, axes = plt.subplots(len(model_names), top_k + 1,
                             figsize=(1.7 * (top_k + 1), 1.9 * len(model_names)))
    axes = np.atleast_2d(axes)
    for r, name in enumerate(model_names):
        arr = stat_dumps[name][query]
        i = int(np.flatnonzero(arr["sources"] == source_idx)[0])
        axes[r, 0].imshow(celeba_raw[int(source_idx)][0])
        axes[r, 0].set_ylabel(name, fontsize=7)
        axes[r, 0].set_title("reference", fontsize=7)
        for c, idx in enumerate(arr["top10"][i, :top_k], start=1):
            ax = axes[r, c]
            ax.imshow(celeba_raw[int(idx)][0])
            hit = int(idx) in targets
            for spine in ax.spines.values():
                spine.set_edgecolor("green" if hit else "red")
                spine.set_linewidth(3)
            ax.set_title(f"#{c} {'HIT' if hit else 'miss'}", fontsize=7)
    for ax in axes.ravel():
        ax.set_xticks([]), ax.set_yticks([])
    fig.suptitle(f"query: {query}   |   source {source_idx}", fontsize=10)
    fig.tight_layout()
    return fig


for q in ("-Male, -Mustache", "-Smiling, +Eyeglasses, +Wearing_Hat"):
    succ, fail = select_showcases(q)
    print(f"{q}: success showcases {succ}, all-fail showcases {fail}")
    show_retrieval_grid(q, succ[0])
    show_retrieval_grid(q, fail[0])


---
# 10. Statistical Analysis

Single-run point estimates cannot honestly rank models whose means differ by less than sampling
noise. This section attaches an uncertainty statement to every reported mean and a significance
statement to every comparison, on the official benchmark, using only the two spec-mandated metrics.

**Models under comparison** (the six-model roster used in every table and figure below):

| Display name | Report section | Type | Seeds |
|---|---|---|---|
| **CLIP** | §4.1 — vanilla latent arithmetic | training-free | deterministic |
| **CLIP Enhanced** | §4.2 — arithmetic with all three geometric fixes | training-free | deterministic |
| **CLAY** | §4.3 — reproduction (k=50, rotation) | training-free | deterministic |
| **DGP** | §4.6 — gated projection, uniform gate, centered | training-free | deterministic |
| **Dual Score** | §5.4 — query-conditioned dual-channel fusion (ListNet) | learned | 0, 1, 2 |
| **A-BAP** | §8 — amortized Bayes Φ (ListNet) — **champion** | learned | 0 (seeds 1–2 pending) |

The champion's LambdaLoss sibling — *designed* to improve R@1 — underperformed the ListNet control
on **all three seeds**; it is kept out of the main figures but analyzed below as a faithfully
reported negative result and as the empirical evidence for the size of training-seed variance in
this architecture.

## 10.1 Metrics: spec-locked by design

The specification (§3.1.3) mandates exactly two metrics at $K \in \{1, 5, 10\}$, averaged over all
valid source images of each query:

$$\mathrm{Recall@}K(s) = \mathbb{1}\!\left[\, |R_K(s) \cap G(s)| > 0 \,\right], \qquad
\mathrm{Precision@}K(s) = \frac{|R_K(s) \cap G(s)|}{K}.$$

We report **only** these. Adding non-mandated metrics (MRR, NDCG, …) would dilute comparability and
invite metric shopping — and it is unnecessary: the per-source outcomes $\mathrm{hit@}K(s) \in
\{0,1\}$ and $\mathrm{Precision@}K(s) \in [0,1]$ are well-defined random variables over the fixed
source population, which is exactly the raw material resampling inference needs. Everything below is
a *statistic about* the two spec metrics, never a new metric.

## 10.2 Stratified bootstrap confidence intervals

The 13 benchmark queries are a **fixed benchmark**, not a random sample from a query population.
The random unit is the *source image* within each query. The correct resampling is therefore
**stratified**: queries are strata and never resampled; sources are resampled with replacement
within their query. For $b = 1, \dots, B$ ($B = 1000$):

$$\hat\theta^{*}_{b} = \frac{1}{Q} \sum_{q=1}^{Q} \frac{1}{n_q} \sum_{i=1}^{n_q} m_q\!\big(s^{*}_{q,i,b}\big),
\qquad s^{*}_{q,i,b} \sim \mathrm{Uniform}\{s_{q,1}, \dots, s_{q,n_q}\},$$

which is precisely the statistic of the published `MEAN` row, recomputed per replicate. The 95% CI
is the percentile interval (a BCa refinement changes nothing at $n_q \in [27, 5469]$).

## 10.3 Significance: paired bootstrap — and why *not* a rank test

Per-source $\mathrm{hit@}K$ is **binary**. The Wilcoxon signed-rank test is ill-posed on paired
Bernoulli differences: zero-differences are discarded, the rest are massively tied at $\pm 1$, and
the tested median is degenerate. We instead use the **paired stratified bootstrap of the
difference**: both models are evaluated on the *identical* source population, so each replicate
applies the **same** resampled indices to both models,
$\Delta^{*}_{b} = \hat\theta^{*}_{A,b} - \hat\theta^{*}_{B,b}$ — the index-sharing removes
between-source variance from $\Delta$, which is what gives the test its power. We report the 95%
CI of $\Delta$, the relative gain, and the two-sided bootstrap p-value with small-sample correction

$$p = 2\min\!\left( \frac{\#\{\Delta^{*}_{b} \le 0\} + 1}{B+1},\; \frac{\#\{\Delta^{*}_{b} \ge 0\} + 1}{B+1} \right),$$

significant at $\alpha = 0.05$ iff the CI excludes 0 (CI and p-value cannot contradict by
construction). All p-values are **Holm–Bonferroni corrected** over the
$\{6 \text{ metrics} \times 5 \text{ baselines}\}$ family of each model — training seeds are not
extra hypotheses but replicates of one hypothesis, handled by the stricter worst-seed criterion of
§10.4. One resolution requirement follows: the bootstrap p-value cannot fall below $2/(B+1)$, so
significance runs use $B = 10{,}000$ ($30 \cdot 2/10{,}001 \approx 0.006 < 0.05$); CIs keep
$B = 1000$.

## 10.4 Two sources of randomness — the multi-seed protocol

Bootstrap over sources cannot see training-seed variance (initialization, data ordering, query
sampling). Our protocol: training-free models are deterministic (zero seed variance, a stated
methodological strength); learned models are trained 3× where budget permits. The LambdaLoss family
has the full 3 seeds; the ListNet champion currently has **one** — an explicitly stated limitation
(seeds 1–2 queued on the VM), partially mitigated by its margin over every baseline exceeding the
*entire* seed spread observed in the sibling LambdaLoss family under the identical protocol. Seeds
are **never pooled as data points** ($n = 3$ is far too small for any test): the paired bootstrap
runs **per seed**, and significance is claimed only if it holds for **every** seed — the
conservative *worst-seed criterion*.

## 10.5 Robustness to negation and hard compositional queries

Queries are stratified into a taxonomy following the spec's own complexity levels, with the
project's target case split out: **simple positive**, **simple negative**, **composed** (mixed
polarity), and **pure negation** (`-Male, -Mustache` — no positive constraint anchors the query).
Macro metrics are reported per category, so a model cannot hide a negation failure inside a good
average. One contextual fact for reading the pure-negation row: an oracle restricted to
ground-truth attribute labels ranked by **CLIP-cosine identity** reaches only $R@10 \approx 0.074$
on `-Male, -Mustache` — a ceiling that binds every cosine-identity method, and one the
probabilistic scorer breaks six-fold (§10.8), proving the limit belonged to the scoring rule, not
the backbone.


In [ ]:
# == Inlined statistical library ===============================================
# Bootstrap statistics over per-source outcome dumps. All inference is bootstrap-
# based by deliberate choice (rank tests are ill-posed on paired Bernoulli outcomes,
# §10.3); queries are strata and never resampled (§10.2). Unit-tested in
# test/test_stats_eval.py against closed-form small cases.
from dataclasses import dataclass as _dataclass
import numpy as np

KS = (1, 5, 10)          # spec §3.1.3 — the only K values reported anywhere
TOP_KEEP = 10            # retained ranking prefix (feeds the qualitative grids)
_ALPHA = 0.05
_CHUNK = 1000            # replicates per resampling block — bounds peak memory


def _metric_fields(ks=KS):
    return [f"hit{k}" for k in ks] + [f"prec{k}" for k in ks]


def metric_label(field: str) -> str:
    # "hit5" -> "R@5", "prec10" -> "P@10".
    return ("R@" + field[3:]) if field.startswith("hit") else ("P@" + field[4:])


def _check_paired(a: dict, b: dict) -> None:
    # Paired inference is only valid on the identical source population.
    if list(a) != list(b):
        raise ValueError("dumps cover different query sets — pairing invalid")
    for q in a:
        if not np.array_equal(a[q]["sources"], b[q]["sources"]):
            raise ValueError(f"source sets differ for query {q!r} — pairing invalid")


def _bootstrap_means(rng, arrays, B):
    # Resampled means for several arrays of the SAME source population, sharing one
    # index draw per replicate (the sharing is what makes paired tests paired).
    n = len(arrays[0])
    out = [np.empty(B) for _ in arrays]
    for lo in range(0, B, _CHUNK):
        hi = min(lo + _CHUNK, B)
        idx = rng.integers(0, n, size=(hi - lo, n))
        for a, dest in zip(arrays, out):
            dest[lo:hi] = a[idx].mean(axis=1)
    return out


@_dataclass(frozen=True)
class BootstrapResult:
    point: float; mean: float; std: float
    ci_lo: float; ci_hi: float; samples: np.ndarray


def stratified_bootstrap(dump: dict, B: int = 1000, seed: int = 0,
                         ks=KS) -> dict[str, "BootstrapResult"]:
    # Stratified bootstrap CI (§10.2) — the exact statistic of the CSV MEAN row,
    # recomputed on each replicate; percentile 95% CI.
    rng = np.random.default_rng(seed)
    fields = _metric_fields(ks)
    per_query = {f: [] for f in fields}
    point = {f: [] for f in fields}
    for q, arrays in dump.items():
        vals = [arrays[f].astype(np.float64) for f in fields]
        means = _bootstrap_means(rng, vals, B)
        for f, v, m in zip(fields, vals, means):
            per_query[f].append(m)
            point[f].append(v.mean())
    out = {}
    for f in fields:
        macro = np.mean(per_query[f], axis=0)
        lo, hi = np.percentile(macro, [100 * _ALPHA / 2, 100 * (1 - _ALPHA / 2)])
        out[f] = BootstrapResult(float(np.mean(point[f])), float(macro.mean()),
                                 float(macro.std(ddof=1)), float(lo), float(hi), macro)
    return out


@_dataclass(frozen=True)
class DiffResult:
    delta: float; rel_gain: float
    ci_lo: float; ci_hi: float; p_value: float; samples: np.ndarray


def paired_bootstrap_diff(a: dict, b: dict, B: int = 1000, seed: int = 0,
                          ks=KS) -> dict[str, "DiffResult"]:
    # Paired significance test (§10.3): the SAME resampled indices hit both models;
    # p = 2·min(P(Δ*≤0), P(Δ*≥0)) with the (c+1)/(B+1) small-sample correction.
    _check_paired(a, b)
    rng = np.random.default_rng(seed)
    fields = _metric_fields(ks)
    deltas = {f: [] for f in fields}
    point_a, point_b = {f: [] for f in fields}, {f: [] for f in fields}
    for q in a:
        vals = [a[q][f].astype(np.float64) for f in fields] + \
               [b[q][f].astype(np.float64) for f in fields]
        means = _bootstrap_means(rng, vals, B)                 # shared idx — the pairing
        for j, f in enumerate(fields):
            deltas[f].append(means[j] - means[j + len(fields)])
            point_a[f].append(vals[j].mean())
            point_b[f].append(vals[j + len(fields)].mean())
    out = {}
    for f in fields:
        d = np.mean(deltas[f], axis=0)
        lo, hi = np.percentile(d, [100 * _ALPHA / 2, 100 * (1 - _ALPHA / 2)])
        c_le, c_ge = int((d <= 0).sum()), int((d >= 0).sum())
        p = 2 * min((c_le + 1) / (B + 1), (c_ge + 1) / (B + 1))
        pa, pb = float(np.mean(point_a[f])), float(np.mean(point_b[f]))
        rel = (pa - pb) / pb if pb > 0 else float("inf")
        out[f] = DiffResult(pa - pb, rel, float(lo), float(hi), min(p, 1.0), d)
    return out


def holm_bonferroni(pvals: dict[str, float]) -> dict[str, float]:
    # Step-down family-wise error control across the {metrics × model pairs} family.
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m = len(items)
    adjusted, running_max = {}, 0.0
    for rank, (key, p) in enumerate(items):
        running_max = max(running_max, min(1.0, (m - rank) * p))
        adjusted[key] = running_max
    return {k: adjusted[k] for k in pvals}


def query_category(query_str: str) -> str:
    # Taxonomy of §10.5 — the spec's complexity levels with pure negation split out.
    pos, neg = parse_query(query_str)
    if len(pos) == 1 and not neg:
        return "simple positive"
    if len(neg) == 1 and not pos:
        return "simple negative"
    if neg and not pos:
        return "pure negation"
    return "composed"


def category_breakdown(dump: dict, ks=KS) -> dict[str, dict[str, float]]:
    # Macro R@K/P@K per query category (each query weighs 1/|category|).
    cats: dict[str, list[dict]] = {}
    for q, arrays in dump.items():
        cats.setdefault(query_category(q), []).append(arrays)
    return {cat: {metric_label(f): float(np.mean([a[f].mean() for a in arrs]))
                  for f in _metric_fields(ks)}
            for cat, arrs in cats.items()}


def win_tie_loss(a: dict, b: dict, k: int = 10) -> dict:
    # Per query: does A's mean beat B's? Per source (paired): both/only-A/only-B/
    # neither — shows whether A fixes B's failures or merely trades sources.
    _check_paired(a, b)
    wins = ties = losses = 0
    cont = {"both": 0, "only_a": 0, "only_b": 0, "neither": 0}
    for q in a:
        ma, mb = a[q][f"hit{k}"].mean(), b[q][f"hit{k}"].mean()
        wins, ties, losses = wins + (ma > mb), ties + (ma == mb), losses + (ma < mb)
        ha, hb = a[q][f"hit{k}"].astype(bool), b[q][f"hit{k}"].astype(bool)
        cont["both"] += int((ha & hb).sum())
        cont["only_a"] += int((ha & ~hb).sum())
        cont["only_b"] += int((~ha & hb).sum())
        cont["neither"] += int((~ha & ~hb).sum())
    return {"per_query": {"win": wins, "tie": ties, "loss": losses},
            "per_source@k": cont}


def macro_vs_micro(dump: dict, ks=KS) -> dict[str, dict[str, float]]:
    # Macro (each query 1/13 — spec convention) vs micro (each source equal weight).
    fields = _metric_fields(ks)
    macro = {f: np.mean([a[f].mean() for a in dump.values()]) for f in fields}
    micro = {f: np.concatenate([a[f] for a in dump.values()]).mean() for f in fields}
    return {"macro": {metric_label(f): float(macro[f]) for f in fields},
            "micro": {metric_label(f): float(micro[f]) for f in fields}}


def recall_curve(dump: dict, gt: list[dict], ks=range(1, TOP_KEEP + 1)) -> np.ndarray:
    # Macro Recall@K for EVERY K = 1…10 from the stored top-10 prefixes.
    gt_by_query = {e["query"]: e["ground_truth"] for e in gt}
    curves = []
    for q, arrays in dump.items():
        g = {int(s): {int(t) for t in ts} for s, ts in gt_by_query[q].items()}
        top = arrays[f"top{TOP_KEEP}"]
        hits = np.array([[len(set(top[i, :k]) & g[int(s)]) > 0 for k in ks]
                         for i, s in enumerate(arrays["sources"])], dtype=np.float64)
        curves.append(hits.mean(axis=0))
    return np.mean(curves, axis=0)


def plot_ci_bars(results_by_model, fields=("hit1", "hit5", "hit10"),
                 path=None, title="Recall@K with 95% CI"):
    # Grouped bars + percentile-CI error bars (not ±std) — overlap reads as significance.
    models = list(results_by_model)
    x = np.arange(len(fields))
    width = 0.8 / len(models)
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, m in enumerate(models):
        res = [results_by_model[m][f] for f in fields]
        pts = [r.point for r in res]
        err = np.array([[r.point - r.ci_lo for r in res],
                        [r.ci_hi - r.point for r in res]])
        ax.bar(x + i * width, pts, width, yerr=err, capsize=3, label=m)
    ax.set_xticks(x + width * (len(models) - 1) / 2)
    ax.set_xticklabels([metric_label(f) for f in fields])
    ax.set_ylabel("macro mean"), ax.set_title(title), ax.legend(fontsize=8)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=120)
    return fig


def plot_bootstrap_violin(results_by_model, field="hit1", path=None, title=None):
    # Full bootstrap distributions — separated violins are a visual significance statement.
    models = list(results_by_model)
    data = [results_by_model[m][field].samples for m in models]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.violinplot(data, showmedians=True)
    ax.boxplot(data, widths=0.08, showfliers=False)
    ax.set_xticks(range(1, len(models) + 1))
    ax.set_xticklabels(models, rotation=20, ha="right", fontsize=8)
    ax.set_ylabel(f"bootstrap macro {metric_label(field)}")
    ax.set_title(title or f"Bootstrap distribution of {metric_label(field)} (B={len(data[0])})")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=120)
    return fig


In [ ]:
# == Per-source dumps: load cached .npz, or regenerate from any scorer =========
# The statistics need the PRE-AVERAGE view of the spec metrics: per query,
#   sources [n], hit{K} [n], prec{K} [n], top10 [n,10].
# Each cached dump was parity-gated against its published CSV (means must agree to
# half an ulp of the 4-decimal CSV format) before being written on the VM.
PER_SOURCE_DIR = OUTPUT_DIR / "per_source"
_SEP = "::"


def load_per_source(path):
    with np.load(path) as z:
        manifest = json.loads(bytes(z["__manifest__"]).decode())
        dump = {q: {} for q in manifest["queries"]}
        for key in z.files:
            if key != "__manifest__":
                q, field = key.split(_SEP, 1)
                dump[q][field] = z[key]
    return dump


def dump_from_chunk(score_chunk, ks=KS):
    # Regenerate a dump live from any score_chunk seam of this notebook (e.g.
    # make_abap_chunk(...)) — same atoms evaluate_scorer averages, nothing new.
    out = {}
    for entry in gt_list:
        pi, ni, free = query_indices(entry["query"])
        sources = sorted(int(s) for s in entry["ground_truth"])
        arrays = {"sources": np.asarray(sources, dtype=np.int64),
                  f"top{TOP_KEEP}": np.empty((len(sources), TOP_KEEP), dtype=np.int64)}
        for k in ks:
            arrays[f"hit{k}"] = np.empty(len(sources), dtype=np.uint8)
            arrays[f"prec{k}"] = np.empty(len(sources), dtype=np.float32)
        for i, src in enumerate(sources):
            s = score_chunk([src], pi, ni, free).squeeze(0)
            s[src] = float("-inf")
            ranking = torch.topk(s, TOP_KEEP).indices.tolist()
            targets = {int(t) for t in entry["ground_truth"][str(src)]}
            arrays[f"top{TOP_KEEP}"][i] = ranking
            for k in ks:
                arrays[f"hit{k}"][i] = recall_at_k(ranking, targets, k)
                arrays[f"prec{k}"][i] = precision_at_k(ranking, targets, k)
        out[entry["query"]] = arrays
    return out


MODELS = {   # display name -> cached per-source dump (report nomenclature, §10 roster)
    "CLIP":          "tier0_CLIP_baseline.npz",
    "CLIP Enhanced": "tier0_enhanced_all_fixes.npz",
    "CLAY":          "tier1_CLAY_k50_rotH.npz",
    "DGP":           "tier2d_tau100.0_a1.0_center.npz",
    "Dual Score":    "tier3_dualscore.npz",
    "A-BAP":         "tier5_Bprime_listnet_seed0.npz",
}
ABLATION = {f"A-BAP-LambdaLoss seed{s}": f"tier5_Bprime_lambdaloss_seed{s}.npz"
            for s in (0, 1, 2)}


def _load_available(mapping):
    out = {}
    for name, fname in mapping.items():
        path = PER_SOURCE_DIR / fname
        if path.exists():
            out[name] = load_per_source(path)
        else:
            print(f"[missing] {fname} — regenerate with dump_from_chunk(...) or pull "
                  f"the VM bundle; '{name}' is skipped below.")
    return out


stat_dumps = _load_available(MODELS)
ablation_dumps = _load_available(ABLATION)
B, B_SIG = 1000, 10_000       # CI replicates / significance replicates (§10.3)
STATS_FIG_DIR = OUTPUT_DIR / "visual"
STATS_FIG_DIR.mkdir(exist_ok=True)
print(f"Loaded per-source dumps: {list(stat_dumps)}  (+{len(ablation_dumps)} ablation)")


## 10.6 Bootstrap confidence intervals (all models, all spec metrics)


In [ ]:
# Point estimate, bootstrap mean±std, percentile 95% CI for every model × metric.
boot = {name: stratified_bootstrap(d, B=B, seed=0) for name, d in stat_dumps.items()}

rows = []
for name, res in boot.items():
    row = {"model": name}
    for f, r in res.items():
        row[metric_label(f)] = f"{r.point:.4f} ± {r.std:.4f}  [{r.ci_lo:.4f}, {r.ci_hi:.4f}]"
    rows.append(row)
pd.DataFrame(rows).set_index("model")


In [ ]:
# Seed variance (§10.4): mean ± std ACROSS seeds (training variance), reported
# separately from the per-seed bootstrap CI (test-set variance) — never conflated.
if ablation_dumps:
    fam = {n: stratified_bootstrap(d, B=B, seed=0) for n, d in ablation_dumps.items()}
    print(f"A-BAP-LambdaLoss ablation family: n={len(fam)} seeds")
    for f in ("hit1", "hit5", "hit10"):
        pts = np.array([r[f].point for r in fam.values()])
        print(f"  {metric_label(f)}: across-seed mean={pts.mean():.4f}  "
              f"std={pts.std(ddof=1):.4f}")
print("A-BAP (champion): n=1 seed — across-seed std unavailable (§10.4 limitation). "
      "The LambdaLoss family above bounds the expected seed spread under the "
      "identical training protocol.")


## 10.7 Significance: A-BAP vs every baseline (paired bootstrap, worst-seed criterion)


In [ ]:
# Paired bootstrap Δ with 95% CI + Holm-corrected p (§10.3). The champion and the
# multi-seed LambdaLoss family are each tested against every baseline.
CHAMPIONS = (["A-BAP"] if "A-BAP" in stat_dumps else []) + list(ablation_dumps)
BASELINES = [n for n in ("CLIP", "CLIP Enhanced", "CLAY", "DGP", "Dual Score")
             if n in stat_dumps]
test_dumps = {**stat_dumps, **ablation_dumps}

diff_rows, adj = [], {}
for champ in CHAMPIONS:
    # Holm family = the {metrics × baselines} comparisons of THIS model (§10.3);
    # seeds are replicates of one hypothesis, handled by the worst-seed criterion.
    family_p = {}
    for base in BASELINES:
        diffs = paired_bootstrap_diff(test_dumps[champ], stat_dumps[base], B=B_SIG, seed=0)
        for f, r in diffs.items():
            family_p[f"{champ} vs {base} | {metric_label(f)}"] = r.p_value
            diff_rows.append({"comparison": f"{champ} vs {base}",
                              "metric": metric_label(f), "delta": r.delta,
                              "rel_gain_%": 100 * r.rel_gain,
                              "CI95": f"[{r.ci_lo:.4f}, {r.ci_hi:.4f}]",
                              "p_raw": r.p_value})
    adj.update(holm_bonferroni(family_p))

diff_table = pd.DataFrame(diff_rows)
diff_table["p_holm"] = [adj[f"{r.comparison} | {r.metric}"] for r in diff_table.itertuples()]
diff_table["significant"] = diff_table["p_holm"] < 0.05
diff_table.round(4)


In [ ]:
# Worst-seed verdict (§10.4): a claim stands only if it holds for EVERY trained seed.
for fam, label in (("A-BAP vs", "A-BAP champion (n=1 seed)"),
                   ("A-BAP-LambdaLoss seed", "LambdaLoss ablation family")):
    for base in BASELINES:
        for metric in ("R@1", "R@5", "R@10"):
            sub = diff_table[(diff_table.metric == metric)
                             & diff_table.comparison.str.endswith(f"vs {base}")
                             & diff_table.comparison.str.startswith(fam)]
            if len(sub):
                verdict = "SIGNIFICANT (all seeds)" if sub.significant.all() else \
                          f"NOT robust ({int(sub.significant.sum())}/{len(sub)} seeds pass)"
                print(f"{label:<28s} vs {base:<16s} {metric:<5s} "
                      f"min Δ={sub.delta.min():+.4f}  -> {verdict}")


## 10.8 Headline figures — CI bars, bootstrap distributions, per-category robustness

Cells regenerate the figures live from the dumps when available (they overwrite the committed PNGs
in `output/visual/`); with dumps absent, the committed PNGs are displayed instead.


In [ ]:
from IPython.display import Image as _Image, display as _display


def _fig_or_cached(live_fn, png_name):
    # Live regeneration when dumps are loaded; committed PNG otherwise.
    if stat_dumps:
        return live_fn(path=STATS_FIG_DIR / png_name)
    _display(_Image(str(STATS_FIG_DIR / png_name), width=760))


_fig_or_cached(lambda path: plot_ci_bars(boot, path=path), "stats_ci_bars.png")


In [ ]:
_fig_or_cached(lambda path: plot_bootstrap_violin(boot, field="hit1", path=path),
               "stats_violin_r1.png")


In [ ]:
_fig_or_cached(lambda path: plot_bootstrap_violin(boot, field="hit10", path=path),
               "stats_violin_r10.png")


In [ ]:
# Per-category robustness (§10.5): a model cannot hide a negation failure in a
# good average. Note A-BAP's pure-negation R@10 = 0.44 vs the cosine-identity
# oracle's 0.074 ceiling — the six-fold break referenced in §10.5.
cat_rows = []
for name, d in stat_dumps.items():
    for cat, metrics in category_breakdown(d).items():
        cat_rows.append({"model": name, "category": cat, **metrics})
pd.DataFrame(cat_rows).pivot_table(index="model", columns="category",
                                   values="R@10").round(4)


In [ ]:
# == 10. Edge-case queries -- per-query R@1 across the 6-model roster ============================
# Hand-picked from the queries already discussed as edge cases above: pure negation
# ("-Male, -Mustache", Sec 6.4/9.4), the hardest composed query (Sec 9.4 showcase), and the
# strongest negation lift reported for BAP (Sec 6.4). Extends stats_ci_bars.png (aggregate R@K)
# down to the query level for exactly the cases the report calls out by name.
_EDGE_QUERIES = ["-Male, -Mustache", "-Smiling, +Eyeglasses, +Wearing_Hat", "+Chubby, -Young"]
_edge = pd.DataFrame({name: [d[q]["hit1"].mean() if q in d else float("nan")
                              for q in _EDGE_QUERIES]
                       for name, d in stat_dumps.items()}, index=_EDGE_QUERIES)
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(_EDGE_QUERIES))
width = 0.8 / len(_edge.columns)
for i, name in enumerate(_edge.columns):
    ax.bar(x + i * width, _edge[name], width, label=name)
ax.set_xticks(x + width * (len(_edge.columns) - 1) / 2)
ax.set_xticklabels(_EDGE_QUERIES, rotation=10, ha="right", fontsize=8)
ax.set_ylabel("R@1"), ax.set_title("Edge-case queries -- R@1 across the roster")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "visual" / "report_10_edge_case_queries.png", dpi=120)
plt.show()
_edge.round(4)


## 10.9 Where does the gain come from? Win/tie/loss and paired contingency


In [ ]:
# Per query: on how many of the 13 does the champion beat the baseline? Per source
# (K=10): both / only champion / only baseline / neither — does the champion FIX
# baseline failures or merely trade sources?
champ = CHAMPIONS[0]
for base in BASELINES:
    wtl = win_tie_loss(test_dumps[champ], stat_dumps[base], k=10)
    print(f"{champ} vs {base}")
    print(f"  per-query  W/T/L = {wtl['per_query']['win']}/"
          f"{wtl['per_query']['tie']}/{wtl['per_query']['loss']}")
    c = wtl["per_source@k"]
    print(f"  per-source @10: both {c['both']} | only champion {c['only_a']} | "
          f"only baseline {c['only_b']} | neither {c['neither']}   (n={sum(c.values())})")


## 10.10 Robustness to the averaging convention, and the full recall curve


In [ ]:
# Macro (each query 1/13 — spec convention) vs micro (each source equal):
# num_sources spans 27…5469, so agreement proves the headline number is not an
# averaging artifact.
mm_rows = []
for name, d in stat_dumps.items():
    mm = macro_vs_micro(d)
    mm_rows.append({"model": name,
                    "R@10 macro": mm["macro"]["R@10"], "R@10 micro": mm["micro"]["R@10"],
                    "R@5 macro": mm["macro"]["R@5"], "R@5 micro": mm["micro"]["R@5"]})
pd.DataFrame(mm_rows).set_index("model").round(4)


In [ ]:
# Recall@K for EVERY K = 1…10 (from the stored top-10 prefixes) — locates where in
# the ranking the gain materializes. (DGP and CLIP Enhanced overlap almost exactly
# through K=5 — a real tie, consistent with their overlapping CIs in §10.6.)
def _recall_curve_fig(path):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for name, d in stat_dumps.items():
        ax.plot(range(1, 11), recall_curve(d, gt_list), marker="o", ms=3, label=name)
    ax.set_xlabel("K"), ax.set_ylabel("macro Recall@K"), ax.legend(fontsize=7)
    ax.set_title("Recall@K, K = 1…10")
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    return fig


_fig_or_cached(_recall_curve_fig, "stats_recall_curve.png")


**Statistical verdict.** A model claim is accepted only when (i) its bootstrap CI separates from
the baseline's, (ii) the paired Δ CI excludes zero after Holm correction, and (iii) for learned
models, (i)–(ii) hold for every training seed. Under this standard, **A-BAP is significantly better
than every baseline on every spec metric** (all 30 Holm-corrected comparisons pass), with the
single-seed caveat of §10.4 stated rather than hidden. The per-category breakdown (§10.8) and the
qualitative grids (§9.4) localize *where* the improvement lives — predominantly on
negation-bearing queries, which is precisely the compositional capability this project set out to
build.


## 10.11 Beyond the mandatory benchmark — custom stress queries

The spec (§3.1) encourages additional custom queries, provided they are explicitly detailed and
rigorously interpreted. The mandatory set leaves specific claims of this report under-tested, so
each custom query below targets exactly one of them.

**Ground truth — the real thing, verified by reconstruction.** The official JSON is
authoritative and untouched; custom queries get their own `Evaluation/celeba_custom_evaluation.json`
generated by `src/generate_custom_gt.py`, which implements the instructor's full protocol
per image, with **no subsampling and no caps**:

- a **source** is valid iff it **violates every ± constraint** (each axis must demand a real
  modification — a `+a` query from an already-`a` source is twin retrieval, not a compositional
  edit) and has ≥ 5 valid targets (§3.1.2);
- a **target** is valid iff it strictly satisfies every constraint AND lies within Hamming ≤ 2
  of the source on the non-queried attributes (§3.1.1), computed via the popcount identity
  $\mathrm{ham}(s,x) = \lVert a_s \rVert_1 + \lVert a_x \rVert_1 - 2\,a_s a_x^{\top}$,
  chunked over sources.

**Validation is exact, not approximate:** `--verify-official` rebuilds every official query from
the attribute database and requires **identity** with the instructor's JSON — same source sets,
same target sets. It passes **14/14 queries with zero mismatches**, so the generator is provably
equivalent to the official protocol; the rule core is additionally unit-tested on hand-computed
corpora in `test/test_custom_gt.py`. (An earlier draft of this benchmark omitted the
violates-every-constraint source policy; the resulting twin-retrieval sources inflated every
cosine method — the reconstruction gate now makes that class of error impossible.)

**The nine queries** (all valid sources kept — sizes are whatever the protocol yields):

| Query | Hypothesis under test | sources | mean tgt/src |
|---|---|---:|---:|
| `-Young, -Smiling` | pure-negation **replication** — is the `-Male,-Mustache` floor-break (27 sources) a one-query fluke? | 686 | 19.9 |
| `-No_Beard` | **double negation** ("not beardless"): CLIP affirmation bias vs attribute-space scoring | 2002 | 22.1 |
| `+Heavy_Makeup, -Wearing_Lipstick` | **anti-correlated** pair — attacks the residual first-order (tree) dependence weak spot | 244 | 8.6 |
| `+Gray_Hair, -Young` | **correlation-consistent** contrast to the above (aging direction) | 532 | 11.3 |
| `+Big_Lips` | **probe-weak** attribute (probe acc 69.6%) — probe-noise propagation quantified | 6948 | 26.7 |
| `+Pale_Skin` | probe-weak / low-support control for the above | 3195 | 11.9 |
| `+Bald` | **rare attribute** (~2% prevalence): sparsity vs discriminativeness | 784 | 9.7 |
| `+Smiling, +Young, -Male, +Heavy_Makeup` | **depth-4**, corpus-mode attributes (small-n: wide-CI point estimate) | 28 | 10.9 |
| `+Male, -Young, +Eyeglasses, -Smiling` | **depth-4**, hard variant (constraints rare & conflicting) | 90 | 16.8 |

Evaluation is inference-only (`src/run_custom_eval.py`): every model re-scores from its cached
artifacts / saved checkpoint through its published seam — nothing is retrained, and each dump is
parity-gated against its CSV. Dual Score is omitted (weights VM-only).

In [ ]:
# == Custom benchmark — parity-gated per-source dumps + bootstrap CIs ==========
CUSTOM_PS_DIR = OUTPUT_DIR / "customQueries_celebA" / "per_source"
CUSTOM_MODELS = {   # display name -> dump stem (produced by src/run_custom_eval.py)
    "CLIP":          "custom_CLIP",
    "CLIP Enhanced": "custom_CLIP_Enhanced",
    "CLAY":          "custom_CLAY",
    "DGP":           "custom_DGP",
    "A-BAP":         "custom_A-BAP",
}
custom_dumps = {}
for _name, _stem in CUSTOM_MODELS.items():
    _path = CUSTOM_PS_DIR / f"{_stem}.npz"
    if _path.exists():
        custom_dumps[_name] = load_per_source(_path)
    else:
        print(f"[missing] {_path.name} — '{_name}' skipped (run src/run_custom_eval.py).")

custom_boot = {n: stratified_bootstrap(d, B=B, seed=SEED) for n, d in custom_dumps.items()}
pd.DataFrame([
    {"model": n, **{metric_label(f): f"{r.point:.4f}  [{r.ci_lo:.4f}, {r.ci_hi:.4f}]"
                    for f, r in res.items()}}
    for n, res in custom_boot.items()
]).set_index("model")


In [ ]:
# Per-query R@5 — the hypothesis-level view (rows follow the frozen benchmark order).
_custom_queries = list(next(iter(custom_dumps.values())))
pd.DataFrame({n: [d[q]["hit5"].mean() for q in _custom_queries]
              for n, d in custom_dumps.items()}, index=_custom_queries).round(4)

In [ ]:
# == Paired significance: Δ = A-BAP − baseline (B_SIG replicates, Holm-corrected) ==
if "A-BAP" in custom_dumps:
    _pvals, _rows = {}, []
    for _ref in (n for n in custom_dumps if n != "A-BAP"):
        for _f, _r in paired_bootstrap_diff(custom_dumps["A-BAP"], custom_dumps[_ref],
                                            B=B_SIG, seed=SEED).items():
            _pvals[f"{_ref} | {metric_label(_f)}"] = _r.p_value
            _rows.append({"vs": _ref, "metric": metric_label(_f),
                          "delta": round(_r.delta, 4),
                          "CI95": f"[{_r.ci_lo:+.4f}, {_r.ci_hi:+.4f}]",
                          "CI excludes 0": (_r.ci_lo > 0) or (_r.ci_hi < 0)})
    _adj = holm_bonferroni(_pvals)
    custom_sig = pd.DataFrame(_rows)
    custom_sig["p_holm"] = [_adj[f"{r.vs} | {r.metric}"] for r in custom_sig.itertuples()]
    display(custom_sig[custom_sig.metric.isin(["R@1", "R@5", "R@10"])].reset_index(drop=True))
else:
    print("[skip] A-BAP dump unavailable — significance table omitted.")

**Reading the custom benchmark** (macro means; per-query n is the true protocol yield, 28–6948):

- **The custom set is a genuine stress benchmark**: every model scores *below* its mandatory-set
  mean (CLAY 0.0085 R@1 here vs 0.0098 there; CLIP 0.0127 vs 0.0397; A-BAP 0.0532 vs 0.1104) —
  these queries are harder, not easier, which is what stress tests are for.
- **Aggregate.** A-BAP 0.053 / 0.199 / 0.305 (R@1/5/10) vs ≤ 0.018 / 0.061 / 0.103 for every
  baseline: the paired Δ is significant on **every metric against every baseline** (all Holm
  p ≈ 0.005, CIs far from zero) — on this out-of-benchmark query set the champion's margin is
  *proportionally larger* than on the mandatory set (3–4× the best cosine method at R@5).
- **Pure negation replicates, decisively** (`-Young, -Smiling`): A-BAP R@5 0.184 vs best
  baseline 0.023 (DGP collapses to 0.004). The `-Male,-Mustache` floor-break was not a
  one-query fluke — on a 686-source pure-negation query the probabilistic scorer is ~8× the
  best cosine method.
- **Double negation** (`-No_Beard`): A-BAP still wins (R@5 0.188 vs 0.092), but this is
  *vanilla CLIP's best relative showing* — subtracting $t(\text{No\_Beard})$ effectively adds
  beard semantics, so the affirmation bias becomes an accidental asset when the attribute name
  already contains the negation. (Notably CLIP Enhanced *hurts* here, 0.031: its delta
  normalization dilutes the one text direction that happened to point the right way.)
- **The correlation pair isolates correlation *hardness*, not direction, as the driver.** Both
  members collapse the cosine methods (≤ 0.025 R@5); A-BAP holds 0.107 on the anti-correlated
  `+Heavy_Makeup, -Wearing_Lipstick` and 0.102 on the consistent `+Gray_Hair, -Young` — a
  4–8× margin, but both are among its own weakest queries: attribute correlation beyond the
  first-order tree remains the structural weak point, now measured on a query
  *designed* to break it.
- **Probe quality predicts the margin** (observed directly): `+Big_Lips`
  (probe 69.6%) is A-BAP's thinnest single-attribute edge (R@1 0.033 vs 0.023; R@5 2.0×),
  while `+Pale_Skin` holds 2.5× at R@5 and `+Bald` 2.0× at 4× the absolute level.
- **Rare-but-visible is everyone's best case, and the champion's most** (`+Bald`): 0.471 vs
  0.239 (CLIP Enhanced) — a reliable probe on a rare attribute makes the posterior strongly
  informative where text deltas stay generic.
- **Depth-4 is where cosine scoring dies and the Bayes score does not.** On the hard variant
  (`+Male, -Young, +Eyeglasses, -Smiling`, 90 sources) CLIP, CLIP Enhanced and CLAY score
  **exactly 0.000 at every K**; DGP reaches 0.056; A-BAP scores **0.356 R@5 / 0.600 R@10** —
  each added constraint multiplies the difficulty of a one-vector cosine but merely adds one
  log-likelihood term to the Bayes score. The corpus-mode variant (28 sources — a wide-CI point
  estimate, stated as such) is near-zero for everyone (A-BAP 0.036, one source hit): when the
  target set is tiny *and* every constraint sits at the corpus mode, no method has signal left.

**Verdict.** On nine queries the mandatory benchmark never asked, the champion's advantage does
not merely persist — it widens (3–4× the best cosine baseline at R@5, significant everywhere).
The stress tests also localize its two honest weak spots: correlation-heavy constraint pairs
(held at ~0.10, far from solved) and probe-weak attributes (margin thins with probe accuracy).
Cosine methods, by contrast, hit literal zero on hard compositional depth — the substrate
argument of §6, reproduced out-of-benchmark.

---
# 11. Other Dataset Tests

The spec (§2) lists **OVAD** and **AwA2** as optional datasets “useful to develop your
project,” not required benchmarks. Both studies below are *secondary*: they re-run the
full CelebA model roster on a dataset with a structurally different attribute space, to
probe how each method's inductive bias transfers once the convenient properties of
CelebA (dense per-image binary labels, a large train split) are taken away. Each
subchapter is deliberately compact — the main deliverable is the CelebA benchmark above.

## 11.A Cross-Dataset Transfer: AwA2

**What AwA2 is.** Animals with Attributes 2 (Xian et al., 2018) is a zero-shot-learning
benchmark of **37,322 images across 50 animal classes** (e.g. `zebra`, `polar+bear`,
`persian+cat`), each class annotated with an **85-dimensional binary predicate vector**
(e.g. `stripes`, `swims`, `arctic`, `meatteeth`) describing shared visual / behavioural
properties of the species. Unlike CelebA, **the attributes are not per-image** — every
photograph of a given class inherits the *same* 85-bit row, so annotation granularity is
the class, not the instance. This is the property that drives everything below: it is
the sharpest structural contrast to CelebA's dense per-image labels among the spec's
suggested datasets.

This section re-runs the full CelebA model roster on AwA2's 7,468-image test corpus to
probe how each method transfers to a dataset with a very different attribute structure.

### 11.A.1 Protocol and ground truth

**CelebA attributes are per-image; AwA2 attributes are class-level** — every image of a
class shares that class's 85-bit row. This forces one adaptation of the §3.1.1 rule. A valid
source must **violate** every ± constraint of the query (the source policy §10.11 verified
on CelebA), and a valid target must **strictly satisfy** every constraint — but source and
target are then necessarily *different species*, and AwA2's class rows never sit closer than
free-attribute Hamming 3 to one another (proven in the cell below), so the spec's fixed
radius of 2 admits no pair at all. The ground truth therefore uses an **adaptive**
free-attribute Hamming radius: the nearest satisfying classes, +2 slack — the class-level
generalization of the spec's per-image ≤ 2 rule. Every source with ≥ 5 valid targets is
kept, with its **complete, uncapped** target set — 12 queries, 650–6,919 sources each,
exact Recall/Precision@K (`Evaluation/awa2_evaluation.json`, generated by
`src/awa2/generate_gt.py`).

Everything else is held fixed: frozen CLIP ViT-B/32 database over the 7,468-image test
corpus, Recall/Precision@{1, 5, 10}, the §10 statistical machinery verbatim, and a single
adapter seam (`src/awa2/adapter.py`) rebinding only data providers — the scoring math is
byte-identical code.

**Models under comparison** (display name ← AwA2 run)

| Display name | Method | Type | direction source |
|---|---|---|---|
| **LDE** | mined visual directions, flat arithmetic | training-free | visual |
| **GDE** | mined visual directions, geodesic + rejection | training-free | visual |
| **CLIP Enhanced** | text deltas, all geometry fixes (§4.2) | training-free | text |
| **DGP** | gated projection, uniform gate (§4.6) | training-free | text |
| **Dual Score** | query-conditioned dual-channel fusion (§5.4) | learned | text + probe |
| **CLAY** | pre-SVD prompt stacking (k=50, rotH) | training-free | text subspace |
| **A-BAP** | CD-Bayes + learned Φ (§8) — *CelebA champion* | learned | probe |

> **A-BAP** is our CelebA champion (`tier5_Bprime_listnet`). On CelebA it wins decisively;
> on AwA2, as §11.A.3 shows, its rule-scoring design pays off far less — **Dual Score**, which
> fuses a CLIP-cosine backbone with learned probe channels, is the strongest method here.

### 11.A.2 Why the radius must adapt: class-level annotation vs the spec radius

With class-level annotation, a valid source (violates the constraints) and a valid target
(satisfies them) are necessarily different species. The cell below measures, from the class
attribute matrix alone, the free-attribute Hamming distance from every violating class to
its nearest satisfying class, per query: the minimum across the whole curated set is **3**
and the median ≈ 20 of ~82 free attributes — every value exceeds the spec's fixed radius
of 2. This is the structural fact that motivates the adaptive radius above: it keeps the
spec's *intent* (targets are the source's nearest constraint-satisfying neighbours) while
remaining well-defined on class-level bits.


In [ ]:
# == §11.2 proof — nearest satisfying target class per query (local artifacts only) ==
# From the cached class-attribute matrix: per curated query, how many classes violate /
# satisfy the constraints, and the min/median free-attribute Hamming from a violating
# (source-eligible) class to its nearest satisfying (target-eligible) class. Every min
# exceeds the spec's fixed radius of 2 — this is what forces the adaptive radius that
# Evaluation/awa2_evaluation.json is generated with (nearest satisfying classes +2 slack).
import json as _json

_AWA2_ART = PROJECT_ROOT / "artifacts" / "awa2"
_cm = torch.tensor(_json.load(open(_AWA2_ART / "class_attribute_matrix.json")),
                   dtype=torch.float32)
_p2i = {n: i for i, n in enumerate(_json.load(open(_AWA2_ART / "predicate_names.json")))}
_AWA2_QUERIES = ["+stripes", "-furry", "+spots", "+swims", "+horns", "+longneck",
                 "-quadrapedal", "+swims & -ocean", "+big & -meat", "+forest & -arctic",
                 "+furry & -water & +ground", "+meat & -domestic & +fierce"]

_rows = []
for _q in _AWA2_QUERIES:
    _pos, _neg = parse_query(_q)
    _pi, _ni = [_p2i[a] for a in _pos], [_p2i[b] for b in _neg]
    _free = [c for c in range(_cm.shape[1]) if c not in set(_pi) | set(_ni)]
    _sat = torch.ones(_cm.shape[0], dtype=torch.bool)
    _viol = torch.ones(_cm.shape[0], dtype=torch.bool)
    for _a in _pi:
        _sat &= _cm[:, _a] == 1
        _viol &= _cm[:, _a] == 0
    for _b in _ni:
        _sat &= _cm[:, _b] == 0
        _viol &= _cm[:, _b] == 1
    if not _sat.any() or not _viol.any():
        _rows.append({"query": _q, "viol_classes": int(_viol.sum()),
                      "sat_classes": int(_sat.sum()), "min_ham": None, "median_ham": None,
                      "classes_within_radius_2": 0})
        continue
    _F = _cm[:, _free]
    _mins = (_F[_viol].unsqueeze(1) != _F[_sat].unsqueeze(0)).sum(-1).min(dim=1).values
    _rows.append({"query": _q, "viol_classes": int(_viol.sum()),
                  "sat_classes": int(_sat.sum()), "min_ham": int(_mins.min()),
                  "median_ham": int(_mins.median()),
                  "classes_within_radius_2": int((_mins <= 2).sum())})
pd.DataFrame(_rows).set_index("query")


In [ ]:
# == §11.3 benchmark — cached per-source dumps (schema of §10) ================
# Adaptive-radius class-level GT (§11.1-§11.2): sources violate, targets strictly
# satisfy, complete uncapped target sets. Trained-on-VM dumps may be absent locally;
# a missing model is announced and skipped.
AWA2_EVAL_JSON = PROJECT_ROOT / "Evaluation" / "awa2_evaluation.json"
AWA2_PS_DIR    = OUTPUT_DIR / "awa2" / "per_source"
AWA2_FIG_DIR   = OUTPUT_DIR / "awa2" / "visual"
AWA2_FIG_DIR.mkdir(parents=True, exist_ok=True)

awa2_gt_list = (json.loads(AWA2_EVAL_JSON.read_text(encoding="utf-8"))
                if AWA2_EVAL_JSON.exists() else None)
if awa2_gt_list is None:
    print(f"[missing] {AWA2_EVAL_JSON} — the recall-curve figure will be skipped.")

AWA2_MODELS = {  # ordered training-free (visual) -> training-free (text) -> learned
    "LDE":           "awa2_tier_lde.npz",
    "GDE":           "awa2_tier_gde.npz",
    "CLIP Enhanced": "awa2_tier0_baseline.npz",
    "DGP":           "awa2_tier2d_tau100.0_a1.0_center.npz",
    "Dual Score":    "awa2_tier3_dualscore_seed0.npz",
    "CLAY":          "awa2_tier_clay.npz",
    "A-BAP":         "awa2_tier5_bprime_listnet.npz",
}
awa2_dumps = {}
for _name, _fname in AWA2_MODELS.items():
    _path = AWA2_PS_DIR / _fname
    if _path.exists():
        awa2_dumps[_name] = load_per_source(_path)
    else:
        print(f"[missing] {_fname} — '{_name}' skipped (pull its NPZ from the VM bundle).")
print("AwA2 per-source dumps loaded:", list(awa2_dumps))


In [ ]:
# == Bootstrap CIs — every AwA2 model × spec metric (machinery of §10.6) =======
awa2_boot = {n: stratified_bootstrap(d, B=B, seed=SEED) for n, d in awa2_dumps.items()}
pd.DataFrame([
    {"model": n, **{metric_label(f): f"{r.point:.4f}  [{r.ci_lo:.4f}, {r.ci_hi:.4f}]"
                    for f, r in res.items()}}
    for n, res in awa2_boot.items()
]).set_index("model")

## 11.A.3 The result — Dual Score leads, A-BAP mid-pack

The headline is the **R@1 column**: **Dual Score leads clearly** (0.265), roughly double the
next-best method (CLIP Enhanced, 0.126), with A-BAP and DGP close behind (~0.10) and the
mined-direction pair (LDE, GDE) and CLAY well below (0.01–0.04).

**Ranking (mean R@1, this benchmark):**

| method | AwA2 R@1 | AwA2 R@5 | AwA2 R@10 |
|---|---:|---:|---:|
| Dual Score | **0.2648** | **0.4885** | **0.5854** |
| CLIP Enhanced | 0.1262 | 0.2583 | 0.3302 |
| A-BAP | 0.0975 | 0.2245 | 0.2942 |
| DGP | 0.1013 | 0.2051 | 0.2707 |
| LDE | 0.0422 | 0.1146 | 0.1669 |
| GDE | 0.0374 | 0.1040 | 0.1567 |
| CLAY | 0.0143 | 0.0535 | 0.0875 |

> **On this benchmark, unlike CelebA, there is no clean training-free vs. learned split and
> no clean visual vs. text split.** Dual Score (learned, text+probe) is far ahead of every
> training-free method; A-BAP (learned, probe) sits in the *middle* of the pack, not at the
> bottom; and the purely visual-mining pair (LDE/GDE), which relies only on class centroid
> contrasts with zero learned signal, trails almost everyone. The mechanism section (§11.5)
> unpacks why.


In [ ]:
# == Headline figures — CI bars + full recall curve ============================
plot_ci_bars(awa2_boot, path=AWA2_FIG_DIR / "awa2_stats_ci_bars.png",
             title="AwA2 — Recall@K with 95% CI")
if awa2_gt_list is not None:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for _name, _dump in awa2_dumps.items():
        ax.plot(range(1, 11), recall_curve(_dump, awa2_gt_list),
                marker="o", ms=3, label=_name)
    ax.set_xlabel("K"); ax.set_ylabel("macro Recall@K"); ax.legend(fontsize=7)
    ax.set_title("AwA2 — Recall@K, K = 1…10"); fig.tight_layout()
    fig.savefig(AWA2_FIG_DIR / "awa2_stats_recall_curve.png", dpi=120)


## 11.A.4 Significance — is Dual Score's lead real?

Paired stratified bootstrap of Δ = Dual Score − model (the champion on this benchmark vs
each other method), Holm–Bonferroni corrected over the {6 metrics × models} family — the
machinery of §10.3 verbatim. As there, the **CI of Δ** is the significance statement (it and
the p-value cannot contradict by construction).


In [ ]:
# == Paired significance: Δ = Dual Score − model, Holm-corrected (machinery of §10.7) ==
AWA2_CHAMP = "Dual Score"
if AWA2_CHAMP in awa2_dumps:
    _pvals, _rows = {}, []
    for _ref in (n for n in awa2_dumps if n != AWA2_CHAMP):
        for _f, _r in paired_bootstrap_diff(awa2_dumps[AWA2_CHAMP], awa2_dumps[_ref],
                                            B=B, seed=SEED).items():
            _pvals[f"{_ref} | {metric_label(_f)}"] = _r.p_value
            _rows.append({"vs": _ref, "metric": metric_label(_f),
                          "delta": round(_r.delta, 4),
                          "CI95": f"[{_r.ci_lo:+.4f}, {_r.ci_hi:+.4f}]",
                          "rel_%": round(100 * _r.rel_gain, 1), "p_raw": _r.p_value,
                          "CI excludes 0": (_r.ci_lo > 0) or (_r.ci_hi < 0)})
    _adj = holm_bonferroni(_pvals)
    awa2_sig = pd.DataFrame(_rows)
    awa2_sig["p_holm"] = [_adj[f"{r.vs} | {r.metric}"] for r in awa2_sig.itertuples()]
    display(awa2_sig[awa2_sig.metric.isin(["R@1", "R@5", "R@10"])].reset_index(drop=True))
else:
    print(f"[skip] champion dump '{AWA2_CHAMP}' unavailable — significance table omitted.")


Reading the table:

- **Every comparison's CI excludes zero at R@1/R@5/R@10** — Dual Score's lead over each of
  the other six methods is decisive, not sampling noise. The margin ranges from +0.139
  (vs. CLIP Enhanced) to +0.251 (vs. CLAY).
- **Dual Score − A-BAP at R@1 = +0.167**, CI `[+0.162, +0.173]`: the CelebA champion's
  rule-scoring design is not competitive with a fused cosine+probe score here, though the
  gap is smaller than against the purely training-free methods (LDE +0.223, GDE +0.227,
  CLAY +0.251) — A-BAP's probe channel is still contributing *something*, just not enough to
  catch a method that keeps a CLIP-cosine backbone.
- **Dual Score − LDE / − GDE are the largest margins among "does visual mining help"**:
  +0.223 and +0.227 respectively — with this exact, uncapped ground truth, mining directions
  purely from class centroids (no learned signal, no probe channel) is the weakest strategy
  on this dataset.

> **p-value floor.** With 12 benchmark queries but hundreds-to-thousands of per-source
> samples per query, the bootstrap p-values saturate near the resolution floor (~0.002 raw,
> lifted to ≈ 0.072 after Holm across 36 tests) for every comparison. So — exactly as §10.3
> argues from the other side — read the **Δ confidence intervals**, not the p-values, as the
> significance statement: every one here excludes zero, so Dual Score's lead holds across
> the whole metric family.


## 11.A.5 Why Dual Score wins, and where each method's bias shows

### The ground truth is a class relation, but not a linear one

**CelebA attributes are per-image; AwA2 attributes are class-level.** Every image of an AwA2
class shares that class's 85-bit row, so the ground truth collapses to a 50×50 class
relation: a valid source class *violates* every ± constraint, a valid target class
*strictly satisfies* every constraint and sits within the adaptive Hamming radius (nearest
satisfying classes + 2 slack) of the source.

This is *not* the same as saying the task is easy or that raw cosine already solves it. CLIP
clusters species tightly, so cosine carries real signal — but the target set for a composite
query (e.g. `+meat & -domestic & +fierce`) is often a handful of specific, non-adjacent
classes, not simply "the nearest neighbours of the reference." That is exactly where a
single cosine-based query vector runs out of expressiveness, and where a **learned,
query-conditioned combination of cosine and per-attribute evidence** — Dual Score's design —
has room to add value that pure geometry (LDE/GDE/DGP/CLAY) or pure rule-scoring (A-BAP)
cannot supply alone.

### Dual Score's advantage: it keeps the cosine backbone *and* adds learned channels

Dual Score fuses a CLIP-cosine-based query with trained presence/absence probe channels
(`β·s_neg + γ·s_pos`) in a single learned scorer. On queries where cosine is already a
strong signal (`+stripes` 0.550, `+spots` 0.369, `+swims` 0.381) it does not throw that
signal away the way CLAY's pre-SVD subspace stacking does; and on queries where cosine alone
struggles with negation (`-furry` 0.404, `-quadrapedal` 0.382 — both **far** above every
training-free method's score on the same query) its learned probe channels recover exactly
the negation signal cosine is known to lack (CLIP's text encoder has no "not X"
representation — Alhamoud et al. 2025). This dual structure is *why* it separates cleanly
from every other method at every K.

### A-BAP: not "last", but mismatched to single-attribute geometry

A-BAP is not uniformly the weakest method here — it actually **wins its query** against
every training-free method on `-furry` (0.088), `+horns` (0.182), `+longneck` (0.230),
`+big & -meat` (0.123), and `+meat & -domestic & +fierce` (0.190), several of which are
negation-heavy or multi-attribute composites where scoring the *rule*,
$S = \sum_a \log p_a + \sum_b \log(1-p_b) + \log P(\text{Hamming}\le\text{radius})$, still
pays off — the same inductive bias that makes it CelebA's champion (§6.3). Where it loses
badly is the single, strongly-visual positive queries (`+stripes` 0.038, `+swims` 0.045,
`-quadrapedal` 0.043) — cases where CLIP's raw cosine alone already separates the target
classes cleanly, and A-BAP's Poisson-binomial tail over ~80 free attributes at per-probe
error ε ≈ 0.14 (an expected ≈ 11 predicted flips, an order of magnitude above the Hamming
radius) adds probe noise on top of a signal that needed no rule-scoring help.

### Why CLAY and pure visual-mining trail

**CLAY** stacks all condition-prompt embeddings and scores in the pre-SVD subspace,
discarding most of the raw identity direction. On a task where the target set is a small,
specific subset of classes (not simply "near the reference"), throwing away identity is the
worst possible trade — CLAY is the weakest method on 9 of 12 queries, including a
`+furry & -water & +ground` R@1 of 0.000.

**LDE and GDE** mine directions purely from class centroid contrasts
($v_a = \text{mean}(\text{images with } a) - \mu$) with *no learned component at all*. That
is a strong, zero-modality-gap signal for single positive attributes with a wide, visually
coherent has-$a$ set (`+swims & +ocean`: LDE 0.299, GDE 0.270 — among the best scores for
that query across all seven methods) but it has no mechanism to combine multiple constraints
adaptively, weight them by query, or represent negation beyond fixed subtraction/rejection.
That is enough to win a handful of queries but not enough to compete with a method that
*learns* how to combine cosine and attribute evidence per query — hence LDE/GDE trail Dual
Score by the largest margins in the significance table above.

**DGP** sits between: a uniform gate over text-direction geometry recovers some of
CLIP-cosine's identity signal (`+stripes` 0.317, `+spots` 0.245 — close to Dual Score's own
numbers on those two) but, like CLAY, it is built entirely from text prompts and inherits
the modality gap on attributes with no strong lexical-visual correspondence
(`-quadrapedal` 0.002, `-furry` 0.003).

### LDE vs GDE: flat arithmetic edges geodesic, on both losing to learned fusion

First, the relationship: **LDE is not an enhancement of GDE — it is the linear special case
GDE generalizes** (Trager et al.'s "ideal words" = GDE with $\mathcal M = \mathbb R^n$:
intrinsic mean → arithmetic mean, Exp/Log → identity). In our code LDE is a zero-diff
ablation of GDE (same mined directions, only the geometric operators differ).

LDE beats GDE on 10 of 12 queries (mean R@1 0.042 vs 0.037) — a small but consistent margin,
though on this benchmark neither method is competitive with Dual Score. The gap decomposes
into the same two operator differences documented in the CelebA study:

- **Negation: rejection vs subtraction.** GDE removes the *entire* component along
  $\hat v_a$: $q \leftarrow q - (q\cdot\hat v_a)\hat v_a$. For a broad attribute like
  `quadrapedal` (~34 of 50 classes), $v_a$ points near the dominant axis of animal identity
  itself, so the projection is large for every reference and rejection risks amputating the
  identity anchor. LDE's subtraction shifts by the small fixed $\lVert v_a\rVert$ and leaves
  identity intact — visible in `-quadrapedal` (LDE 0.003 vs GDE 0.001) even though both are
  near the noise floor on this query.
- **Curvature: Log/Exp maps vs flat arithmetic.** GDE linearizes at one global $\mu$; the
  log-map factor $\theta/\sin\theta$ amplifies with distance from $\mu$. AwA2 spans 50
  well-separated species clusters (large $\theta$ spread) vs CelebA's single tight face
  cluster, so the tangent approximation is stretched exactly where AwA2 lives — the flat
  composition avoids that distortion, but the ceiling for *either* geometry-only approach is
  well below what learned per-query fusion (Dual Score) achieves.


## 11.A.6 What the AwA2 excursion establishes

**On this benchmark, Dual Score is the clear winner** (R@1 0.265, roughly double the
next-best method), because it is the only method that both (a) keeps a CLIP-cosine
backbone — preserving identity and the strong within-species visual signal AwA2 offers —
and (b) *learns* additive probe channels that recover the negation semantics CLIP's text
encoder cannot represent on its own. Every other method gives up one of those two
ingredients: CLAY and DGP discard identity information via text-only subspace/gate
constructions; LDE/GDE mine visual directions with no learned combination mechanism; A-BAP
scores the attribute-rule directly with no cosine backbone at all.

**A-BAP is not simply "the worst method"** — it wins several negation-heavy and
multi-attribute composite queries (`-furry`, `+horns`, `+longneck`,
`+meat & -domestic & +fierce`) against every training-free alternative, the same inductive
bias that makes it CelebA's champion. It loses on single, strongly-visual positive
attributes, where its probe-noise tail adds variance a clean cosine signal did not need.

This is what a transfer study should produce: not "the best method in isolation", but a map
of which methods' inductive biases transfer and which do not. Dual Score's architecture — a
learned, query-conditioned *combination* of geometric and semantic evidence, rather than a
commitment to either alone — is the one that generalizes best across both datasets in this
study, which is itself a useful finding for motivating the project's fusion-module design
(spec §3, item 2). The excursion also establishes a granularity lesson (§11.A.2): the spec's
fixed identity radius is only definable when attributes are annotated at the same
granularity as the retrieval unit — class-level bits force the adaptive-radius
generalization used here.

*Reproduction:* the runners in `src/awa2/` — `generate_gt.py` (ground truth), then
`tier0_baseline.py`, `tier_clay.py`, `tier_gde.py`, `tier_dgp.py` (training-free, local),
and `tier_dualscore.py`, `tier5_fusion.py` (trained, VM) — followed by `compare.py` for the
statistical comparison reproduced above.


## 11.B Cross-Dataset Transfer: OVAD

**What OVAD is.** The Open-Vocabulary Attribute Detection dataset (Bravo et al., CVPR
2023) is built on **2,000 COCO val2017 images**, but the retrieval unit is not the whole
photograph — it is an **object instance**: every one of the **14,300 annotated bounding
boxes** inside those images (a person, a chair, a dog, a car door, …) is its own labeled
entity. Each instance carries a **117-dimensional attribute vector** spanning 19 semantic
groups — colors, materials, patterns, textures, states (`open`/`closed`), cleanliness,
size, and face expression among them — describing fine-grained visual properties of that
specific object, not the scene around it. Two things make OVAD structurally distinct
from both CelebA and AwA2:

1. **Labels are tri-state, not binary**: `att_vec[i] ∈ {1, 0, −1}`, where `1` = positively
   labeled, `0` = negatively labeled, and `−1` = *not annotated* (occluded, ambiguous, or
   inapplicable — e.g. "material: wood" is simply not scored for a person instance).
   Roughly 13% of (instance, attribute) entries are unlabeled.
2. **There is no train split** — OVAD ships as a single 2,000-image, 14,300-instance
   annotation set. After filtering instances too small for CLIP's 32px patch size, the
   retrieval corpus is **8,810 instance-crops** spanning 1,967 of the 2,000 images.

This section re-runs the same CelebA model roster — training-free (LDE, GDE, CLIP
Enhanced, DGP, CLAY) and trained (Dual Score, A-BAP) — on OVAD's instance-crop corpus, to
see how each method's inductive bias fares under a *third*, again structurally different,
attribute regime: per-instance (like CelebA) but tri-state, small-sample, and
117-dimensional (like neither).

### 11.B.1 Protocol and ground truth

OVAD keeps CelebA's per-instance annotation granularity — so, unlike AwA2, **no radius
adaptation is needed for that reason.** But two other adaptations are required, both
driven by the tri-state labels and the absence of a train split:

- **Masked relative Hamming distance.** For a candidate pair, the comparison is only
  valid on attributes both sides have labeled: `V(s,t) = {i : att_vec[s,i] ≠ −1 ∧
  att_vec[t,i] ≠ −1}`. The relaxed-Hamming rule is then computed as a **rate**, not a
  fixed count — `D_Hrel(s,t) = (1/|V(s,t)|) · Σ_{i∈V(s,t)} 1[att_vec[s,i] ≠ att_vec[t,i]]`,
  and a target is valid iff `D_Hrel ≤ 0.05` — the same 5% tolerance CelebA's fixed
  radius-2-of-40 rule encodes, adapted to a variable, tri-state-masked denominator instead
  of a fixed count of 40 (`OVAD_Dataset/OVAD_Implementation_Plan.md` §3). Strict ±
  satisfaction on queried attributes still applies, and a candidate is excluded outright
  if a queried attribute is unlabeled on it (cannot verify the constraint).
- **No train split to mine directions from** — AwA2's LDE/GDE (mean visual-direction
  contrasts from a train split) have no train split to draw on here. We substitute a
  training-free, leave-one-out variant mined from the test corpus itself (used only for
  the direction cache, never for evaluation) so the visual-mining family stays in the
  comparison.

Every source with ≥ 5 valid targets is kept, with its complete, uncapped target set — 12
queries over the 8,810-instance corpus, **2,762 total (source, query) evaluation rows**,
exact Recall/Precision@K (`Evaluation/ovad_evaluation.json`, generated by
`src/ovad/generate_gt.py`). The adapter (`src/ovad/adapter.py`) rebinds only data
providers (attribute tables, tri-state-masked probe/tree/calibration trainers, the
query-string parser) — the single-seam pattern validated on AwA2 — so the scoring math in
every tier is byte-identical to its CelebA counterpart.

**Queries** (curated from OVAD's `head`/`medium` frequency buckets so ground truth is not
vacuous; composed queries chain up to 3 constraints):

| # | Query | Valid sources |
|---:|---|---:|
| 1 | `+ clothes_color:blue` | 497 |
| 2 | `− clothes_color:black` | 308 |
| 3 | `+ material:wood/wooden/bamboo/hardwood` | 821 |
| 4 | `+ pattern:striped/lined/pinstriped` | 351 |
| 5 | `+ state:closed` | 327 |
| 6 | `+ face_expression:happy/…/joyful` | 167 |
| 7 | `− cleanliness:unclean/dirt/…` | 90 |
| 8 | `+ size:big/large/giant/huge` | 101 |
| 9 | `− clothes_color:black & + face_expression:happy/…` | 55 |
| 10 | `+ state:closed & − cleanliness:unclean/…` | 5 |
| 11 | `+ material:wood/… & − cleanliness:unclean/…` | 16 |
| 12 | `+ clothes_color:blue & − clothes_color:black & + face_expression:happy/…` | 24 |

**Models under comparison** (identical roster and naming convention to §11.A, ordered
training-free visual → training-free text → learned):

| Display name | Method | Type | direction source |
|---|---|---|---|
| **LDE** | mined visual directions (leave-one-out, test corpus) | training-free | visual |
| **GDE** | mined visual directions, geodesic + rejection | training-free | visual |
| **CLIP Enhanced** | text deltas, all geometry fixes (§4.2) | training-free | text |
| **DGP** | gated projection, uniform gate (§4.6) | training-free | text |
| **Dual Score** | query-conditioned dual-channel fusion (§5.4) | learned | text + probe |
| **CLAY** | pre-SVD prompt stacking | training-free | text subspace |
| **A-BAP** | CD-Bayes + learned Φ (§8) — *CelebA champion* | learned | probe |

> As in §11.A, **A-BAP** is CelebA's champion (`tier5_Bprime_listnet`). The result below is
> a *sharper* version of the AwA2 story: on OVAD, A-BAP does not just lose its lead — it
> finishes **last of all seven methods, on every metric**, by a wide margin. §11.B.5
> explains why this is a data-volume ceiling, not a bug.


In [ ]:
# == §11.B.1 benchmark — cached per-source dumps (schema of §10) =================
# Masked relative-Hamming GT (§11.B.1): sources violate, targets strictly satisfy on
# labeled attributes, complete uncapped target sets. Trained-on-VM dumps may be absent
# locally; a missing model is announced and skipped.
OVAD_EVAL_JSON = PROJECT_ROOT / "Evaluation" / "ovad_evaluation.json"
OVAD_PS_DIR    = OUTPUT_DIR / "ovad" / "per_source"
OVAD_FIG_DIR   = OUTPUT_DIR / "ovad" / "visual"
OVAD_FIG_DIR.mkdir(parents=True, exist_ok=True)

ovad_gt_list = (json.loads(OVAD_EVAL_JSON.read_text(encoding="utf-8"))
                if OVAD_EVAL_JSON.exists() else None)
if ovad_gt_list is None:
    print(f"[missing] {OVAD_EVAL_JSON} — the recall-curve figure will be skipped.")

OVAD_MODELS = {  # ordered training-free (visual) -> training-free (text) -> learned
    "LDE":           "ovad_tier_lde.npz",
    "GDE":           "ovad_tier_gde.npz",
    "CLIP Enhanced": "ovad_tier0_baseline.npz",
    "DGP":           "ovad_tier2d_tau100.0_a1.0_center.npz",
    "Dual Score":    "ovad_tier3_dualscore_seed0.npz",
    "CLAY":          "ovad_tier_clay.npz",
    "A-BAP":         "ovad_tier5_bprime_listnet.npz",
}
ovad_dumps = {}
for _name, _fname in OVAD_MODELS.items():
    _path = OVAD_PS_DIR / _fname
    if _path.exists():
        ovad_dumps[_name] = load_per_source(_path)
    else:
        print(f"[missing] {_fname} — '{_name}' skipped (pull its NPZ from the VM bundle).")
print("OVAD per-source dumps loaded:", list(ovad_dumps))


In [ ]:
# == Bootstrap CIs — every OVAD model × spec metric (machinery of §10.6) =========
ovad_boot = {n: stratified_bootstrap(d, B=B, seed=SEED) for n, d in ovad_dumps.items()}
pd.DataFrame([
    {"model": n, **{metric_label(f): f"{r.point:.4f}  [{r.ci_lo:.4f}, {r.ci_hi:.4f}]"
                    for f, r in res.items()}}
    for n, res in ovad_boot.items()
]).set_index("model")


### 11.B.2 The result — every training-free cosine method beats every learned method

Mean recall across the 12 queries (2,762 total source rows):

| method | OVAD R@1 | OVAD R@5 | OVAD R@10 | type |
|---|---:|---:|---:|---|
| **Dual Score** | **0.0840** | **0.2536** | **0.4119** | learned, text+probe |
| DGP | 0.0844 | 0.2455 | 0.3972 | training-free, text |
| CLIP Enhanced | 0.0710 | 0.2415 | 0.3565 | training-free, text |
| LDE | 0.0504 | 0.1759 | 0.2789 | training-free, visual |
| GDE | 0.0493 | 0.1684 | 0.2717 | training-free, visual |
| CLAY | 0.0267 | 0.1158 | 0.1741 | training-free, text subspace |
| **A-BAP** | **0.0043** | **0.0919** | 0.1906 | learned, probe — *CelebA champion* |

The ranking inverts what CelebA showed. A-BAP is **last on every metric**: its R@5 (0.092)
is barely a third of Dual Score's (0.254), and its R@1 (0.004) is effectively zero — across
~2,762 evaluation rows it fails to place a single valid target first for all but a handful
of them. This is a sharper collapse than AwA2's (there A-BAP was *mid-pack*, 0.098 R@1
against a 0.265 leader — a real gap, but a respectable absolute score); on OVAD, A-BAP's
absolute score is the weakest number recorded for *any* method, on *any* of the three
datasets, in the whole project.


In [ ]:
# == Headline figures — CI bars + full recall curve =========================
plot_ci_bars(ovad_boot, path=OVAD_FIG_DIR / "ovad_stats_ci_bars.png",
             title="OVAD — Recall@K with 95% CI")
if ovad_gt_list is not None:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for _name, _dump in ovad_dumps.items():
        ax.plot(range(1, 11), recall_curve(_dump, ovad_gt_list),
                marker="o", ms=3, label=_name)
    ax.set_xlabel("K"); ax.set_ylabel("macro Recall@K"); ax.legend(fontsize=7)
    ax.set_title("OVAD — Recall@K, K = 1…10"); fig.tight_layout()
    fig.savefig(OVAD_FIG_DIR / "ovad_stats_recall_curve.png", dpi=120)


### 11.B.3 Significance — is the collapse real?

Paired stratified bootstrap of Δ = Dual Score − model (this benchmark's best method vs
each other, including the CelebA champion), Holm–Bonferroni corrected over the {6 metrics
× 6 comparisons} family — the machinery of §10.3 verbatim.


In [ ]:
# == Paired significance: Δ = Dual Score − model, Holm-corrected (machinery of §10.7) ==
OVAD_CHAMP = "Dual Score"
if OVAD_CHAMP in ovad_dumps:
    _pvals, _rows = {}, []
    for _ref in (n for n in ovad_dumps if n != OVAD_CHAMP):
        for _f, _r in paired_bootstrap_diff(ovad_dumps[OVAD_CHAMP], ovad_dumps[_ref],
                                            B=B, seed=SEED).items():
            _pvals[f"{_ref} | {metric_label(_f)}"] = _r.p_value
            _rows.append({"vs": _ref, "metric": metric_label(_f),
                          "delta": round(_r.delta, 4),
                          "CI95": f"[{_r.ci_lo:+.4f}, {_r.ci_hi:+.4f}]",
                          "rel_%": round(100 * _r.rel_gain, 1), "p_raw": _r.p_value,
                          "CI excludes 0": (_r.ci_lo > 0) or (_r.ci_hi < 0)})
    _adj = holm_bonferroni(_pvals)
    ovad_sig = pd.DataFrame(_rows)
    ovad_sig["p_holm"] = [_adj[f"{r.vs} | {r.metric}"] for r in ovad_sig.itertuples()]
    display(ovad_sig[ovad_sig.metric.isin(["R@1", "R@5", "R@10"])].reset_index(drop=True))
else:
    print(f"[skip] champion dump '{OVAD_CHAMP}' unavailable — significance table omitted.")


Reading the table: **every comparison's CI excludes zero at R@1/R@5/R@10.** Dual Score's
margin over A-BAP is the largest in the family (Δ ≈ +0.08 R@1, +0.16 R@5) — larger, in
relative terms, than its margin over any training-free method. The gap is not sampling
noise; it is the same structural effect §11.B.4–.5 explain.


### 11.B.4 Mechanism — the Poisson-binomial tail becomes scoring noise

A-BAP's score is dominated by a term (`hamming_tail_log`) that computes an *exact*
Poisson-binomial tail probability — `log P(Σ_c D_c ≤ hamming_max)` over the free
(non-queried) attributes, where each `D_c` is a Bernoulli mismatch indicator built from
probe posteriors. This is precisely the term that wins CelebA: it scores the ground
truth's own generative rule (a relaxed Hamming ball) instead of approximating it
geometrically. Its reliability depends on how the GT's **truncation radius**
(`hamming_max`) compares to the **expected number of probe-induced mismatches** over the
free attribute count `F`:

| | CelebA | **OVAD** |
|---|---:|---:|
| Attributes | 40 | **117** |
| Free attributes in a typical query (`F`) | ≈ 37 | ≈ 114 |
| GT tolerance (relative) | 5% (2/40) | 5% (by construction) |
| `hamming_max` (absolute) | 2 | 6 |
| Probe test accuracy, mean / **worst head** | 88.5% / **69.7%** | 92.3% / **55.2%** |
| Expected mismatches `F·ε` | 37 × 0.115 ≈ **4.3** | 114 × 0.077 ≈ **8.8** |

`hamming_max = 6` is correctly tuned — it matches OVAD's 5% tolerance exactly the way
CelebA's `hamming_max = 2` matches its own. The problem is that `F·ε` (expected mismatches
under the probes' *own* error rate) already sits close to the truncation radius on both
datasets — and OVAD has 3× the attributes to accumulate probe error over, so `F·ε` roughly
doubles (4.3 → 8.8) even though mean probe accuracy looks similar, because the *worst*
heads (rare attributes with few positive examples) drag it down. Once `F·ε ≫ hamming_max`,
the Poisson-binomial tail saturates toward "almost certainly too many mismatches" for
nearly every candidate — it stops discriminating true positives from true negatives and
instead reports the probes' aggregate noise floor.

**Confirmation via a closed-form λ-sweep** (no retraining — a pure scoring sweep over the
frozen Stage A′ core, `S = constraint + tail + λ·cos(v_x, v_ref)`):

| λ | R@1 | R@5 |
|---|---|---|
| 0 (Bayes core alone) | 0.0183 | 0.0724 |
| 0.5 | 0.0166 | 0.1019 |
| 1 | 0.0205 | 0.1275 |
| 2 (CelebA's own tuned value) | 0.0343 | 0.1489 |
| 8 | 0.0436 | 0.1529 |

R@5 **doubles** as raw CLIP cosine is blended back in (λ: 0→8) — mirroring the same curve
AwA2's diagnostic produced. The probe-mediated tail/tree core is close to uninformative on
OVAD in isolation.

**Training does not rescue it.** The learned correction Φ (`BayesPhi`, same architecture as
CelebA's) is zero-initialized so training starts from the closed-form score exactly. On
OVAD, training barely moves anything: the tail gain `κ` sits at **0.995 for the entire
second half of training** (statistically indistinguishable from its 1.0 init), and the
ListNet loss oscillates in a flat 3.5–3.7 band for 20 epochs with no visible downward
trend. On CelebA the same training loop descends monotonically and lifts R@5 by +0.039
over the untrained core. On OVAD, Φ never learns a useful correction — the trained
checkpoint is, for practical purposes, still its own initialization.

### 11.B.5 Why this can't be tuned away

Three compounding constraints sit on OVAD simultaneously, and none of them are present
together on CelebA:

| | CelebA | **OVAD** |
|---|---:|---:|
| Training pool size | 162,770 images | **6,216** instance-crops |
| Attributes fit by probe/tree/Φ | 40 | **117** |
| Images-per-attribute ratio | ≈ 4,069 | **≈ 53** |
| Label completeness | 100% | **~87%** (tri-state, ~13% unlabeled) |

**≈26× less training data** must fit **~3× more parameters that scale with attribute
count** (a 117-head linear probe, a 117×117 Chow-Liu tree, 117 temperature scalars, a
multi-head `BayesPhi` network) — the images-per-attribute ratio drops from ≈4,069 to ≈53,
nearly two orders of magnitude, exactly where every one of these components needs *more*
data to fit reliably as its dimensionality grows. Layered on top, the tri-state labels mean
every masked estimator works from strictly less effective evidence per attribute even
before the 26× pool shrinkage is counted. No realistic retuning of `hamming_max`,
probe epochs, hidden size, or weight decay closes this gap — every one of those scalars is
already fit fresh on OVAD's own split; the bottleneck is the *ratio* of trainable structure
to available evidence, which only a fundamentally more data-efficient dependency model (or
far more labeled data) could fix.

**What survives:** reusing CelebA's own tuned λ = 2.0 in the closed-form `Aprime_C4` scorer
(Stage A′ core + a fixed, non-learned `λ·cos` term — no OVAD-specific retuning) reaches
**R@5 = 0.149**, more than double the bare Bayes core (0.072) and 62% above the trained
champion (0.092) — still short of the training-free cosine methods (~0.24–0.25), but the
CD-Bayes row that survives transfer, and it survives for the reason §11.A.5 already gives
for AwA2: **the more a score is anchored in raw CLIP visual identity, the better it
transfers off CelebA; the more it depends on the learned probe/tree/tail apparatus, the
more exposed it is to exactly the attribute-count / data-volume regime OVAD sits in.**


### 11.B.6 What the OVAD excursion establishes

**On OVAD, every training-free cosine method beats every learned method, and A-BAP —
CelebA's champion — finishes last of seven.** The mechanism is the same one §11.A already
isolated on AwA2 (a rule-scoring design that wins when it has enough per-attribute
evidence to keep its worst-case probe reliable, and loses when it doesn't) — OVAD simply
sits in a far more extreme version of that regime: 3× the attributes, ~26× less training
data, and tri-state rather than dense binary labels. The result is not a mistuned
hyperparameter but a genuine data-volume ceiling: `κ` frozen at 0.995 and a flat training
loss are direct evidence that Φ never received a usable gradient signal, not that it was
stopped early or under-tuned.

Read together, the two excursions bracket the same finding from opposite directions:
**AwA2 shows the CelebA champion's design degrading gracefully to mid-pack** when
attributes move from per-image to per-class; **OVAD shows it collapsing outright** when
per-instance attributes triple in count while training data shrinks by an order of
magnitude. Both point to the same design lesson that motivates this project's fusion
module Φ (spec §3, item 2): a scorer that keeps a CLIP-cosine backbone and *adds* learned
structure on top (Dual Score, on both excursions) transfers far better than one that
replaces cosine with a purely learned combinatorial rule (A-BAP) — however well that rule
performs on the dataset it was tuned for.

*Reproduction:* the runners in `src/ovad/` — `generate_gt.py` (ground truth), then
`tier0_baseline.py`, `tier_clay.py`, `tier_gde.py` (LDE + GDE), `tier_dgp.py`
(training-free, local), and `tier_dualscore.py`, `tier5_fusion.py` (trained, VM) —
followed by `compare.py` for the statistical comparison reproduced above. The closed-form
λ-sweep of §11.B.4 needs no training (`python src/ovad/tier5_fusion.py --diagnose`, CPU-safe,
under a minute). Full write-up: `notebooks/ovad_negative_transfer_analysis.md`.


# 12. Conclusion

We followed the assignment's arc to its end and then one step past it. The training-free study
showed that within single-vector cosine scoring, geometry is exhausted at mean R@5 ≈ 0.115;
learned geometric fusion raised the ceiling to 0.175 (Dual Score) but could not touch the negation floor,
because the floor was never an operator problem — it was a *scoring-substrate* problem. Reading
the benchmark's ground-truth rule as a generative process over attribute space, and scoring it as
Bayesian inference over calibrated probe posteriors, broke the floor with zero training; the
correlation-aware refinements and the amortized fusion module Φ — provably initialized at the
closed form — delivered the final **R@1 0.110 / R@5 0.291 / R@10 0.413**, a 1.6× improvement in
R@10 over our strongest geometric model (Dual Score). The AwA2 excursion (§11) then put the result in
perspective: on class-level attributes — where the spec radius must generalize to an
adaptive one — the rule-scoring bias that wins CelebA is no longer dominant, and Dual
Score's learned fusion of a cosine backbone with probe channels leads decisively, mapping
each method to the inductive bias it embodies. *CLAY turned VLM space into a text-conditional
similarity space; we turned it into a text-conditional probability space.*

# 13. References

- Berasi et al. — *Not only text: Exploring compositionality of visual representations in
  VLMs.* CVPR 2025. (GDE)
- Lim et al. — *CLAY: Conditional visual similarity modulation in vision-language embedding
  space.* CVPR 2026.
- Liu et al. — *Deep learning face attributes in the wild.* ICCV 2015. (CelebA)
- Xian et al. — *Zero-shot learning — a comprehensive evaluation of the good, the bad and the
  ugly.* IEEE TPAMI, 2018. (AwA2)
- Radford et al. — *Learning transferable visual models from natural language supervision.*
  ICML 2021. (CLIP)
- Liang et al. — *Mind the Gap: Understanding the modality gap in multi-modal contrastive
  learning.* NeurIPS 2022.
- Alhamoud et al. — *Vision-language models do not understand negation.* arXiv 2025. (NegBench)
- Oldfield et al. — *Parts-of-speech grounded subspaces in vision-language models.* NeurIPS 2023.
- Trager et al. — *Linear spaces of meanings: compositional structures in VLMs.* ICCV 2023. (LDE)
- Guo et al. — *On calibration of modern neural networks.* ICML 2017.
- Chow, Liu — *Approximating discrete probability distributions with dependence trees.*
  IEEE Trans. IT, 1968.
- Friedman et al. — *Bayesian network classifiers.* Machine Learning, 1997. (TAN)
- Cao et al. — *Learning to rank: from pairwise approach to listwise approach.* ICML 2007. (ListNet)
- Wang et al. — *The LambdaLoss framework for ranking metric optimization.* CIKM 2018.
- Bogolin et al. — *Cross-modal retrieval with querybank normalisation.* CVPR 2022. (QB-Norm)
- Schroff et al. — *FaceNet.* CVPR 2015. (online hard-negative mining)
- van den Oord et al. — *Representation learning with contrastive predictive coding.* arXiv 2018. (InfoNCE)
- Kim et al. — *Interpretability beyond feature attribution (TCAV).* ICML 2018. (concept directions)
